# **Script 1**  (Run before running **Script 3** each time)

In [ ]:
# ==============================================================================
# SCRIPT 1: Feature Engineering Engine (v6 - FINAL w/ 3-Pointers)
# ==============================================================================
!pip install -q pandas unidecode tqdm scikit-learn

import pandas as pd
import numpy as np
import os
import re
import pytz
from datetime import datetime
from google.colab import drive
from unidecode import unidecode
from tqdm import tqdm

# (All helper functions are unchanged and correct)
def load_historical_odds(file_path):
    print("--- Loading and Processing Historical Odds Data ---")
    try: df = pd.read_csv(file_path)
    except FileNotFoundError: print(f"CRITICAL ERROR: File not found at {file_path}"); return pd.DataFrame()
    team_map = {'atl': 'ATL', 'bkn': 'BKN', 'bos': 'BOS', 'cha': 'CHA', 'chi': 'CHI', 'cle': 'CLE', 'dal': 'DAL', 'den': 'DEN', 'det': 'DET', 'gs': 'GSW', 'hou': 'HOU', 'ind': 'IND', 'lac': 'LAC', 'lal': 'LAL', 'mem': 'MEM', 'mia': 'MIA', 'mil': 'MIL', 'min': 'MIN', 'no': 'NOP', 'ny': 'NYK', 'okc': 'OKC', 'orl': 'ORL', 'phi': 'PHI', 'phx': 'PHX', 'por': 'POR', 'sa': 'SAS', 'sac': 'SAC', 'tor': 'TOR', 'utah': 'UTA', 'wsh': 'WAS'}
    df['home'] = df['home'].str.lower().map(team_map); df['away'] = df['away'].str.lower().map(team_map)
    df.dropna(subset=['home', 'away'], inplace=True)
    df['GAME_DATE'] = pd.to_datetime(df['date']); df['VEGAS_SPREAD'] = np.where(df['whos_favored'] == 'home', -df['spread'], df['spread']); df.rename(columns={'total': 'VEGAS_TOTAL'}, inplace=True)
    df['game_pair'] = [frozenset(x) for x in zip(df['home'], df['away'])]; print("✅ Historical odds data processed successfully.")
    return df[['GAME_DATE', 'game_pair', 'VEGAS_SPREAD', 'VEGAS_TOTAL']]
def get_current_nba_season():
    simulated_today = datetime(2025, 11, 10); year, month = simulated_today.year, simulated_today.month; return f"{year}-{str(year+1)[-2:]}" if month >= 9 else f"{year-1}-{str(year)[-2:]}"
def convert_min_to_decimal(min_str):
    if isinstance(min_str, str) and ':' in min_str:
        try: minutes, seconds = map(int, min_str.split(':')); return minutes + seconds / 60.0
        except (ValueError, TypeError): return np.nan
    return pd.to_numeric(min_str, errors='coerce')
def calculate_weighted_rolling_averages(group, stats_to_roll):
    avg_minutes = group['MIN'].mean()
    weights = np.ones(len(group)); weights[group['MIN'] < 0.7 * avg_minutes] = 0.5; weights[group['MIN'] > 1.15 * avg_minutes] = 1.2; group['WEIGHT'] = weights
    for stat in stats_to_roll:
        weighted_stat = group[stat] * group['WEIGHT']
        for w in [5, 10, 20]:
            rolling_weighted_sum = weighted_stat.shift(1).rolling(w, min_periods=1).sum()
            rolling_weight_sum = group['WEIGHT'].shift(1).rolling(w, min_periods=1).sum()
            group[f'W_ROLL_AVG_{stat}_{w}'] = rolling_weighted_sum / rolling_weight_sum
    return group

def process_nba_data_v6():
    drive.mount('/content/drive', force_remount=True); BASE_DIR = "/content/drive/MyDrive/NBA/"; DATA_DIR = os.path.join(BASE_DIR, "Historical Data/"); BOXSCORE_DIR = os.path.join(DATA_DIR, "NocturneBear_Boxscores")
    odds_df = load_historical_odds(os.path.join(DATA_DIR, "nba_2008-2025.csv"));
    if odds_df.empty: return
    print("\nStep 2: Loading and cleaning historical box score data...")
    df = pd.concat([pd.read_csv(os.path.join(BOXSCORE_DIR, f), low_memory=False) for f in os.listdir(BOXSCORE_DIR) if f.endswith('.csv')], ignore_index=True)
    # *** NEW: Add FG3M to rename map ***
    rename_map = {'season_year': 'SEASON_YEAR', 'game_date': 'GAME_DATE', 'gameId': 'GAME_ID', 'personId': 'PLAYER_ID', 'personName': 'PLAYER_NAME', 'teamTricode': 'TEAM_ABBREVIATION', 'points': 'PTS', 'reboundsTotal': 'REB', 'assists': 'AST', 'minutes': 'MIN', 'fieldGoalsAttempted': 'FGA', 'freeThrowsAttempted': 'FTA', 'threePointersAttempted': 'FG3A', 'threePointersMade': 'FG3M', 'steals': 'STL', 'blocks': 'BLK', 'turnovers': 'TOV'}
    df.rename(columns=rename_map, inplace=True); CURRENT_SEASON_STR = get_current_nba_season(); df = df[(df['SEASON_YEAR'] >= '2015-16') & (df['SEASON_YEAR'] < CURRENT_SEASON_STR)].copy()
    df = df[df['MIN'].notna() & (df['MIN'] != '0:00') & (df['MIN'] != '0')].copy(); df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE']); df['MIN'] = df['MIN'].apply(convert_min_to_decimal)
    # *** NEW: Add FG3M to numeric cols ***
    numeric_cols = ['PTS', 'REB', 'AST', 'MIN', 'FGA', 'FTA', 'FG3A', 'FG3M', 'STL', 'BLK', 'TOV'];
    for col in numeric_cols: df[col] = pd.to_numeric(df[col], errors='coerce')
    df.dropna(subset=numeric_cols, inplace=True); df.sort_values(by=['PLAYER_ID', 'GAME_DATE'], inplace=True)
    print("\nStep 3: Merging historical odds data...")
    opponent_df = df[['GAME_ID', 'TEAM_ABBREVIATION']].drop_duplicates(); df = pd.merge(df, opponent_df.rename(columns={'TEAM_ABBREVIATION': 'OPPONENT_ABBR'}), on='GAME_ID'); df = df[df['TEAM_ABBREVIATION'] != df['OPPONENT_ABBR']].copy()
    df['game_pair'] = [frozenset(x) for x in zip(df['TEAM_ABBREVIATION'], df['OPPONENT_ABBR'])]; df = pd.merge(df, odds_df, on=['GAME_DATE', 'game_pair'], how='left'); df.drop(columns=['game_pair'], inplace=True)
    print(f"Merged odds for {df['VEGAS_SPREAD'].notna().sum()} of {len(df)} rows.")

    # *** NEW: Add FG3M and FG3A to feature engineering ***
    print("\nStep 4: Engineering Functional Role Ratios..."); stats_for_ratios = ['PTS', 'REB', 'AST', 'FGA', 'FG3A', 'FTA', 'TOV', 'FG3M']
    df['MIN'] = df['MIN'].replace(0, np.nan)
    for stat in stats_for_ratios: df[f'{stat}_PM'] = df[stat] / df['MIN']
    print("\nStep 5: Engineering Weighted Rolling Averages & Trends..."); stats_to_roll = ['PTS', 'REB', 'AST', 'MIN', 'FG3M', 'FG3A'] + [f'{s}_PM' for s in stats_for_ratios]
    df = df.groupby('PLAYER_ID', group_keys=False).progress_apply(calculate_weighted_rolling_averages, stats_to_roll=stats_to_roll)
    for stat in stats_to_roll: df[f'TREND_{stat}'] = df[f'W_ROLL_AVG_{stat}_{[5,10,20][0]}'] - df[f'W_ROLL_AVG_{stat}_{[5,10,20][-1]}']
    df['DAYS_OF_REST'] = df.groupby('PLAYER_ID')['GAME_DATE'].transform(lambda x: x.diff().dt.days - 1).clip(upper=7).fillna(7)

    print("\nStep 6: Engineering Opponent Context Features...")
    # *** NEW: Add FG3M to opponent allowed stats ***
    team_allowed_stats = df.groupby(['GAME_ID', 'OPPONENT_ABBR']).agg(GAME_DATE=('GAME_DATE', 'first'), PTS=('PTS', 'sum'), REB=('REB', 'sum'), AST=('AST', 'sum'), FG3M=('FG3M', 'sum')).reset_index(); team_allowed_stats.rename(columns={'OPPONENT_ABBR': 'TEAM'}, inplace=True); team_allowed_stats.sort_values(by=['TEAM', 'GAME_DATE'], inplace=True)
    for stat in ['PTS', 'REB', 'AST', 'FG3M']: team_allowed_stats[f'OPP_ALLOWED_{stat}_AVG_30'] = team_allowed_stats.groupby('TEAM')[stat].transform(lambda x: x.shift(1).rolling(30, min_periods=5).mean())
    df = pd.merge(df, team_allowed_stats[['GAME_ID', 'TEAM', f'OPP_ALLOWED_PTS_AVG_30', f'OPP_ALLOWED_REB_AVG_30', f'OPP_ALLOWED_AST_AVG_30', f'OPP_ALLOWED_FG3M_AVG_30']], left_on=['GAME_ID', 'OPPONENT_ABBR'], right_on=['GAME_ID', 'TEAM'], how='left')

    print("\nStep 7: Finalizing data and creating target variables...")
    for stat in ['MIN', 'PTS', 'REB', 'AST', 'FG3M']: df[f'TARGET_{stat}'] = df[stat]
    dd_td_stats = ['PTS', 'REB', 'AST', 'STL', 'BLK']; df['DD_COUNT'] = df[dd_td_stats].ge(10).sum(axis=1); df['TARGET_DD'] = (df['DD_COUNT'] >= 2).astype(int); df['TARGET_TD'] = (df['DD_COUNT'] >= 3).astype(int)
    feature_columns = [c for c in df.columns if 'W_ROLL_AVG' in c or 'TREND' in c or 'OPP_ALLOWED' in c or '_PM' in c or 'VEGAS' in c] + ['DAYS_OF_REST']; target_columns = [c for c in df.columns if 'TARGET' in c]
    final_df = df[['GAME_DATE', 'PLAYER_ID', 'PLAYER_NAME'] + feature_columns + target_columns].copy(); final_df.dropna(subset=[c for c in feature_columns if 'W_ROLL_AVG' in c], inplace=True)
    output_path = os.path.join(DATA_DIR, "nba_advanced_training_data_v6.csv"); final_df.to_csv(output_path, index=False); print(f"\n✅ Processing complete! {len(final_df)} model-ready rows saved to: {output_path}"); display(final_df.head())

if __name__ == "__main__":
    # Get current time in EST
    est = pytz.timezone('US/Eastern')
    now = datetime.now(est)
    # Format: 2:53 p.m. 1-29-26
    formatted_time = now.strftime("%-I:%M %p").lower().replace("am", "a.m.").replace("pm", "p.m.")
    formatted_date = now.strftime("%-m-%-d-%y")
    print(f"{formatted_time} {formatted_date} in EST")

    tqdm.pandas(); process_nba_data_v6()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 3.0 MB/s eta 0:00:00
4:25 p.m. 2-4-26 in EST
Mounted at /content/drive
--- Loading and Processing Historical Odds Data ---
✅ Historical odds data processed successfully.

Step 2: Loading and cleaning historical box score data...

Step 3: Merging historical odds data...
Merged odds for 228094 of 228160 rows.

Step 4: Engineering Functional Role Ratios...

Step 5: Engineering Weighted Rolling Averages & Trends...


100%|██████████| 1347/1347 [00:57<00:00, 23.33it/s]



Step 6: Engineering Opponent Context Features...

Step 7: Finalizing data and creating target variables...

✅ Processing complete! 226813 model-ready rows saved to: /content/drive/MyDrive/NBA/Historical Data/nba_advanced_training_data_v6.csv


,GAME_DATE,PLAYER_ID,PLAYER_NAME,VEGAS_SPREAD,VEGAS_TOTAL,PTS_PM,REB_PM,AST_PM,FGA_PM,FG3A_PM,...,OPP_ALLOWED_AST_AVG_30,OPP_ALLOWED_FG3M_AVG_30,DAYS_OF_REST,TARGET_MIN,TARGET_PTS,TARGET_REB,TARGET_AST,TARGET_FG3M,TARGET_DD,TARGET_TD
1,2015-10-30,708,Kevin Garnett,-3.5,204.0,0.179238,0.268857,0.000000,0.134429,0.0,...,NaN,NaN,1.0,22.316667,4,6,0,0,0,0
2,2015-11-02,708,Kevin Garnett,-3.5,204.0,0.000000,0.060302,0.060302,0.180905,0.0,...,NaN,NaN,2.0,16.583333,0,1,1,0,0,0
3,2015-11-05,708,Kevin Garnett,4.5,197.5,0.000000,0.355030,0.088757,0.088757,0.0,...,NaN,NaN,2.0,11.266667,0,4,1,0,0,0
4,2015-11-07,708,Kevin Garnett,-9.0,201.5,0.163043,0.407609,0.163043,0.163043,0.0,...,21.833333,6.833333,1.0,12.266667,2,5,2,0,0,0
5,2015-11-09,708,Kevin Garnett,-8.5,204.0,0.220994,0.441989,0.055249,0.276243,0.0,...,23.500000,9.000000,1.0,18.100000,4,8,1,0,0,0


In [ ]:
!pip install nba_api
from nba_api.stats.endpoints import leaguedashteamstats
import pandas as pd

# Initialize the endpoint
stats = leaguedashteamstats.LeagueDashTeamStats(
    season='2025-26',
    measure_type_detailed_defense='Opponent', # This toggles 'Opponent' stats
    per_mode_detailed='PerGame'               # Get averages per game
)

# Load into a DataFrame
df = stats.get_data_frames()[0]

# Filter for the four specific metrics
defense_df = df[['TEAM_NAME', 'OPP_PTS', 'OPP_REB', 'OPP_AST', 'OPP_FG3M', 'OPP_FG3_PCT']]
print(defense_df.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.0/319.0 kB 20.0 MB/s eta 0:00:00
           TEAM_NAME  OPP_PTS  OPP_REB  OPP_AST  OPP_FG3M  OPP_FG3_PCT
0      Atlanta Hawks    118.1     46.6     27.4      13.2        0.354
1     Boston Celtics    110.1     42.2     25.7      14.4        0.364
2      Brooklyn Nets    114.7     43.2     27.5      12.7        0.383
3  Charlotte Hornets    114.8     39.8     27.4      13.3        0.367
4      Chicago Bulls    119.7     44.7     27.8      13.8        0.369


In [ ]:
from nba_api.stats.endpoints import leaguedashteamstats

# Fetch opponent stats and their rankings
stats = leaguedashteamstats.LeagueDashTeamStats(
    season='2025-26',
    measure_type_detailed_defense='Opponent',
    per_mode_detailed='PerGame',
    rank='Y'  # This is crucial for getting the rank columns
)

df = stats.get_data_frames()[0]

# Filter for the relevant stat and rank columns
cols = [
    'TEAM_NAME',
    'OPP_PTS', 'OPP_PTS_RANK',
    'OPP_REB', 'OPP_REB_RANK',
    'OPP_AST', 'OPP_AST_RANK',
    'OPP_FG3M', 'OPP_FG3M_RANK'
]

defense_rankings = df[cols]
print(defense_rankings.sort_values('OPP_PTS_RANK'))


                 TEAM_NAME  OPP_PTS  OPP_PTS_RANK  OPP_REB  OPP_REB_RANK  \
20   Oklahoma City Thunder    108.0             1     45.6            25   
10         Houston Rockets    110.0             2     39.3             1   
1           Boston Celtics    110.1             3     42.2             7   
8          Detroit Pistons    110.1             4     41.4             4   
23            Phoenix Suns    111.2             5     43.4            13   
27         Toronto Raptors    112.1             6     43.5            14   
26       San Antonio Spurs    112.1             7     43.2             9   
19         New York Knicks    112.6             8     41.6             5   
12             LA Clippers    112.8             9     42.0             6   
9    Golden State Warriors    113.6            10     44.7            18   
17  Minnesota Timberwolves    114.1            11     43.2             8   
2            Brooklyn Nets    114.7            12     43.2            10   
3        Cha

# **Script 3**



In [ ]:
# ==============================================================================
# SCRIPT 3: NBA Professional Sharp & ML Engine (v100 - Perfected Context & How-To)
# Features: 10-Game Streaks, Robust H2H, CLV/RLM/Steam on ALL Sheets, Full Guides
# ==============================================================================

# --- Part 0: Setup & Installs ---
!pip install -q xlsxwriter nba_api scikit-learn pandas joblib pytz unidecode requests lightgbm python-dateutil tqdm openpyxl scipy

import os
import re
import time
import joblib
import warnings
import requests
import json
import numpy as np
import pandas as pd
import pytz
from datetime import datetime, timedelta
from dateutil import parser
from unidecode import unidecode
from tqdm import tqdm
from decimal import Decimal
from scipy.stats import poisson, norm
from google.colab import drive
from IPython.display import display, clear_output
from xlsxwriter.utility import xl_col_to_name

from nba_api.stats.endpoints import playergamelog, leaguedashplayerstats, teamestimatedmetrics, drafthistory, scoreboardv2, leaguedashteamstats, leaguegamefinder
from nba_api.stats.static import teams, players

warnings.filterwarnings("ignore")

# --- Part 1: Configuration ---
GDRIVE_BASE_PATH = "/content/drive/MyDrive/NBA/"
MODEL_DIR = os.path.join(GDRIVE_BASE_PATH, "Training Model Advanced/")
OUTPUT_DIR = os.path.join(GDRIVE_BASE_PATH, "Predictions/")
GAMELOG_LAKE_DIR = os.path.join(GDRIVE_BASE_PATH, "gamelog_lake/")
INITIAL_LINES_PATH = os.path.join(OUTPUT_DIR, "initial_nba_lines.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True); os.makedirs(GAMELOG_LAKE_DIR, exist_ok=True)

# *** 1. MARKET TOGGLES ***
ENABLE_MARKETS = {
    "Moneyline": True, "Spreads": True, "Totals": True,
    "Points": True, "Rebounds": True, "Assists": True, "Threes": True,
    "Combos": True, "DoubleDouble": True, "TripleDouble": True
}

FILTER_OUT_PLAYERS = True

_GAME_MAP = {"Moneyline": "h2h", "Spreads": "spreads", "Totals": "totals"}
_PROP_MAP = {
    "Points": {'player_points': 'PTS'}, "Rebounds": {'player_rebounds': 'REB'}, "Assists": {'player_assists': 'AST'},
    "Threes": {'player_threes': 'FG3M'},
    "Combos": {'player_points_rebounds_assists': 'PRA', 'player_points_rebounds': 'P+R', 'player_points_assists': 'P+A', 'player_rebounds_assists': 'R+A'},
    "DoubleDouble": {'player_double_double': 'DD'}, "TripleDouble": {'player_triple_double': 'TD'}
}

ACTIVE_GAME_MARKETS = [v for k, v in _GAME_MAP.items() if ENABLE_MARKETS.get(k, False)]
MARKET_NAME_MAP = {}
for category, enabled in ENABLE_MARKETS.items():
    if enabled and category in _PROP_MAP: MARKET_NAME_MAP.update(_PROP_MAP[category])
ALL_PROP_MARKETS_LIST = list(MARKET_NAME_MAP.keys())

# *** EXACT KEYS ***
ODDS_API_KEYS = [
    "9b96859c5e0781bb4c5809215a5ed94b", "f4cfcea7cc6120a97559f69d6049596e", "7b9405e2febc561e554b7169a42cefd1", "146dff42ae19a83208b059d4855095c9",
    "f8474479351ccc685ccae9f4716c23c5", "39756c093848b6968c899a4e1ded85f9", "9705a1f6b609305047eb8ec9243cc54a", "4217574ccdd90732483d98f24772f66b",
    "66953d96fcb2ec72a47e7f39901258e7", "94304f58fb95fd1d45159b9c8d292724", "e384411e342cac5c9408804c622703f1", "dab5c3da80b0415f64d4d94890974cdf",
    "9705a1f6b609305047eb8ec9243cc54a", "3aae94b51cc3dcb8e0623bf2633a695a", "a4e4d3eef7d360055b96a1351560817f", "1bfcac154fb2b490bf55a80a18275db1",
    "7b64d450f91079f60211cb33033594b4", "ec9dc1e1aa0543c1b3fa3d2576fabffd", "6bbfe433274aef6fee5aa5543ad69055", "1f5e9cdc190854c605b533917b2c9b82",
    "858b7234b5f77661345f205b1c5ec121", "39756c093848b6968c899a4e1ded85f9", "21caebe8058c338a7ba76f6c9fb550cb", "94304f58fb95fd1d45159b9c8d292724",
    "129b9d482b270ff3809e11b0c445ef5c", "a11767c08916b29f824150bffbbf6d9c", "e7e814c95b32c4a9fb5b72acefb7389b", "3a3cd8a78555c46afa842aa8bdf1455e",
    "3b9adf0f0e27964f58de1f9b4af3adfc", "f098da95884f42852df5ff12f5704741", "768fb25c7f33328bc53b61b81e607bd9"
]

SHARP_BOOKS = ["pinnacle", "betfair_ex_eu", "betonlineag", "bovada", "betus", "betrivers", "betmgm", "espnbet"]
RETAIL_BOOKS = ["draftkings", "fanduel"]
ALL_BOOKS_FOR_API = list(set(SHARP_BOOKS + RETAIL_BOOKS))
ROLLING_WINDOWS = [5, 10, 20]
CONFIG = {
    "GAMES_PLAYED_THRESHOLD_ACCLIMATED": 15,
    "GAMES_PLAYED_THRESHOLD_LOW_VOLUME": 20,
    "STEAM_THRESHOLD": 1.5,
    "RLM_CONFIDENCE": 0.65
}

# --- Part 2: Helper Functions ---
def normalize_player_name(name):
    if name is None: return None
    try: name = str(name)
    except: return None
    name = unidecode(name.lower()); name = re.sub(r'\s+(jr|sr|ii|iii|iv)\.?$', '', name); name = re.sub(r'[^\w\s]', '', name); return re.sub(r'\s+', ' ', name).strip()
def get_team_abbr_map(): return {team['full_name']: team['abbreviation'] for team in teams.get_teams()}

def convert_min_to_decimal(min_str):
    if isinstance(min_str, str) and ':' in min_str:
        try: minutes, seconds = map(int, min_str.split(':')); return minutes + seconds / 60.0
        except: return np.nan
    return pd.to_numeric(min_str, errors='coerce')

def derive_position(row):
    pts = row.get('Pred_PTS', 0); reb = row.get('Pred_REB', 0); ast = row.get('Pred_AST', 0)
    if reb >= 8.0: return 'C'
    if reb >= 6.0 and ast < 2.5: return 'C/F'
    if ast >= 6.0: return 'PG'
    if ast >= 4.5 and reb < 4.0: return 'PG'
    if reb >= 5.0 and pts >= 12.0: return 'F'
    if pts >= 10.0 and reb < 4.5: return 'G'
    return 'G/F'

def fetch_espn_nba_injuries():
    print("\nFetching live injury report from ESPN API (NBA)...")
    try:
        teams_url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams"
        teams_data = requests.get(teams_url, timeout=10).json()
        team_list = teams_data['sports'][0]['leagues'][0]['teams']
        all_injuries = []
        for team in tqdm(team_list, desc="Fetching Team Rosters for Injuries"):
            team_id = team['team']['id']
            roster_url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams/{team_id}?enable=roster"
            try: roster_data = requests.get(roster_url, timeout=10).json()
            except Exception: continue
            for player in roster_data.get('team', {}).get('athletes', []):
                if 'injuries' in player and player['injuries']:
                    injury_obj = player['injuries'][0]
                    status = injury_obj.get('status', 'Unknown')
                    all_injuries.append({'Player_ESPN': player.get('displayName'), 'Status': status, 'Injury Details': injury_obj.get('date', '') + " " + injury_obj.get('type', {}).get('description', '')})

        if not all_injuries: print("   ⚠️ WARNING: No injury entries found."); return pd.DataFrame()
        injury_df = pd.DataFrame(all_injuries)
        injury_df['match_key'] = injury_df['Player_ESPN'].apply(normalize_player_name)
        print(f"   ✅ Successfully fetched {len(injury_df)} injury entries.")
        return injury_df
    except Exception as e: print(f"   ❌ FAILED to fetch ESPN injuries: {e}"); return pd.DataFrame()

def get_live_team_metrics(season):
    print(f"   📊 Fetching Team Metrics for {season}...")
    metrics_dict = {}
    try:
        metrics_df = teamestimatedmetrics.TeamEstimatedMetrics(season=season, league_id='00', timeout=60).get_data_frames()[0]
        if not metrics_df.empty:
            cols = ['E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING', 'E_PACE', 'E_EFG_PCT', 'E_OREB_PCT', 'GP', 'W', 'L']
            metrics_dict = metrics_df.set_index('TEAM_ID')[cols].to_dict('index')
            return metrics_dict
    except Exception: pass

    if not metrics_dict:
        try:
            adv_stats = leaguedashteamstats.LeagueDashTeamStats(season=season, measure_type_detailed_defense='Advanced', timeout=60).get_data_frames()[0]
            if not adv_stats.empty:
                adv_stats['E_OFF_RATING'] = adv_stats['OFF_RATING']; adv_stats['E_DEF_RATING'] = adv_stats['DEF_RATING']
                adv_stats['E_NET_RATING'] = adv_stats['NET_RATING']; adv_stats['E_PACE'] = adv_stats['PACE']
                adv_stats['E_EFG_PCT'] = adv_stats['EFG_PCT']; adv_stats['E_OREB_PCT'] = adv_stats['OREB_PCT']
                cols = ['E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING', 'E_PACE', 'E_EFG_PCT', 'E_OREB_PCT', 'GP', 'W', 'L']
                metrics_dict = adv_stats.set_index('TEAM_ID')[cols].to_dict('index')
        except Exception: pass
    return metrics_dict

def get_defense_ranks(season):
    print("   🛡️ Fetching Opponent Defense Rankings (Raw & Calculated)...")
    try:
        # We fetch the raw 'Opponent' stats. This is the most reliable way.
        stats = leaguedashteamstats.LeagueDashTeamStats(
            season=season,
            measure_type_detailed_defense='Opponent',
            per_mode_detailed='PerGame',
            timeout=60
        ).get_data_frames()[0]

        if stats.empty: return {}, {}

        # Standardize column names (API varies sometimes)
        col_pts = 'OPP_PTS' if 'OPP_PTS' in stats.columns else 'PTS'
        col_reb = 'OPP_REB' if 'OPP_REB' in stats.columns else 'REB'
        col_ast = 'OPP_AST' if 'OPP_AST' in stats.columns else 'AST'
        col_3pm = 'OPP_FG3M' if 'OPP_FG3M' in stats.columns else 'FG3M'

        # *** MANUALLY CALCULATE RANKS FROM RAW STATS ***
        # Rank 1 = Lowest Value (Best Defense)
        stats['CALC_RANK_PTS'] = stats[col_pts].rank(method='min', ascending=True)
        stats['CALC_RANK_REB'] = stats[col_reb].rank(method='min', ascending=True)
        stats['CALC_RANK_AST'] = stats[col_ast].rank(method='min', ascending=True)
        stats['CALC_RANK_FG3M'] = stats[col_3pm].rank(method='min', ascending=True)

        ranks = {}
        raw_stats = {}

        for _, row in stats.iterrows():
            tid = str(row['TEAM_ID'])
            ranks[tid] = {
                'PTS': int(row['CALC_RANK_PTS']),
                'REB': int(row['CALC_RANK_REB']),
                'AST': int(row['CALC_RANK_AST']),
                'FG3M': int(row['CALC_RANK_FG3M'])
            }
            raw_stats[tid] = {
                'PTS': float(row.get(col_pts, 115.0)),
                'REB': float(row.get(col_reb, 44.0)),
                'AST': float(row.get(col_ast, 25.0)),
                'FG3M': float(row.get(col_3pm, 12.0))
            }
        return ranks, raw_stats
    except Exception as e:
        print(f"   ❌ FAILED to fetch defense ranks: {e}")
        return {}, {}

def get_nba_feng_ratings(season):
    print("   🧮 Calculating Feng/Massey Ratings...")
    try:
        games = leaguegamefinder.LeagueGameFinder(season_nullable=season, league_id_nullable='00', season_type_nullable='Regular Season', timeout=60).get_data_frames()[0]
        if games.empty: return {}
        games = games[games['WL'].isin(['W', 'L'])]
        games['GAME_DATE'] = pd.to_datetime(games['GAME_DATE'])
        team_ids = sorted(games['TEAM_ID'].unique())
        n_teams = len(team_ids)
        team_to_idx = {t: i for i, t in enumerate(team_ids)}
        M = np.zeros((n_teams, n_teams))
        p = np.zeros(n_teams)
        grouped = games.groupby('GAME_ID')
        for _, g_df in grouped:
            if len(g_df) != 2: continue
            row1 = g_df.iloc[0]; row2 = g_df.iloc[1]
            t1, t2 = row1['TEAM_ID'], row2['TEAM_ID']
            s1, s2 = row1['PTS'], row2['PTS']
            margin = s1 - s2
            i, j = team_to_idx[t1], team_to_idx[t2]
            p[i] += margin; p[j] -= margin
            M[i, i] += 1; M[j, j] += 1; M[i, j] -= 1; M[j, i] -= 1
        M[-1, :] = 1; p[-1] = 0
        ratings = np.linalg.solve(M, p)
        return {str(team_ids[i]): ratings[i] for i in range(n_teams)}
    except Exception as e:
        print(f"   ⚠️ Feng Ratings calculation failed: {e}")
        return {}

def calculate_prop_streak(player_id, prop_map, line):
    try:
        gamelog_file = os.path.join(GAMELOG_LAKE_DIR, f"{player_id}.csv")
        if not os.path.exists(gamelog_file): return 0, "-"
        df = pd.read_csv(gamelog_file)
        if df.empty: return 0, "-"
        df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
        df = df.sort_values(by='GAME_DATE', ascending=False).reset_index(drop=True)
        val_series = None
        if prop_map == 'PTS': val_series = df['PTS']
        elif prop_map == 'REB': val_series = df['REB']
        elif prop_map == 'AST': val_series = df['AST']
        elif prop_map == 'FG3M': val_series = df['FG3M']
        elif prop_map == 'PRA': val_series = df['PTS'] + df['REB'] + df['AST']
        elif prop_map == 'P+R': val_series = df['PTS'] + df['REB']
        elif prop_map == 'P+A': val_series = df['PTS'] + df['AST']
        elif prop_map == 'R+A': val_series = df['REB'] + df['AST']
        elif prop_map == 'DD': val_series = ((df['PTS']>=10).astype(int) + (df['REB']>=10).astype(int) + (df['AST']>=10).astype(int)) >= 2

        if val_series is None: return 0, "-"

        streak = 0
        for val in val_series:
            if prop_map == 'DD':
                if val: streak += 1
                else: break
            else:
                if val > line: streak += 1
                else: break

        l10_slice = val_series.head(10)
        hits_in_l10 = 0
        if prop_map == 'DD': hits_in_l10 = l10_slice.sum()
        else: hits_in_l10 = (l10_slice > line).sum()

        return streak, f"{int(hits_in_l10)}/10"
    except Exception: return 0, "-"

def calculate_head_to_head(player_id, prop_map, line, opponent_abbr, season_year):
    try:
        gamelog_file = os.path.join(GAMELOG_LAKE_DIR, f"{player_id}.csv")
        if not os.path.exists(gamelog_file): return "-"
        df = pd.read_csv(gamelog_file)
        if df.empty: return "-"
        df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
        season_start = pd.Timestamp(f"{season_year}-09-01")
        df = df[df['GAME_DATE'] >= season_start]
        if df.empty: return "0/0"
        target_opp = str(opponent_abbr).upper()
        abbr_map = {'PHX': 'PHO', 'PHO': 'PHX', 'GSW': 'GS', 'GS': 'GSW', 'NOP': 'NO', 'NO': 'NOP', 'SAS': 'SA', 'SA': 'SAS', 'NYK': 'NY', 'NY': 'NYK', 'UTA': 'UT', 'UT': 'UTA', 'WAS': 'WSH', 'WSH': 'WAS'}
        alt_opp = abbr_map.get(target_opp, target_opp)
        def is_matchup(m_str):
            if pd.isna(m_str): return False
            try:
                m_str = m_str.replace('.', ''); tokens = m_str.split()
                actual_opp = tokens[-1].upper()
                return actual_opp == target_opp or actual_opp == alt_opp
            except: return False
        df_vs = df[df['MATCHUP'].apply(is_matchup)]
        if df_vs.empty: return "0/0"
        val_series = None
        if prop_map == 'PTS': val_series = df_vs['PTS']
        elif prop_map == 'REB': val_series = df_vs['REB']
        elif prop_map == 'AST': val_series = df_vs['AST']
        elif prop_map == 'FG3M': val_series = df_vs['FG3M']
        elif prop_map == 'PRA': val_series = df_vs['PTS'] + df_vs['REB'] + df_vs['AST']
        elif prop_map == 'P+R': val_series = df_vs['PTS'] + df_vs['REB']
        elif prop_map == 'P+A': val_series = df_vs['PTS'] + df_vs['AST']
        elif prop_map == 'R+A': val_series = df_vs['REB'] + df_vs['AST']
        elif prop_map == 'DD': val_series = ((df_vs['PTS']>=10).astype(int) + (df_vs['REB']>=10).astype(int) + (df_vs['AST']>=10).astype(int)) >= 2
        if val_series is None: return "-"
        hits = 0
        if prop_map == 'DD': hits = val_series.sum()
        else: hits = (val_series > line).sum()
        return f"{int(hits)}/{len(df_vs)}"
    except Exception: return "-"

def get_live_schedule_from_nba_data(iso_date, season_year):
    schedule = {}; print(f"--- Fetching Schedule for {iso_date} via API ---")
    try:
        board = scoreboardv2.ScoreboardV2(game_date=iso_date, timeout=60); games = board.game_header.get_data_frame()
        if games.empty: print(f"Warning: NBA API returned no games for {iso_date}"); return {}
        valid_count = 0
        for _, game in games.iterrows():
            h_id = game['HOME_TEAM_ID']; v_id = game['VISITOR_TEAM_ID']
            if pd.isna(h_id) or pd.isna(v_id): continue
            try: h_str = str(int(h_id)); v_str = str(int(v_id))
            except: continue
            if not h_str.startswith('1610612') or not v_str.startswith('1610612'): continue
            schedule[h_str] = v_str; schedule[v_str] = h_str; valid_count += 1
        print(f"✅ Found {valid_count} valid games on the schedule."); return schedule
    except Exception as e: print(f"❌ Critical Error fetching schedule: {e}"); return {}

def get_draft_class_ids(season_year):
    try: df_draft = drafthistory.DraftHistory(season_year_nullable=season_year, timeout=60).get_data_frames()[0]; return set(df_draft['PERSON_ID'].unique())
    except Exception: return set()
def identify_team_changes(stats_last_season, stats_current_season):
    if stats_last_season.empty or stats_current_season.empty: return set()
    merged = pd.merge(stats_last_season[['PLAYER_ID', 'TEAM_ID']], stats_current_season[['PLAYER_ID', 'TEAM_ID']], on='PLAYER_ID', suffixes=('_last', '_current'))
    return set(merged[merged['TEAM_ID_last'] != merged['TEAM_ID_current']]['PLAYER_ID'].unique())
def get_player_experience(player_id, rookie_ids, traded_player_ids, stats_last_season, stats_current_season):
    gp_last = stats_last_season.loc[stats_last_season['PLAYER_ID'] == player_id, 'GP'].iloc[0] if not stats_last_season.loc[stats_last_season['PLAYER_ID'] == player_id].empty else 99
    gp_current = stats_current_season.loc[stats_current_season['PLAYER_ID'] == player_id, 'GP'].iloc[0] if not stats_current_season.loc[stats_current_season['PLAYER_ID'] == player_id].empty else 0
    is_rookie, is_traded = player_id in rookie_ids, player_id in traded_player_ids; is_acclimated = gp_current >= CONFIG["GAMES_PLAYED_THRESHOLD_ACCLIMATED"]
    if is_rookie: return "Traded Rookie" if is_traded and not is_acclimated else "Rookie"
    else: return "Traded Veteran" if is_traded and not is_acclimated else "Low Volume" if gp_last < CONFIG["GAMES_PLAYED_THRESHOLD_LOW_VOLUME"] else "Veteran"
def get_and_update_gamelogs(player_id, current_season, prev_season):
    gamelog_file = os.path.join(GAMELOG_LAKE_DIR, f"{player_id}.csv")
    if os.path.exists(gamelog_file):
        local_logs = pd.read_csv(gamelog_file); local_logs['GAME_DATE'] = pd.to_datetime(local_logs['GAME_DATE']); last_date = local_logs['GAME_DATE'].max()
        try:
            file_mod_time = datetime.fromtimestamp(os.path.getmtime(gamelog_file)).date()
            if file_mod_time == datetime.now().date(): return pd.read_csv(gamelog_file)
            time.sleep(0.4)
            new_logs = playergamelog.PlayerGameLog(player_id=player_id, date_from_nullable=last_date.strftime('%m/%d/%Y'), timeout=30).get_data_frames()[0]
            if not new_logs.empty:
                new_logs['GAME_DATE'] = pd.to_datetime(new_logs['GAME_DATE']); new_logs = new_logs[new_logs['GAME_DATE'] > last_date]
                if not new_logs.empty: updated_logs = pd.concat([new_logs, local_logs], ignore_index=True).sort_values(by="GAME_DATE", ascending=False).drop_duplicates(subset=['Game_ID']); updated_logs.to_csv(gamelog_file, index=False); return updated_logs
            return local_logs
        except Exception: return local_logs
    else:
        try:
            time.sleep(0.4)
            cur = playergamelog.PlayerGameLog(player_id=player_id, season=current_season, timeout=30).get_data_frames()[0]
            prev = playergamelog.PlayerGameLog(player_id=player_id, season=prev_season, timeout=30).get_data_frames()[0]
            full_logs = pd.concat([cur, prev], ignore_index=True).sort_values(by="GAME_DATE", ascending=False).drop_duplicates(subset=['Game_ID']); full_logs.to_csv(gamelog_file, index=False); return full_logs
        except Exception: return pd.DataFrame()

def calculate_live_features(logs):
    if logs.empty or len(logs) < 5: return {}
    logs_for_calc = logs.copy(); features = {}
    logs_for_calc['MIN'] = logs_for_calc['MIN'].apply(convert_min_to_decimal)
    base_stats = ['PTS', 'REB', 'AST', 'FGA', 'FG3A', 'FG3M', 'FTA', 'TOV']
    for col in base_stats:
        if col in logs_for_calc.columns: logs_for_calc[col] = pd.to_numeric(logs_for_calc[col], errors='coerce')
        else: logs_for_calc[col] = np.nan
    logs_for_calc.dropna(subset=['MIN'], inplace=True);
    if logs_for_calc.empty: return {}
    avg_minutes = logs_for_calc['MIN'].mean()
    weights = np.ones(len(logs_for_calc)); weights[logs_for_calc['MIN'] < 0.7 * avg_minutes] = 0.5; weights[logs_for_calc['MIN'] > 1.15 * avg_minutes] = 1.2
    logs_for_calc['WEIGHT'] = weights
    logs_safe = logs_for_calc[logs_for_calc['MIN'] > 0].copy()
    stats_for_ratios = ['PTS', 'REB', 'AST', 'FGA', 'FG3A', 'FTA', 'TOV', 'FG3M']
    for stat in stats_for_ratios:
        if stat in logs_safe.columns: logs_safe[f'{stat}_PM'] = logs_safe[stat] / logs_safe['MIN']
        else: logs_safe[f'{stat}_PM'] = 0
    stats_to_roll = ['PTS', 'REB', 'AST', 'MIN', 'FG3M', 'FG3A'] + [f'{s}_PM' for s in stats_for_ratios]
    for stat in stats_to_roll:
        if stat not in logs_safe.columns: continue
        for w in ROLLING_WINDOWS:
            recent_logs = logs_safe.head(w)
            features[f'W_ROLL_AVG_{stat}_{w}'] = np.average(recent_logs[stat].fillna(0), weights=recent_logs['WEIGHT']) if not recent_logs.empty else np.nan
        features[f'TREND_{stat}'] = features.get(f'W_ROLL_AVG_{stat}_{ROLLING_WINDOWS[0]}') - features.get(f'W_ROLL_AVG_{stat}_{ROLLING_WINDOWS[-1]}')
    for stat in ['PTS', 'REB', 'AST', 'FG3M']:
        if stat in logs_safe.columns:
            last_20 = logs_safe.head(20)
            if not last_20.empty:
                std_dev = last_20[stat].std(); mean_val = last_20[stat].mean()
                cv = (std_dev / mean_val) if mean_val > 0.5 else 9.9
                features[f'CV_{stat}'] = cv
            else: features[f'CV_{stat}'] = 9.9
    features['DAYS_OF_REST'] = (datetime.utcnow().replace(tzinfo=pytz.utc) - pd.to_datetime(logs_for_calc['GAME_DATE'].iloc[0]).tz_localize('UTC')).days
    for stat in stats_for_ratios: features[f'{stat}_PM'] = logs_safe.head(20)[f'{stat}_PM'].mean() if f'{stat}_PM' in logs_safe.columns else 0
    return features

def calculate_ou_probability(projection, line):
    if pd.isna(projection) or pd.isna(line) or projection <= 0: return 0.5, 0.5
    prob_over = 1 - poisson.cdf(k=line, mu=projection); prob_under = poisson.cdf(k=line - 1, mu=projection)
    return prob_over, prob_under

def get_previous_run_data(current_date_str, output_dir):
    today_file = os.path.join(output_dir, f"{current_date_str} NBA Player Props.xlsx")
    if os.path.exists(today_file):
        print(f"✅ Found earlier run from TODAY ({current_date_str}). Calculating Pro vs Info Window movement...")
        try: return pd.read_excel(today_file, sheet_name='DraftKings_EV_Bets')
        except: return pd.DataFrame()
    else:
        print(f"ℹ️ No earlier run found for TODAY ({current_date_str}). This is likely the 'Pro Window' (First Run).")
        return pd.DataFrame()

def apply_movement_and_ranking(curr_df, prev_df, sharp_props_map):
    if curr_df.empty: return curr_df
    if not prev_df.empty:
        curr_df['Player'] = curr_df['Player'].astype(str); prev_df['Player'] = prev_df['Player'].astype(str)
        if 'Bookmaker' in prev_df.columns:
            curr_df['Bookmaker'] = curr_df['Bookmaker'].astype(str); prev_df['Bookmaker'] = prev_df['Bookmaker'].astype(str)
            prev_clean = prev_df[['Player', 'Prop', 'Bookmaker', 'Edge (%)']].rename(columns={'Edge (%)': 'Prev_Edge'})
            merged = pd.merge(curr_df, prev_clean, on=['Player', 'Prop', 'Bookmaker'], how='left', suffixes=('', '_prev'))
        else:
            prev_clean = prev_df[['Player', 'Prop', 'Edge (%)']].rename(columns={'Edge (%)': 'Prev_Edge'})
            merged = pd.merge(curr_df, prev_clean, on=['Player', 'Prop'], how='left', suffixes=('', '_prev'))
    else: merged = curr_df.copy(); merged['Prev_Edge'] = np.nan

    def get_sharp_data(row):
        key = (row['match_key'], row['Prop'])
        sharp_lines_list = sharp_props_map.get(key, [])
        if sharp_lines_list:
            avg_sharp_line = sum(sharp_lines_list) / len(sharp_lines_list)
            diff = row['Projection'] - avg_sharp_line
            return round(avg_sharp_line, 1), diff
        return np.nan, np.nan
    merged[['Sharp Line', 'Proj vs Sharp']] = merged.apply(lambda r: pd.Series(get_sharp_data(r)), axis=1)

    def get_signal(row):
        curr_edge, prev_edge = row.get('Edge (%)', 0), row.get('Prev_Edge')
        if pd.isna(prev_edge): return "-"
        delta = curr_edge - prev_edge
        if delta > 1.5: return "PRICE IMPROVED"
        if delta < -1.5 and curr_edge > 5: return "SHARPS AGREE"
        return "-"
    merged['Sharp Signal'] = merged.apply(get_signal, axis=1)

    def get_consistency_badge(row):
        cv_pts = row.get('CV_PTS', 9.9); cv_reb = row.get('CV_REB', 9.9); cv_ast = row.get('CV_AST', 9.9)
        prop = row['Prop']
        cv = 9.9
        if prop == 'PTS': cv = cv_pts
        elif prop == 'REB': cv = cv_reb
        elif prop == 'AST': cv = cv_ast
        elif prop == 'FG3M': cv = row.get('CV_FG3M', 9.9)
        elif prop == 'PRA': cv = (cv_pts + cv_reb + cv_ast) / 3
        elif prop == 'P+R': cv = (cv_pts + cv_reb) / 2
        elif prop == 'P+A': cv = (cv_pts + cv_ast) / 2
        elif prop == 'R+A': cv = (cv_reb + cv_ast) / 2
        else: return "-"
        if cv > 5.0: return "-"
        if cv < 0.25: return "🔒 High"
        elif cv < 0.45: return "✅ Med"
        else: return "⚡ Volatile"
    merged['Consistency'] = merged.apply(get_consistency_badge, axis=1)

    def get_ranking(row):
        edge, signal = row.get('Edge (%)', 0), row['Sharp Signal']
        consistency = row.get('Consistency', '-')
        bet_type = row.get('Bet')
        has_sharp_line = not pd.isna(row.get('Sharp Line'))
        is_breakout_candidate = (edge > 25.0) and (row.get('Pred_MIN', 0) > 28) and (row.get('Line', 0) < 15) and (bet_type == 'Over')
        if is_breakout_candidate: return "🚀 BREAKOUT OPP"
        prob = row.get('Prob_Over', 0) if bet_type == 'Over' else row.get('Prob_Under', 0)
        if edge > 35.0: return "Avoid / Lottery"
        if has_sharp_line:
            sharp_line = row['Sharp Line']
            line = row['Line']
            if (bet_type == 'Over' and sharp_line > line) or (bet_type == 'Under' and sharp_line < line):
                if edge > 10.0: return "SHARP MISMATCH"
        if signal == 'PRICE IMPROVED' and edge > 15.0: return "💎 PRIME VALUE"
        if signal == 'PRICE IMPROVED': return "Most Profitable"
        if signal == 'SHARPS AGREE': return "Most Validated"
        if prob > 0.60 and consistency != "⚡ Volatile": return "Safest Bet"
        if edge > 10.0: return "Strong Value"
        if edge > 5.0: return "Solid Play"
        return "-"
    merged['Ranking'] = merged.apply(get_ranking, axis=1)
    return merged

# --- Part 5: Main Execution Workflow ---
def run_daily_projections():
    proj_df = pd.DataFrame()

    try:
        if not os.path.exists('/content/drive'): drive.mount('/content/drive', force_remount=True)
    except Exception as e: print(f"⚠️ Drive mount check skipped: {e}")

    eastern = pytz.timezone('US/Eastern')
    SIMULATION_DATE = None

    if 'SIMULATION_DATE' in globals() and globals()['SIMULATION_DATE'] is not None:
         try: datetime.strptime(globals()['SIMULATION_DATE'], "%Y-%m-%d"); SIMULATION_DATE = globals()['SIMULATION_DATE']
         except: SIMULATION_DATE = None

    if SIMULATION_DATE:
        live_now_et = datetime.strptime(f"{SIMULATION_DATE} 12:00:00", "%Y-%m-%d %H:%M:%S").astimezone(eastern)
        print(f"\n⚠️ RUNNING IN SIMULATION MODE for date: {SIMULATION_DATE}")
        iso_today_str = SIMULATION_DATE
        file_today_str = live_now_et.strftime('%m-%d-%Y')
    else:
        live_now_et = datetime.now(eastern)
        print(f"\n📅 Running Live Mode. Current ET Time: {live_now_et.strftime('%Y-%m-%d %H:%M:%S')}")
        iso_today_str = live_now_et.strftime('%Y-%m-%d')
        file_today_str = live_now_et.strftime('%m-%d-%Y')

    # --- TWO-RUN MODE SELECTION ---
    run_mode = ''
    while run_mode not in ['initial', 'final']:
        run_mode = input("Enter run mode ('initial' or 'final'): ").strip().lower()

    if run_mode == 'initial':
        print("\n🔵 INITIAL RUN DETECTED. Will save opening lines to CSV.")
    else:
        print("\n🔴 FINAL RUN DETECTED. Will compare vs Initial Lines for Steam/RLM/CLV.")

    if live_now_et.month >= 9: start_year = live_now_et.year; end_year = live_now_et.year + 1
    else: start_year = live_now_et.year - 1; end_year = live_now_et.year
    season = f"{start_year}-{str(end_year)[-2:]}"; prev_season = f"{start_year-1}-{str(start_year)[-2:]}"; season_year_for_url = str(start_year)

    print(f"Running projections for {iso_today_str} (Season: {season})")

    try:
        models = {'minutes': joblib.load(os.path.join(MODEL_DIR, "minutes_lgbm_model.pkl")), 'pts': joblib.load(os.path.join(MODEL_DIR, "pts_lgbm_model.pkl")), 'reb': joblib.load(os.path.join(MODEL_DIR, "reb_lgbm_model.pkl")), 'ast': joblib.load(os.path.join(MODEL_DIR, "ast_lgbm_model.pkl")), 'fg3m': joblib.load(os.path.join(MODEL_DIR, "fg3m_lgbm_model.pkl")), 'dd': joblib.load(os.path.join(MODEL_DIR, "dd_lgbm_model.pkl")), 'td': joblib.load(os.path.join(MODEL_DIR, "td_lgbm_model.pkl"))}
        imputers = {name: joblib.load(os.path.join(MODEL_DIR, f"{name}_imputer.pkl")) for name in ['minutes', 'stats']}
        min_feature_names = joblib.load(os.path.join(MODEL_DIR, "minutes_feature_names.pkl")); stat_feature_names = joblib.load(os.path.join(MODEL_DIR, "stats_feature_names.pkl")); mae_values = json.load(open(os.path.join(MODEL_DIR, "mae_values.json")))
        print("✅ All models, imputers, and MAE values loaded successfully.")
    except FileNotFoundError as e: print(f"CRITICAL ERROR: A required model file is missing. Please run Script 2. Details: {e}"); return

    injury_df = fetch_espn_nba_injuries()

    # *** LOAD PLAYER POSITIONS FROM EXCEL ***
    position_map = {}
    POSITIONS_FILE = os.path.join(GDRIVE_BASE_PATH, "Player Positions/2025-26 NBA Player Positions.xlsx")
    if os.path.exists(POSITIONS_FILE):
        try:
            print(f"Loading Player Positions from: {POSITIONS_FILE}")
            pos_df = pd.read_excel(POSITIONS_FILE)
            if 'Player' in pos_df.columns and 'Pos' in pos_df.columns:
                pos_df['match_key'] = pos_df['Player'].apply(normalize_player_name)
                position_map = pos_df.set_index('match_key')['Pos'].to_dict()
                print(f"✅ Successfully loaded {len(position_map)} player positions.")
            else: print("⚠️ Warning: Position file missing 'Player' or 'Pos' columns.")
        except Exception as e: print(f"⚠️ Error loading Position file: {e}")
    else: print(f"⚠️ Warning: Position file not found at {POSITIONS_FILE}. Using proxy logic.")

    scratched_players = []
    if not injury_df.empty:
        scratched_players = injury_df[injury_df['Status'].isin(['Out', 'Injured Reserve', 'Suspended'])]['match_key'].tolist()

    if scratched_players: print(f"   ℹ️ Detected {len(scratched_players)} ruled-out players. Adjusting team usage automatically...")

    print("\n--- Preparing Context & Stats ---")
    stats_current_season, stats_last_season = pd.DataFrame(), pd.DataFrame()
    for attempt in range(3):
        try:
            stats_current_season = leaguedashplayerstats.LeagueDashPlayerStats(season=season, timeout=120).get_data_frames()[0]
            stats_last_season = leaguedashplayerstats.LeagueDashPlayerStats(season=prev_season, timeout=120).get_data_frames()[0]
            if not stats_current_season.empty and not stats_last_season.empty: print("✅ Successfully fetched season stats."); break
        except Exception: time.sleep(5)

    if stats_current_season.empty: print("⚠️ Warning: Could not fetch current season stats.")

    # --- GET DEFENSE RANKS & RAW STATS (UPDATED) ---
    def_ranks, def_stats_raw = get_defense_ranks(season)
    if not def_ranks: print("   ⚠️ Warning: Defense Ranks are empty. API might be down or Season mismatch.")

    # --- GET FENG RATINGS ---
    feng_ratings = get_nba_feng_ratings(season)

    team_metrics = get_live_team_metrics(season)
    schedule = get_live_schedule_from_nba_data(iso_today_str, season_year_for_url) # USE ISO DATE HERE
    if not schedule: print("⚠️ Warning: NBA API returned 0 games. Will attempt to rebuild schedule from Betting Odds.")

    schedule = {str(k): str(v) for k, v in schedule.items() if k is not None and v is not None}
    if not stats_current_season.empty: stats_current_season['TEAM_ID'] = stats_current_season['TEAM_ID'].astype(str).str.split('.').str[0]

    rookie_ids = get_draft_class_ids(season_year_for_url); traded_player_ids = identify_team_changes(stats_last_season, stats_current_season)
    props_df, live_game_lines = pd.DataFrame(), {}
    key, game_events = None, []
    team_abbr_map = get_team_abbr_map()

    print("\n--- Running Initial Projections & Monte Carlo Sims ---")
    all_rosters = stats_current_season
    projection_rows, model_game_rows = [], []

    LEAGUE_AVG_PACE = 99.0
    LEAGUE_AVG_OFF = 115.0

    if team_metrics:
        paces = [m['E_PACE'] for m in team_metrics.values() if m['E_PACE'] > 0]
        offs = [m['E_OFF_RATING'] for m in team_metrics.values() if m['E_OFF_RATING'] > 0]
        if paces: LEAGUE_AVG_PACE = sum(paces) / len(paces)
        if offs: LEAGUE_AVG_OFF = sum(offs) / len(offs)

    if schedule:
        for team_id, opponent_id in tqdm(schedule.items(), desc="Generating Projections"):
            # --- 1. PLAYER PROJECTIONS ---
            roster = all_rosters[all_rosters['TEAM_ID'] == str(team_id)]
            team_scratches = [p for p in scratched_players if p in roster['PLAYER_NAME'].apply(normalize_player_name).tolist()]
            usage_boost_factor = 1.0 + (len(team_scratches) * 0.05)

            for _, row in roster.iterrows():
                if 'GP' in row and row['GP'] > 0 and (row['MIN'] / row['GP']) < 6.0: continue
                player_id, player_name, team_abbr = row['PLAYER_ID'], row['PLAYER_NAME'], row['TEAM_ABBREVIATION']
                match_key = normalize_player_name(player_name)
                logs = get_and_update_gamelogs(player_id, season, prev_season); features = calculate_live_features(logs)
                if not features: continue
                try: opp_id_int = int(opponent_id); opp_metrics = team_metrics.get(opp_id_int, {})
                except: opp_metrics = {}

                features['OPP_ALLOWED_PTS_AVG_30'] = opp_metrics.get('E_DEF_RATING', 115)
                features['OPP_ALLOWED_REB_AVG_30'] = 44.0; features['OPP_ALLOWED_AST_AVG_30'] = 25.0; features['OPP_ALLOWED_FG3M_AVG_30'] = 12.0

                if str(opponent_id) in def_stats_raw:
                    features['OPP_ALLOWED_PTS_AVG_30'] = def_stats_raw[str(opponent_id)]['PTS']
                    features['OPP_ALLOWED_REB_AVG_30'] = def_stats_raw[str(opponent_id)]['REB']
                    features['OPP_ALLOWED_AST_AVG_30'] = def_stats_raw[str(opponent_id)]['AST']
                    features['OPP_ALLOWED_FG3M_AVG_30'] = def_stats_raw[str(opponent_id)]['FG3M']

                features['VEGAS_SPREAD'] = 0; features['VEGAS_TOTAL'] = 225

                try:
                    min_input = pd.DataFrame([features])[min_feature_names]; pred_min = models['minutes'].predict(imputers['minutes'].transform(min_input))[0]; features['Proj_MIN'] = pred_min
                    if usage_boost_factor > 1.0:
                        features['FGA_PM'] = features.get('FGA_PM', 0) * usage_boost_factor
                        pred_min = pred_min * 1.02

                    stat_input = imputers['stats'].transform(pd.DataFrame([features])[stat_feature_names])
                    p_pts = models['pts'].predict(stat_input)[0]; p_reb = models['reb'].predict(stat_input)[0]
                    p_ast = models['ast'].predict(stat_input)[0]; p_3pm = models['fg3m'].predict(stat_input)[0]

                    temp_proj = {'Pred_PTS': p_pts, 'Pred_REB': p_reb, 'Pred_AST': p_ast}
                    if match_key in position_map: player_pos = position_map[match_key]
                    else: player_pos = derive_position(temp_proj)

                    proj = {
                        'PLAYER_ID': player_id, 'match_key': match_key, 'Player': player_name,
                        'Pos': player_pos, 'Experience': get_player_experience(player_id, rookie_ids, traded_player_ids, stats_last_season, stats_current_season),
                        'Team': team_abbr, 'Opponent': teams.find_team_name_by_id(int(opponent_id))['abbreviation'] if opponent_id else "UNK",
                        'OpponentID': str(opponent_id), 'Pred_MIN': pred_min, 'Pred_FGA_PM': features.get('FGA_PM', 0),
                        'Pred_PTS': p_pts, 'Pred_REB': p_reb, 'Pred_AST': p_ast, 'Pred_FG3M': p_3pm
                    }
                    proj['CV_PTS'] = features.get('CV_PTS', 9.9); proj['CV_REB'] = features.get('CV_REB', 9.9)
                    proj['CV_AST'] = features.get('CV_AST', 9.9); proj['CV_FG3M'] = features.get('CV_FG3M', 9.9)
                    clf_features = np.array([proj['Pred_PTS'], proj['Pred_REB'], proj['Pred_AST']]).reshape(1, -1)
                    proj['Pred_DD'] = models['dd'].predict_proba(clf_features)[0][1] * 100; proj['Pred_TD'] = models['td'].predict_proba(clf_features)[0][1] * 100
                    proj['Pred_PRA'] = proj['Pred_PTS'] + proj['Pred_REB'] + proj['Pred_AST']; proj['Pred_P+R'] = proj['Pred_PTS'] + proj['Pred_REB']; proj['Pred_P+A'] = proj['Pred_PTS'] + proj['Pred_AST']; proj['Pred_R+A'] = proj['Pred_REB'] + proj['Pred_AST']
                    projection_rows.append(proj)
                except Exception: continue

            # --- 2. GAME ENGINE (MONTE CARLO UPGRADE) ---
            home_stats, away_stats = team_metrics.get(int(team_id), {}), team_metrics.get(int(opponent_id), {})
            if home_stats and away_stats:
                h_off, a_off = home_stats.get('E_OFF_RATING', 110), away_stats.get('E_OFF_RATING', 110)
                h_def, a_def = home_stats.get('E_DEF_RATING', 110), away_stats.get('E_DEF_RATING', 110)
                h_pace, a_pace = home_stats.get('E_PACE', 99), away_stats.get('E_PACE', 99)
                poss = (h_pace + a_pace) / 2; HCA = 2.5

                # FIXED MODEL TOTAL LOGIC (POINTS PER GAME)
                # Formula: (OffRtg * OppDefRtg / LeagueAvgOff) * (Poss / 100)
                h_exp = ((h_off * a_def) / LEAGUE_AVG_OFF) * (poss / 100.0) + (HCA/2)
                a_exp = ((a_off * h_def) / LEAGUE_AVG_OFF) * (poss / 100.0) - (HCA/2)

                SIMS = 5000
                scores = np.random.multivariate_normal([h_exp, a_exp], [[169, 50], [50, 169]], size=SIMS)
                h_sc, a_sc = scores[:,0], scores[:,1]
                model_spread = -(np.mean(h_sc) - np.mean(a_sc))
                model_total = np.mean(h_sc) + np.mean(a_sc)
                model_win_prob = np.mean(h_sc > a_sc)
                pyth_h = h_off**13.91 / (h_off**13.91 + h_def**13.91)
                pyth_a = a_off**13.91 / (a_off**13.91 + a_def**13.91)
                log5_prob = (pyth_h - pyth_h * pyth_a) / (pyth_h + pyth_a - 2 * pyth_h * pyth_a)
                actual_win_pct = home_stats.get('W', 0) / home_stats.get('GP', 1)
                luck_metric = actual_win_pct - pyth_h
                r_home, r_away = feng_ratings.get(str(team_id), 0), feng_ratings.get(str(opponent_id), 0)
                feng_spread = -(r_home - r_away + HCA) if r_home and r_away else None

                model_game_rows.append({
                    'TeamID': int(team_id),
                    'Model Spread': round(model_spread, 1),
                    'Model Total': round(model_total, 1),
                    'Model Win %': model_win_prob,
                    'Log5 %': log5_prob,
                    'Feng Spread': round(feng_spread, 1) if feng_spread else "-",
                    'Luck': luck_metric,
                    'NetRtg': home_stats.get('E_NET_RATING'),
                    'eFG%': home_stats.get('E_EFG_PCT'),
                    'OREB%': home_stats.get('E_OREB_PCT')
                })

    proj_df = pd.DataFrame(projection_rows)
    if proj_df.empty:
        print(f"\n❌ NO PROJECTIONS GENERATED."); return
    else:
        proj_df['match_key'] = proj_df['Player'].apply(normalize_player_name)
        print(f"✅ Success: Generated {len(proj_df)} player projections. Proceeding to fetch odds...")

    # 5. FETCH ODDS
    game_lines_rows = []
    sharp_props_map = {}

    # Store initial lines for saving later if mode is 'initial'
    initial_lines_to_save = []

    for k in ODDS_API_KEYS:
        try:
            r = requests.get(f"https://api.the-odds-api.com/v4/sports/basketball_nba/odds?apiKey={k}&regions=us&markets=h2h", timeout=10); r.raise_for_status()
            if int(r.headers.get('x-requests-remaining', 0)) > 20: key = k; print(f"Using API key ending in ...{k[-4:]}"); break
        except Exception: continue

    if key:
        print("--- Fetching Sharp & Retail Odds ---")
        try:
            game_markets_to_fetch = ",".join(ACTIVE_GAME_MARKETS)
            books_str = ",".join(ALL_BOOKS_FOR_API)
            url_games = f"https://api.the-odds-api.com/v4/sports/basketball_nba/odds/?apiKey={key}&regions=us,eu&markets={game_markets_to_fetch or 'h2h'}&bookmakers={books_str}"
            response_games = requests.get(url_games, timeout=15); response_games.raise_for_status(); games_data = response_games.json()
            game_events = [game for game in games_data if parser.isoparse(game["commence_time"]).astimezone(eastern).strftime('%Y-%m-%d') == iso_today_str] # USE ISO DATE HERE
            game_events.sort(key=lambda x: x["commence_time"])

            for game in game_events:
                home_team_list = teams.find_teams_by_full_name(game['home_team']); away_team_list = teams.find_teams_by_full_name(game['away_team'])
                if not home_team_list or not away_team_list: continue
                home_team, away_team = home_team_list[0], away_team_list[0]
                home_id, away_id = str(home_team['id']), str(away_team['id'])
                if home_id not in schedule: schedule[home_id] = away_id; schedule[away_id] = home_id

                home_abbr, away_abbr = home_team['abbreviation'], away_team['abbreviation']
                spreads, totals, mls = [], [], []; retail_spread, retail_total, retail_ml, retail_book = np.nan, np.nan, np.nan, "Consensus"
                home_spread_sharp_list, home_spread_retail_val = [], None; over_sharp_list, over_retail_val = [], None

                for book in game.get('bookmakers', []):
                    if book['key'] in SHARP_BOOKS:
                        for market in book.get('markets', []):
                            if market['key'] == 'spreads': spreads.append(market['outcomes'][0]['point'])
                            elif market['key'] == 'totals': totals.append(market['outcomes'][0]['point'])
                            elif market['key'] == 'h2h':
                                for o in market['outcomes']:
                                    if o['name'] == game['home_team']: mls.append(o['price'])
                    if book['key'] in ['draftkings', 'fanduel']:
                        if pd.isna(retail_spread): retail_book = book['title']
                        for market in book.get('markets', []):
                            if market['key'] == 'spreads' and pd.isna(retail_spread): retail_spread = market['outcomes'][0]['point']
                            if market['key'] == 'totals' and pd.isna(retail_total): retail_total = market['outcomes'][0]['point']
                            if market['key'] == 'h2h' and pd.isna(retail_ml):
                                for o in market['outcomes']:
                                    if o['name'] == game['home_team']: retail_ml = o['price']

                    for market in book.get('markets', []):
                        if market['key'] == 'spreads':
                            for o in market['outcomes']:
                                if o['name'] == game['home_team']:
                                    if book['key'] in SHARP_BOOKS: home_spread_sharp_list.append(o['point'])
                                    elif book['key'] in RETAIL_BOOKS and home_spread_retail_val is None: home_spread_retail_val = o['point']
                        if market['key'] == 'totals':
                            for o in market['outcomes']:
                                if o['name'] == 'Over':
                                    if book['key'] in SHARP_BOOKS: over_sharp_list.append(o['point'])
                                    elif book['key'] in RETAIL_BOOKS and over_retail_val is None: over_retail_val = o['point']

                sharp_spread = sum(spreads)/len(spreads) if spreads else None; sharp_total = sum(totals)/len(totals) if totals else None
                try: live_game_lines[int(home_id)] = {'spread': retail_spread, 'total': retail_total}; live_game_lines[int(away_id)] = {'spread': -retail_spread if not pd.isna(retail_spread) else np.nan, 'total': retail_total}
                except: pass

                # *** RECORD INITIAL LINES IF MODE IS INITIAL ***
                if run_mode == 'initial':
                    initial_lines_to_save.append({
                        'GameID': game['id'], 'Home': home_abbr, 'Away': away_abbr,
                        'Initial_Sharp_Spread': sharp_spread, 'Initial_Sharp_Total': sharp_total
                    })

                model_data = next((item for item in model_game_rows if item["TeamID"] == int(home_id)), {})
                spread_diff = abs(retail_spread - sharp_spread) if (not pd.isna(retail_spread) and sharp_spread is not None) else 0
                total_diff = abs(retail_total - sharp_total) if (not pd.isna(retail_total) and sharp_total is not None) else 0
                signal = "SHARP MISMATCH" if (spread_diff >= 1.5 or total_diff >= 1.5) else ("Diff Found" if (spread_diff >= 0.5) else "-")

                # *** NEW MARKET MOVEMENT LOGIC (FINAL RUN) ***
                movement_signal, clv_spread, clv_total = "-", "-", "-"
                if run_mode == 'final':
                    try:
                        initial_df = pd.read_csv(INITIAL_LINES_PATH)
                        # Match on GameID first, fallback to Teams if needed (Odds API IDs change rarely but safe to check)
                        matched_row = initial_df[initial_df['GameID'] == game['id']]
                        if matched_row.empty:
                            matched_row = initial_df[(initial_df['Home'] == home_abbr) & (initial_df['Away'] == away_abbr)]

                        if not matched_row.empty:
                            init_spread = matched_row.iloc[0]['Initial_Sharp_Spread']
                            init_total = matched_row.iloc[0]['Initial_Sharp_Total']

                            # Spread Analysis
                            if pd.notna(init_spread) and sharp_spread is not None:
                                spread_move = sharp_spread - init_spread
                                clv_spread = round(spread_move * -1, 1) # If line moves -5 -> -6, move is -1. If I bet -5, I gained 1pt CLV.
                                if abs(spread_move) >= CONFIG["STEAM_THRESHOLD"]:
                                    movement_signal = f"🔥 Steam Chase ({'Home' if spread_move < 0 else 'Away'})"
                                elif 'Model Win %' in model_data:
                                    win_prob = model_data['Model Win %']
                                    if win_prob > CONFIG["RLM_CONFIDENCE"] and spread_move > 0.5: # Favored Home, Line moved Away
                                        movement_signal = "⚠️ RLM Alert (Fade Home)"
                                    elif (1-win_prob) > CONFIG["RLM_CONFIDENCE"] and spread_move < -0.5: # Favored Away, Line moved Home
                                        movement_signal = "⚠️ RLM Alert (Fade Away)"

                            # Total Analysis
                            if pd.notna(init_total) and sharp_total is not None:
                                clv_total = round(sharp_total - init_total, 1)

                    except Exception as e: pass

                pick_spread, pick_total, pick_ml = "-", "-", "-"
                if home_spread_retail_val is not None and home_spread_sharp_list:
                    avg_home_sharp = sum(home_spread_sharp_list) / len(home_spread_sharp_list)
                    if abs(home_spread_retail_val - avg_home_sharp) >= 1.0:
                        pick_spread = f"{home_abbr} {home_spread_retail_val:+.1f}" if avg_home_sharp < home_spread_retail_val else f"{away_abbr} {-1 * home_spread_retail_val:+.1f}"

                if over_retail_val is not None and over_sharp_list:
                    avg_over_sharp = sum(over_sharp_list) / len(over_sharp_list)
                    if abs(over_retail_val - avg_over_sharp) >= 1.0:
                        pick_total = f"Over {over_retail_val}" if avg_over_sharp > over_retail_val else f"Under {over_retail_val}"

                if not pd.isna(retail_ml) and 'Model Win %' in model_data:
                    if retail_ml > 1.0 and model_data['Model Win %'] > ((1.0 / retail_ml) + 0.05): pick_ml = f"{home_abbr} ML"

                game_lines_rows.append({
                    'Game': f"{away_abbr} @ {home_abbr}",
                    'Time': parser.isoparse(game["commence_time"]).astimezone(eastern).strftime('%-I:%M %p'),
                    'Retail Book': retail_book,
                    'Retail Spread': retail_spread, 'Sharp Spread': round(sharp_spread, 1) if sharp_spread else "-",
                    'Model Spread': model_data.get('Model Spread', '-'),
                    'Feng Spread': model_data.get('Feng Spread', '-'),
                    'Spread Diff': round(spread_diff, 1), 'Spread Pick': pick_spread,
                    'Retail Total': retail_total, 'Sharp Total': round(sharp_total, 1) if sharp_total else "-",
                    'Model Total': model_data.get('Model Total', '-'),
                    'Total Diff': round(total_diff, 1), 'Total Pick': pick_total,
                    'Retail ML': retail_ml,
                    'Model Win %': f"{model_data.get('Model Win %', 0)*100:.1f}%" if 'Model Win %' in model_data else "-",
                    'Log5 Win %': f"{model_data.get('Log5 %', 0)*100:.1f}%" if 'Log5 %' in model_data else "-",
                    'ML Pick': pick_ml,
                    'Home Luck': round(model_data.get('Luck', 0), 3) if 'Luck' in model_data else "-",
                    'Home Net Rtg': model_data.get('NetRtg', '-'),
                    'Movement Signal': movement_signal,
                    'CLV (Spread)': clv_spread,
                    'CLV (Total)': clv_total,
                    'Signal': signal
                })
        except Exception as e: print(f"Failed to fetch game list/lines: {e}"); game_events = []

        if run_mode == 'initial':
            if initial_lines_to_save:
                pd.DataFrame(initial_lines_to_save).to_csv(INITIAL_LINES_PATH, index=False)
                print(f"\n✅ INITIAL RUN COMPLETE. Saved {len(initial_lines_to_save)} game lines to {INITIAL_LINES_PATH}")
                return # Stop execution for initial run
            else:
                print("\n❌ NO GAMES FOUND TO SAVE.")
                return

        # ... (Props fetching code remains same, omitted for brevity but assumed present in flow)
        if game_events:
            all_props_raw = []
            market_query = ','.join(ALL_PROP_MARKETS_LIST) if ALL_PROP_MARKETS_LIST else ""
            if market_query:
                for game in tqdm(game_events, desc="Fetching Player Props"):
                    event_id = game['id']
                    away_abbr = team_abbr_map.get(game['away_team'], '?'); home_abbr = team_abbr_map.get(game['home_team'], '?')
                    books_str = ",".join(ALL_BOOKS_FOR_API)
                    url_props = f"https://api.the-odds-api.com/v4/sports/basketball_nba/events/{event_id}/odds?apiKey={key}&regions=us,eu&markets={market_query}&bookmakers={books_str}"
                    try:
                        r = requests.get(url_props, timeout=20); r.raise_for_status(); payload = r.json()
                        for book in payload.get('bookmakers', []):
                            for market in book.get('markets', []):
                                for outcome in market.get('outcomes', []):
                                    player_name = outcome['description']
                                    norm_name = normalize_player_name(player_name)
                                    prop_name = MARKET_NAME_MAP.get(market['key'])
                                    line = outcome.get('point')

                                    if book['key'] in SHARP_BOOKS and line is not None:
                                        dict_key = (norm_name, prop_name)
                                        if dict_key not in sharp_props_map: sharp_props_map[dict_key] = []
                                        sharp_props_map[dict_key].append(line)
                                        continue

                                    if book['key'] in ['draftkings', 'fanduel']:
                                        all_props_raw.append({'Game': f"{away_abbr} @ {home_abbr}", 'Game_Time': parser.isoparse(game["commence_time"]).astimezone(eastern).strftime('%-I:%M %p'), 'Sort_Time': game["commence_time"], 'Player': player_name, 'Prop': prop_name, 'Line': line, 'Decimal_Odds': outcome.get('price'), 'OutcomeName': outcome['name'], 'Bookmaker': book['title']})
                    except Exception: continue
            if all_props_raw:
                props_df = pd.DataFrame(all_props_raw)
                props_df['match_key'] = props_df['Player'].apply(normalize_player_name)

    if props_df.empty: print("\n--- Displaying Projections Only (No Betting Odds Found) ---"); display(proj_df[['Player', 'Experience', 'Team', 'Opponent', 'Pred_MIN', 'Pred_PTS', 'Pred_REB', 'Pred_AST', 'Pred_FG3M']].round(2).head(20)); return

    print("\n--- Running Value Analysis & Filtering Injuries ---")
    ou_props = props_df[props_df['OutcomeName'].isin(['Over', 'Under'])].copy(); yn_props = props_df[props_df['OutcomeName'].isin(['Yes', 'No'])].copy()
    merged_ou = pd.DataFrame()
    if not ou_props.empty:
        ou_pivot = ou_props.pivot_table(index=['match_key', 'Player', 'Prop', 'Line', 'Bookmaker', 'Game', 'Game_Time', 'Sort_Time'], columns='OutcomeName', values='Decimal_Odds').reset_index().rename(columns={'Over': 'Over_Odds', 'Under': 'Under_Odds'})
        merged_ou = pd.merge(ou_pivot, proj_df.drop(columns=['Player'], errors='ignore'), on='match_key', how='inner')
        if not merged_ou.empty:
            merged_ou['Projection'] = merged_ou.apply(lambda r: r.get(f"Pred_{r['Prop']}", np.nan), axis=1)
            merged_ou['Proj Range Low'] = (merged_ou['Projection'] - merged_ou['Prop'].map(mae_values)).clip(lower=0); merged_ou['Proj Range High'] = merged_ou['Projection'] + merged_ou['Prop'].map(mae_values)
            merged_ou['Proj Range'] = merged_ou.apply(lambda r: f"{r['Proj Range Low']:.1f}-{r['Proj Range High']:.1f}", axis=1)
            merged_ou[['Prob_Over', 'Prob_Under']] = merged_ou.apply(lambda r: calculate_ou_probability(r['Projection'], r['Line']), axis=1, result_type='expand')
            def determine_bet(row):
                if row['Line'] < row['Proj Range Low'] and row['Prob_Over'] > 0.53: return 'Over'
                elif row['Line'] > row['Proj Range High'] and row['Prob_Under'] > 0.53: return 'Under'
                return 'Pass'
            merged_ou['Bet'] = merged_ou.apply(determine_bet, axis=1)
            merged_ou['Edge (%)'] = np.where(merged_ou['Bet'] == 'Over', (merged_ou['Prob_Over'] * merged_ou['Over_Odds']) - 1, np.where(merged_ou['Bet'] == 'Under', (merged_ou['Prob_Under'] * merged_ou['Under_Odds']) - 1, 0)) * 100

    merged_yn = pd.DataFrame()
    if not yn_props.empty:
        yn_pivot = yn_props.pivot_table(index=['match_key', 'Player', 'Prop', 'Bookmaker', 'Game', 'Game_Time', 'Sort_Time'], columns='OutcomeName', values='Decimal_Odds').reset_index().rename(columns={'Yes': 'Yes_Odds', 'No': 'No_Odds'})
        merged_yn = pd.merge(yn_pivot, proj_df.drop(columns=['Player'], errors='ignore'), on='match_key', how='inner')
        if not merged_yn.empty:
            merged_yn['Projection'] = merged_yn.apply(lambda r: r.get(f"Pred_{r['Prop']}", np.nan), axis=1)
            merged_yn['Model Score (%)'] = merged_yn['Projection']; merged_yn['Implied Prob (%)'] = (1 / merged_yn['Yes_Odds']) * 100
            merged_yn['Edge (%)'] = merged_yn['Model Score (%)'] - merged_yn['Implied Prob (%)']; merged_yn['Bet'] = np.where(merged_yn['Edge (%)'] > 5, 'Yes', 'Pass')

    full_analysis_df = pd.concat([merged_ou, merged_yn])
    if not injury_df.empty and not full_analysis_df.empty: full_analysis_df = pd.merge(full_analysis_df, injury_df[['match_key', 'Status']], on='match_key', how='left'); full_analysis_df['Status'] = full_analysis_df['Status'].fillna('Active')
    elif not full_analysis_df.empty: full_analysis_df['Status'] = 'Active'

    if not full_analysis_df.empty and FILTER_OUT_PLAYERS: full_analysis_df = full_analysis_df[~full_analysis_df['Status'].isin(['Out', 'Injured Reserve', 'Suspended'])]

    prev_run_df = get_previous_run_data(file_today_str, OUTPUT_DIR) # USE FILE DATE HERE

    full_analysis_df = apply_movement_and_ranking(full_analysis_df, prev_run_df, sharp_props_map)

    # --- MAP GAME SIGNALS TO PROPS ---
    if game_lines_rows:
        game_move_map = {}
        for row in game_lines_rows:
            game_move_map[row['Game']] = {
                'Game Move': row.get('Movement Signal', '-'),
                'Game CLV': row.get('CLV (Spread)', '-')
            }

        full_analysis_df['Game Move'] = full_analysis_df['Game'].map(lambda x: game_move_map.get(x, {}).get('Game Move', '-'))
        full_analysis_df['Game CLV'] = full_analysis_df['Game'].map(lambda x: game_move_map.get(x, {}).get('Game CLV', '-'))

    # ==============================================================================
    # --- NEW FEATURE: STREAKS, DEFENSE RANKS & HEAD-TO-HEAD ---
    # ==============================================================================
    if not full_analysis_df.empty:
        print("\n--- Calculating Player Streaks, Opponent Ranks & H2H ---")

        streak_data = []
        rank_data = []
        h2h_data = []

        for idx, row in tqdm(full_analysis_df.iterrows(), total=len(full_analysis_df), desc="Analyzing Trends"):
            # 1. Defense Rank (FIXED)
            opp_id = row.get('OpponentID')
            opp_abbr = row.get('Opponent')
            prop_cat = row.get('Prop')
            rank_val = 99

            # Map props to the keys in def_ranks
            stat_key_map = {'Points': 'PTS', 'Rebounds': 'REB', 'Assists': 'AST', 'Threes': 'FG3M'}

            # FIXED: Force string lookup for OpponentID
            if opp_id and str(opp_id) in def_ranks:
                if prop_cat in stat_key_map:
                    rank_val = def_ranks[str(opp_id)][stat_key_map[prop_cat]]
                elif prop_cat in ['Combos', 'DoubleDouble']: # Approximate using Points Rank
                     rank_val = def_ranks[str(opp_id)]['PTS']

            rank_data.append(rank_val)

            # Setup Map for Logs
            pid = row.get('PLAYER_ID')
            line = row.get('Line')
            log_map = {'Points': 'PTS', 'Rebounds': 'REB', 'Assists': 'AST', 'Threes': 'FG3M', 'Combos': 'PRA'}
            mapped_prop = log_map.get(prop_cat, prop_cat)
            if prop_cat == 'DoubleDouble': mapped_prop = 'DD'
            if 'Pts+Reb+Ast' in str(row.get('OutcomeName', '')): mapped_prop = 'PRA'
            elif 'Pts+Reb' in str(row.get('OutcomeName', '')): mapped_prop = 'P+R'
            elif 'Pts+Ast' in str(row.get('OutcomeName', '')): mapped_prop = 'P+A'
            elif 'Reb+Ast' in str(row.get('OutcomeName', '')): mapped_prop = 'R+A'

            # 2. Streaks (10 Game Window)
            s_curr, s_l10 = calculate_prop_streak(pid, mapped_prop, line)
            streak_str = f"W{s_curr} ({s_l10})"
            streak_data.append(streak_str)

            # 3. Head-to-Head (vs Opponent this season)
            h2h_val = calculate_head_to_head(pid, mapped_prop, line, opp_abbr, start_year)
            h2h_data.append(h2h_val)

        full_analysis_df['Opp Rank'] = rank_data
        full_analysis_df['Streak'] = streak_data
        full_analysis_df['vs Opp'] = h2h_data

        def fmt_rank(r):
            if r == 99: return "-"
            if r >= 25: return f"{r} (Easy)"
            if r <= 5: return f"{r} (Hard)"
            return str(r)

        full_analysis_df['Opp Rank'] = full_analysis_df['Opp Rank'].apply(fmt_rank)

    # ==============================================================================

    if not full_analysis_df.empty:
        full_analysis_df['Projection'] = full_analysis_df['Projection'].round(2)
        if 'Edge (%)' in full_analysis_df.columns: full_analysis_df['Edge (%)'] = full_analysis_df['Edge (%)'].round(1)
        if 'Proj vs Sharp' in full_analysis_df.columns: full_analysis_df['Proj vs Sharp'] = full_analysis_df['Proj vs Sharp'].astype(float).round(1)

        final_bets = full_analysis_df[full_analysis_df['Bet'] != 'Pass'].copy()
    else: final_bets = pd.DataFrame()

    out_file = os.path.join(OUTPUT_DIR, f"{file_today_str} NBA Player Props.xlsx") # USE FILE DATE HERE

    with pd.ExcelWriter(out_file, engine='xlsxwriter') as writer:
        workbook  = writer.book
        header_fmt = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'bg_color': '#D9D9D9', 'border': 1})
        center_fmt = workbook.add_format({'align': 'center', 'valign': 'vcenter', 'border': 1})
        green_fmt = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        red_fmt = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        orange_fmt = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C5700', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        blue_fmt = workbook.add_format({'bg_color': '#BDD7EE', 'font_color': '#1F4E78', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        purple_fmt = workbook.add_format({'bg_color': '#CCC0DA', 'font_color': '#3F3F76', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        perfect_fmt = workbook.add_format({'bg_color': '#00FF00', 'font_color': '#000000', 'align': 'center', 'valign': 'vcenter', 'border': 1, 'bold': True})
        yellow_fmt = workbook.add_format({'bg_color': '#FFFF00', 'font_color': '#000000', 'align': 'center', 'valign': 'vcenter', 'border': 1}) # For Steam

        green_row_fmt = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        blue_row_fmt = workbook.add_format({'bg_color': '#BDD7EE', 'font_color': '#1F4E78', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        orange_row_fmt = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C5700', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        red_row_fmt = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        purple_row_fmt = workbook.add_format({'bg_color': '#CCC0DA', 'font_color': '#3F3F76', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        yellow_row_fmt = workbook.add_format({'bg_color': '#FFFF00', 'font_color': '#000000', 'align': 'left', 'valign': 'vcenter', 'border': 1})

        ws_howto = workbook.add_worksheet("HOW TO USE")
        ws_howto.set_column(0, 0, 30); ws_howto.set_column(1, 1, 120); ws_howto.freeze_panes(1, 0)
        row = 0
        ws_howto.write(row, 0, "Item", header_fmt); ws_howto.write(row, 1, "Description", header_fmt); row += 1

        # *** NEW SECTION: Market Movement (Context & Props) ***
        ws_howto.write(row, 0, "NEW: Market Analysis (Game & Props Context)", header_fmt); ws_howto.write(row, 1, "Applies to 'Game_Lines' AND Player Prop Sheets", header_fmt); row += 1
        ws_howto.write(row, 0, "Movement: 🔥 Steam Chase", yellow_row_fmt); ws_howto.write(row, 1, "Massive, sudden line move (>1.5 pts) caused by sharp money. Strongest confirmation signal.", yellow_row_fmt); row += 1
        ws_howto.write(row, 0, "Movement: ⚠️ RLM Alert", orange_row_fmt); ws_howto.write(row, 1, "Reverse Line Movement. Line moved AGAINST the heavy favorite. Major trap warning.", orange_row_fmt); row += 1
        ws_howto.write(row, 0, "CLV (Closing Line Value)", green_row_fmt); ws_howto.write(row, 1, "Green (>0) means you beat the market. Red (<0) means the line moved against you.", green_row_fmt); row += 1
        ws_howto.write(row, 0, "Strategy: Prop Correlation", blue_row_fmt); ws_howto.write(row, 1, "If betting a Player OVER, look for 'Steam Chase' on their Team. Strong team performance correlates with player performance.", blue_row_fmt); row += 1
        ws_howto.write(row, 0, "Signal: SHARP MISMATCH", purple_row_fmt); ws_howto.write(row, 1, "Retail line significantly differs from Sharp Book (Pinnacle/Betfair) line.", purple_row_fmt); row += 1

        row += 1
        ws_howto.write(row, 0, "Ranking: SHARP MISMATCH", purple_row_fmt); ws_howto.write(row, 1, "Massive Value + Price Improved. The line moved in your favor AND the Edge is huge.", purple_row_fmt); row += 1
        ws_howto.write(row, 0, "Ranking: Most Profitable", green_row_fmt); ws_howto.write(row, 1, "Price Improved. Line moved in your favor since yesterday.", green_row_fmt); row += 1
        ws_howto.write(row, 0, "Ranking: Most Validated", orange_row_fmt); ws_howto.write(row, 1, "Sharps Agree. Line moved against you, but model still sees value.", orange_row_fmt); row += 1
        ws_howto.write(row, 0, "Ranking: Safest Bet", blue_row_fmt); ws_howto.write(row, 1, "High Probability (>60%) of winning regardless of odds.", blue_row_fmt); row += 1
        ws_howto.write(row, 0, "Ranking: Avoid / Lottery", red_row_fmt); ws_howto.write(row, 1, "Suspiciously high edge (>35%). Likely an injury risk.", red_row_fmt); row += 1
        ws_howto.write(row, 0, "Ranking: PRIME VALUE", blue_row_fmt); ws_howto.write(row, 1, "High Edge + Price Improved, but no Sharp Line confirmation available yet.", blue_row_fmt); row += 1
        ws_howto.write(row, 0, "Ranking: BREAKOUT OPP", red_row_fmt); ws_howto.write(row, 1, "Model projects Big Minutes for a low-line player. High Upside but volatile.", red_row_fmt); row += 1

        row += 1
        ws_howto.write(row, 0, "BETTING SCHEDULE", header_fmt); row += 1
        ws_howto.write(row, 0, "4 Hours Out (Pro Window)", blue_row_fmt); ws_howto.write(row, 1, "Sportsbooks raise limits. Professional money enters. Run as 'initial' mode.", blue_row_fmt); row += 1
        ws_howto.write(row, 0, "90 Minutes Out (Info Window)", orange_row_fmt); ws_howto.write(row, 1, "Injury reports released. Run as 'final' mode to see Steam/RLM.", orange_row_fmt); row += 1

        row += 1
        ws_howto.write(row, 0, "MODEL ACCURACY RANKING", header_fmt); row += 1
        ws_howto.write(row, 0, "1. Threes (FG3M)", green_row_fmt); ws_howto.write(row, 1, "Most Accurate (MAE ~0.13). Trust small edges here. Green Highlight at > 5%.", green_row_fmt); row += 1
        ws_howto.write(row, 0, "2. Assists (AST)", green_row_fmt); ws_howto.write(row, 1, "Very Accurate (MAE ~0.28). Great for Point Guard props. Green Highlight at > 8%.", green_row_fmt); row += 1
        ws_howto.write(row, 0, "3. Rebounds (REB)", blue_row_fmt); ws_howto.write(row, 1, "Strong (MAE ~0.50). Focus on Centers/consistent rebounders. Green Highlight at > 8%.", blue_row_fmt); row += 1
        ws_howto.write(row, 0, "4. Points (PTS)", orange_row_fmt); ws_howto.write(row, 1, "Highest Variance (MAE ~1.15). Only bet if Edge > 10% or it's a Breakout scenario. Green Highlight at > 12%.", orange_row_fmt); row += 1

        row += 1
        ws_howto.write(row, 0, "CONSISTENCY GUIDE", header_fmt); row += 1
        ws_howto.write(row, 0, "Consistency: 🔒 High", green_row_fmt); ws_howto.write(row, 1, "Reliable Player. Low Variance (CV < 0.25). Safe to trust projection.", green_row_fmt); row += 1
        ws_howto.write(row, 0, "Consistency: ⚡ Volatile", red_row_fmt); ws_howto.write(row, 1, "Boom or Bust. High Variance (CV > 0.45). Bet with caution even if edge is high.", red_row_fmt); row += 1

        sheet_config = ['DraftKings_EV_Bets', 'DraftKings_EV_SGP', 'FanDuel_EV_Bets', 'FanDuel_EV_SGP']
        for sheet_name in sheet_config:
            bookmaker = 'DraftKings' if 'DraftKings' in sheet_name else 'FanDuel'
            is_sgp = 'SGP' in sheet_name
            book_df = final_bets[final_bets['Bookmaker'] == bookmaker].copy() if not final_bets.empty else pd.DataFrame()
            if not book_df.empty:
                # *** CUSTOM COLUMN ORDER - ADDED GAME MOVE/CLV ***
                out_cols = [
                    'Game_Time', 'Game', 'Status', 'Experience', 'Team', 'Pos', 'Player', 'Prop',
                    'Line', 'Bet', 'Sharp Line', 'Proj vs Sharp',
                    'Projection', 'Proj Range', 'Edge (%)',
                    'Consistency', 'Opp Rank', 'Streak', 'vs Opp',
                    'Game Move', 'Game CLV', # NEW COLUMNS
                    'Sharp Signal', 'Ranking'
                ]

                if 'Sort_Time' in book_df.columns:
                    book_df['sort_rank'] = book_df['Ranking'].map({
                        "SHARP MISMATCH": 0, "💎 PRIME VALUE": 1, "Most Profitable": 2, "Most Validated": 3, "Safest Bet": 4,
                        "🚀 BREAKOUT OPP": 5, "Strong Value": 6, "Solid Play": 7, "Avoid / Lottery": 99
                    }).fillna(99)

                    if is_sgp: final_df = book_df.sort_values(['Sort_Time', 'Game', 'sort_rank', 'Edge (%)'], ascending=[True, True, True, False]).reindex(columns=out_cols)
                    else: final_df = book_df.sort_values(['sort_rank', 'Edge (%)'], ascending=[True, False]).reindex(columns=out_cols)
                else: final_df = book_df.reindex(columns=out_cols)

                final_df.to_excel(writer, sheet_name=sheet_name, index=False, startrow=1, header=False)
                ws = writer.sheets[sheet_name]; ws.freeze_panes(1, 0)
                ws.autofilter(0, 0, len(final_df), len(out_cols) - 1)
                for col, val in enumerate(final_df.columns):
                    ws.write(0, col, val, header_fmt)
                    max_len = len(str(val))
                    for cell_val in final_df[val].astype(str): max_len = max(max_len, len(cell_val))
                    ws.set_column(col, col, max_len + 4, center_fmt)

                last_row = len(final_df) + 1
                def get_col(name): return xl_col_to_name(out_cols.index(name))

                # Highlight Sharp Signal when data is present (PURPLE)
                ws.conditional_format(f'{get_col("Sharp Signal")}2:{get_col("Sharp Signal")}{last_row}', {'type': 'text', 'criteria': 'not containing', 'value': '-', 'format': purple_fmt})

                # Highlight Good Bets (Green) / Bad Bets (Red) for Edge and Bet columns
                ws.conditional_format(f'{get_col("Bet")}2:{get_col("Bet")}{last_row}', {'type': 'text', 'criteria': 'not containing', 'value': 'Pass', 'format': green_fmt})
                ws.conditional_format(f'{get_col("Edge (%)")}2:{get_col("Edge (%)")}{last_row}', {'type': 'cell', 'criteria': '>', 'value': 8, 'format': green_fmt})
                ws.conditional_format(f'{get_col("Edge (%)")}2:{get_col("Edge (%)")}{last_row}', {'type': 'cell', 'criteria': '<', 'value': 0, 'format': red_fmt})

                # Ranking Formats
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'SHARP', 'format': purple_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Profitable', 'format': green_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'PRIME', 'format': blue_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Validated', 'format': orange_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Safest', 'format': blue_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Avoid', 'format': red_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'BREAKOUT', 'format': red_fmt})

                # Status/Consistency Formats
                ws.conditional_format(f'{get_col("Consistency")}2:{get_col("Consistency")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'High', 'format': green_fmt})
                ws.conditional_format(f'{get_col("Consistency")}2:{get_col("Consistency")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Volatile', 'format': red_fmt})
                ws.conditional_format(f'{get_col("Status")}2:{get_col("Status")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Questionable', 'format': orange_fmt})

                # Opp Rank Formats
                ws.conditional_format(f'{get_col("Opp Rank")}2:{get_col("Opp Rank")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Easy', 'format': green_fmt})
                ws.conditional_format(f'{get_col("Opp Rank")}2:{get_col("Opp Rank")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Hard', 'format': red_fmt})

                # H2H / Trend Formats
                for h2h_val in ['1/1', '2/2', '3/3', '4/4']:
                    ws.conditional_format(f'{get_col("vs Opp")}2:{get_col("vs Opp")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': h2h_val, 'format': green_fmt})
                for trend_val in ['8/10', '9/10', '10/10']:
                    ws.conditional_format(f'{get_col("Streak")}2:{get_col("Streak")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': trend_val, 'format': green_fmt})

                # NEW: Game Move & CLV Formats
                ws.conditional_format(f'{get_col("Game Move")}2:{get_col("Game Move")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Steam', 'format': yellow_fmt})
                ws.conditional_format(f'{get_col("Game Move")}2:{get_col("Game Move")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'RLM', 'format': orange_fmt})
                ws.conditional_format(f'{get_col("Game CLV")}2:{get_col("Game CLV")}{last_row}', {'type': 'cell', 'criteria': '>', 'value': 0, 'format': green_fmt})
                ws.conditional_format(f'{get_col("Game CLV")}2:{get_col("Game CLV")}{last_row}', {'type': 'cell', 'criteria': '<', 'value': 0, 'format': red_fmt})

                # *** PERFECT BET FORMULA ***
                base_criteria = f'AND(OR(ISNUMBER(SEARCH("SHARP", ${get_col("Ranking")}2)), ISNUMBER(SEARCH("Validated", ${get_col("Ranking")}2))), ${get_col("Edge (%)")}2>=8, ${get_col("Edge (%)")}2<=30, OR(ISNUMBER(SEARCH("AGREE", ${get_col("Sharp Signal")}2)), ISNUMBER(SEARCH("IMPROVED", ${get_col("Sharp Signal")}2))), ISERROR(SEARCH("Hard", ${get_col("Opp Rank")}2)))'
                pos_criteria = f'IF(${get_col("Prop")}2="Assists", ISNUMBER(SEARCH("G", ${get_col("Pos")}2)), IF(${get_col("Prop")}2="Rebounds", OR(ISNUMBER(SEARCH("F", ${get_col("Pos")}2)), ISNUMBER(SEARCH("C", ${get_col("Pos")}2))), TRUE))'
                final_formula = f'=AND({base_criteria}, {pos_criteria})'
                ws.conditional_format(f'{get_col("Player")}2:{get_col("Player")}{last_row}', {'type': 'formula', 'criteria': final_formula, 'format': perfect_fmt})

            else: writer.book.add_worksheet(sheet_name).write(0, 0, "No bets found.")

        if game_lines_rows and ACTIVE_GAME_MARKETS:
            gl_df = pd.DataFrame(game_lines_rows)
            # Add new columns to export
            gl_cols = ['Time', 'Game', 'Retail Book', 'Retail Spread', 'Sharp Spread', 'Model Spread', 'Feng Spread', 'Spread Diff', 'Spread Pick', 'Retail Total', 'Sharp Total', 'Model Total', 'Total Diff', 'Total Pick', 'Retail ML', 'Model Win %', 'Log5 Win %', 'ML Pick', 'Home Luck', 'Movement Signal', 'CLV (Spread)', 'CLV (Total)', 'Signal']
            gl_df = gl_df.reindex(columns=gl_cols).sort_values(['Signal', 'Spread Diff'], ascending=[False, False])
            gl_df.to_excel(writer, sheet_name='Game_Lines', index=False); ws_gl = writer.sheets['Game_Lines']
            ws_gl.freeze_panes(1, 0)
            ws_gl.autofilter(0, 0, len(gl_df), len(gl_cols) - 1)
            for col, val in enumerate(gl_df.columns):
                ws_gl.write(0, col, val, header_fmt)
                ws_gl.set_column(col, col, 15 + 4, center_fmt)

            last_row = len(gl_df) + 1

            # Formats for New Columns
            sig_col = xl_col_to_name(gl_cols.index('Signal'))
            move_col = xl_col_to_name(gl_cols.index('Movement Signal'))
            clv_s_col = xl_col_to_name(gl_cols.index('CLV (Spread)'))
            pick_s_col = xl_col_to_name(gl_cols.index('Spread Pick'))
            diff_s_col = xl_col_to_name(gl_cols.index('Spread Diff'))

            ws_gl.conditional_format(f'{sig_col}2:{sig_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'SHARP', 'format': purple_fmt})
            ws_gl.conditional_format(f'{sig_col}2:{sig_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Diff', 'format': orange_fmt})

            # New Highlights for Picks/Diffs
            ws_gl.conditional_format(f'{pick_s_col}2:{pick_s_col}{last_row}', {'type': 'text', 'criteria': 'not containing', 'value': '-', 'format': green_fmt})
            ws_gl.conditional_format(f'{diff_s_col}2:{diff_s_col}{last_row}', {'type': 'cell', 'criteria': '>=', 'value': 1.0, 'format': green_fmt})

            # New Highlights for Steam/RLM/CLV
            ws_gl.conditional_format(f'{move_col}2:{move_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Steam', 'format': yellow_fmt})
            ws_gl.conditional_format(f'{move_col}2:{move_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'RLM', 'format': orange_fmt})
            ws_gl.conditional_format(f'{clv_s_col}2:{clv_s_col}{last_row}', {'type': 'cell', 'criteria': '>', 'value': 0, 'format': green_fmt})
            ws_gl.conditional_format(f'{clv_s_col}2:{clv_s_col}{last_row}', {'type': 'cell', 'criteria': '<', 'value': 0, 'format': red_fmt})

    print(f"\n✅ Ultimate Analysis Exported: {out_file}")

    if not final_bets.empty:
        print("\n" + "="*60); print("🏆 BEST BET PER GAME"); print("="*60)
        best_card = final_bets.copy()
        best_card['Win_Prob'] = np.where(best_card['Bet'] == 'Over', best_card['Prob_Over'], best_card['Prob_Under'])
        best_card['Priority'] = best_card['Sharp Signal'].apply(lambda x: 1 if x in ['PRICE IMPROVED', 'SHARPS AGREE'] else 0)
        best_card = best_card.sort_values(['Game', 'Priority', 'Edge (%)'], ascending=[True, False, False])
        quick_card = best_card.groupby('Game').head(1).sort_values('Game_Time')
        quick_card['Win_Prob'] = (quick_card['Win_Prob'] * 100).round(1).astype(str) + "%"
        quick_card['Edge (%)'] = quick_card['Edge (%)'].round(1).astype(str) + "%"
        display(quick_card[['Game_Time', 'Game', 'Player', 'Status', 'Prop', 'Bet', 'Consistency', 'Line', 'Edge (%)', 'Win_Prob', 'Sharp Signal']].rename(columns={'Sharp Signal': 'Signal'}))

    if final_bets.empty: print("\nNOTE: No bets met the strict criteria.")
    print("\n✅ Script Completed. Exiting...")

if __name__ == "__main__":
    pd.set_option('display.max_rows', 100); pd.set_option('display.max_columns', None)
    run_daily_projections()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.0/319.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 4.9 MB/s eta 0:00:00

📅 Running Live Mode. Current ET Time: 2026-02-04 21:43:22
Enter run mode ('initial' or 'final'): Final

🔴 FINAL RUN DETECTED. Will compare vs Initial Lines for Steam/RLM/CLV.
Running projections for 2026-02-04 (Season: 2025-26)
✅ All models, imputers, and MAE values loaded successfully.

Fetching live injury report from ESPN API (NBA)...


Fetching Team Rosters for Injuries: 100%|██████████| 30/30 [00:05<00:00,  5.52it/s]


   ✅ Successfully fetched 136 injury entries.
Loading Player Positions from: /content/drive/MyDrive/NBA/Player Positions/2025-26 NBA Player Positions.xlsx
✅ Successfully loaded 509 player positions.
   ℹ️ Detected 99 ruled-out players. Adjusting team usage automatically...

--- Preparing Context & Stats ---
✅ Successfully fetched season stats.
   🛡️ Fetching Opponent Defense Rankings (Raw & Calculated)...
   🧮 Calculating Feng/Massey Ratings...
   📊 Fetching Team Metrics for 2025-26...
--- Fetching Schedule for 2026-02-04 via API ---
✅ Found 7 valid games on the schedule.

--- Running Initial Projections & Monte Carlo Sims ---


Generating Projections: 100%|██████████| 14/14 [05:40<00:00, 24.33s/it]


✅ Success: Generated 208 player projections. Proceeding to fetch odds...
Using API key ending in ...d94b
--- Fetching Sharp & Retail Odds ---


...


--- Running Value Analysis & Filtering Injuries ---
ℹ️ No earlier run found for TODAY (02-04-2026). This is likely the 'Pro Window' (First Run).

--- Calculating Player Streaks, Opponent Ranks & H2H ---


Analyzing Trends: 100%|██████████| 541/541 [00:12<00:00, 43.21it/s]



✅ Ultimate Analysis Exported: /content/drive/MyDrive/NBA/Predictions/02-04-2026 NBA Player Props.xlsx

🏆 BEST BET PER GAME


,Game_Time,Game,Player,Status,Prop,Bet,Consistency,Line,Edge (%),Win_Prob,Signal
94,10:40 PM,CLE @ LAC,Dean Wade,Active,P+R,Over,⚡ Volatile,10.5,100.7%,99.4%,-
357,7:13 PM,DEN @ NYK,Nikola Jokic,Active,FG3M,Over,⚡ Volatile,1.5,157.3%,62.8%,-
19,7:42 PM,MIN @ TOR,Anthony Edwards,Active,FG3M,Over,⚡ Volatile,1.5,258.5%,85.4%,-
329,8:11 PM,NOP @ MIL,Kyle Kuzma,Active,FG3M,Over,⚡ Volatile,0.5,214.3%,64.1%,-
197,9:43 PM,OKC @ SAS,Jaylin Williams,Active,PTS,Under,⚡ Volatile,15.5,96.3%,99.1%,-



✅ Script Completed. Exiting...


In [ ]:
# ==============================================================================
# SCRIPT 3: NBA Professional Sharp & ML Engine (v97 - Complete & Restored)
# Features: 10-Game Streaks, Robust H2H, Monte Carlo Game Engine, All Helpers Restored
# ==============================================================================

# --- Part 0: Setup & Installs ---
!pip install -q xlsxwriter nba_api scikit-learn pandas joblib pytz unidecode requests lightgbm python-dateutil tqdm openpyxl scipy

import os
import re
import time
import joblib
import warnings
import requests
import json
import numpy as np
import pandas as pd
import pytz
from datetime import datetime, timedelta
from dateutil import parser
from unidecode import unidecode
from tqdm import tqdm
from decimal import Decimal
from scipy.stats import poisson, norm
from google.colab import drive
from IPython.display import display, clear_output
from xlsxwriter.utility import xl_col_to_name

from nba_api.stats.endpoints import playergamelog, leaguedashplayerstats, teamestimatedmetrics, drafthistory, scoreboardv2, leaguedashteamstats, leaguegamefinder
from nba_api.stats.static import teams, players

warnings.filterwarnings("ignore")

# --- Part 1: Configuration ---
GDRIVE_BASE_PATH = "/content/drive/MyDrive/NBA/"
MODEL_DIR = os.path.join(GDRIVE_BASE_PATH, "Training Model Advanced/")
OUTPUT_DIR = os.path.join(GDRIVE_BASE_PATH, "Predictions/")
GAMELOG_LAKE_DIR = os.path.join(GDRIVE_BASE_PATH, "gamelog_lake/")
os.makedirs(OUTPUT_DIR, exist_ok=True); os.makedirs(GAMELOG_LAKE_DIR, exist_ok=True)

# *** 1. MARKET TOGGLES ***
ENABLE_MARKETS = {
    "Moneyline": True, "Spreads": True, "Totals": True,
    "Points": True, "Rebounds": True, "Assists": True, "Threes": True,
    "Combos": True, "DoubleDouble": True, "TripleDouble": True
}

FILTER_OUT_PLAYERS = True

_GAME_MAP = {"Moneyline": "h2h", "Spreads": "spreads", "Totals": "totals"}
_PROP_MAP = {
    "Points": {'player_points': 'PTS'}, "Rebounds": {'player_rebounds': 'REB'}, "Assists": {'player_assists': 'AST'},
    "Threes": {'player_threes': 'FG3M'},
    "Combos": {'player_points_rebounds_assists': 'PRA', 'player_points_rebounds': 'P+R', 'player_points_assists': 'P+A', 'player_rebounds_assists': 'R+A'},
    "DoubleDouble": {'player_double_double': 'DD'}, "TripleDouble": {'player_triple_double': 'TD'}
}

ACTIVE_GAME_MARKETS = [v for k, v in _GAME_MAP.items() if ENABLE_MARKETS.get(k, False)]
MARKET_NAME_MAP = {}
for category, enabled in ENABLE_MARKETS.items():
    if enabled and category in _PROP_MAP: MARKET_NAME_MAP.update(_PROP_MAP[category])
ALL_PROP_MARKETS_LIST = list(MARKET_NAME_MAP.keys())

# *** EXACT KEYS ***
ODDS_API_KEYS = [
    "9b96859c5e0781bb4c5809215a5ed94b", "f4cfcea7cc6120a97559f69d6049596e", "7b9405e2febc561e554b7169a42cefd1", "146dff42ae19a83208b059d4855095c9",
    "f8474479351ccc685ccae9f4716c23c5", "39756c093848b6968c899a4e1ded85f9", "9705a1f6b609305047eb8ec9243cc54a", "4217574ccdd90732483d98f24772f66b",
    "66953d96fcb2ec72a47e7f39901258e7", "94304f58fb95fd1d45159b9c8d292724", "e384411e342cac5c9408804c622703f1", "dab5c3da80b0415f64d4d94890974cdf",
    "9705a1f6b609305047eb8ec9243cc54a", "3aae94b51cc3dcb8e0623bf2633a695a", "a4e4d3eef7d360055b96a1351560817f", "1bfcac154fb2b490bf55a80a18275db1",
    "7b64d450f91079f60211cb33033594b4", "ec9dc1e1aa0543c1b3fa3d2576fabffd", "6bbfe433274aef6fee5aa5543ad69055", "1f5e9cdc190854c605b533917b2c9b82",
    "858b7234b5f77661345f205b1c5ec121", "39756c093848b6968c899a4e1ded85f9", "21caebe8058c338a7ba76f6c9fb550cb", "94304f58fb95fd1d45159b9c8d292724",
    "129b9d482b270ff3809e11b0c445ef5c", "a11767c08916b29f824150bffbbf6d9c", "e7e814c95b32c4a9fb5b72acefb7389b", "3a3cd8a78555c46afa842aa8bdf1455e",
    "3b9adf0f0e27964f58de1f9b4af3adfc", "f098da95884f42852df5ff12f5704741", "768fb25c7f33328bc53b61b81e607bd9"
]

SHARP_BOOKS = ["pinnacle", "betfair_ex_eu", "betonlineag", "bovada", "betus", "betrivers", "betmgm", "espnbet"]
RETAIL_BOOKS = ["draftkings", "fanduel"]
ALL_BOOKS_FOR_API = list(set(SHARP_BOOKS + RETAIL_BOOKS))
ROLLING_WINDOWS = [5, 10, 20]; CONFIG = { "GAMES_PLAYED_THRESHOLD_ACCLIMATED": 15, "GAMES_PLAYED_THRESHOLD_LOW_VOLUME": 20 }

# --- Part 2: Helper Functions ---
def normalize_player_name(name):
    if name is None: return None
    try: name = str(name)
    except: return None
    name = unidecode(name.lower()); name = re.sub(r'\s+(jr|sr|ii|iii|iv)\.?$', '', name); name = re.sub(r'[^\w\s]', '', name); return re.sub(r'\s+', ' ', name).strip()
def get_team_abbr_map(): return {team['full_name']: team['abbreviation'] for team in teams.get_teams()}

def convert_min_to_decimal(min_str):
    if isinstance(min_str, str) and ':' in min_str:
        try: minutes, seconds = map(int, min_str.split(':')); return minutes + seconds / 60.0
        except: return np.nan
    return pd.to_numeric(min_str, errors='coerce')

# *** Smart Proxy Position Logic ***
def derive_position(row):
    pts = row.get('Pred_PTS', 0); reb = row.get('Pred_REB', 0); ast = row.get('Pred_AST', 0)
    if reb >= 8.0: return 'C'
    if reb >= 6.0 and ast < 2.5: return 'C/F'
    if ast >= 6.0: return 'PG'
    if ast >= 4.5 and reb < 4.0: return 'PG'
    if reb >= 5.0 and pts >= 12.0: return 'F'
    if pts >= 10.0 and reb < 4.5: return 'G'
    return 'G/F'

def fetch_espn_nba_injuries():
    print("\nFetching live injury report from ESPN API (NBA)...")
    try:
        teams_url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams"
        teams_data = requests.get(teams_url, timeout=10).json()
        team_list = teams_data['sports'][0]['leagues'][0]['teams']
        all_injuries = []
        for team in tqdm(team_list, desc="Fetching Team Rosters for Injuries"):
            team_id = team['team']['id']
            roster_url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams/{team_id}?enable=roster"
            try: roster_data = requests.get(roster_url, timeout=10).json()
            except Exception: continue
            for player in roster_data.get('team', {}).get('athletes', []):
                if 'injuries' in player and player['injuries']:
                    injury_obj = player['injuries'][0]
                    status = injury_obj.get('status', 'Unknown')
                    all_injuries.append({'Player_ESPN': player.get('displayName'), 'Status': status, 'Injury Details': injury_obj.get('date', '') + " " + injury_obj.get('type', {}).get('description', '')})

        if not all_injuries: print("   ⚠️ WARNING: No injury entries found."); return pd.DataFrame()
        injury_df = pd.DataFrame(all_injuries)
        injury_df['match_key'] = injury_df['Player_ESPN'].apply(normalize_player_name)
        print(f"   ✅ Successfully fetched {len(injury_df)} injury entries.")
        return injury_df
    except Exception as e: print(f"   ❌ FAILED to fetch ESPN injuries: {e}"); return pd.DataFrame()

# --- Robust Team Metrics (Fail-Safe) ---
def get_live_team_metrics(season):
    print(f"   📊 Fetching Team Metrics for {season}...")
    metrics_dict = {}

    # ATTEMPT 1: TeamEstimatedMetrics (Preferred)
    try:
        metrics_df = teamestimatedmetrics.TeamEstimatedMetrics(season=season, league_id='00', timeout=60).get_data_frames()[0]
        if not metrics_df.empty:
            cols = ['E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING', 'E_PACE', 'E_EFG_PCT', 'E_OREB_PCT', 'GP', 'W', 'L']
            metrics_dict = metrics_df.set_index('TEAM_ID')[cols].to_dict('index')
            print(f"      ✅ Successfully fetched Estimated Metrics.")
            return metrics_dict
    except Exception: pass

    # ATTEMPT 2: LeagueDashTeamStats (Fallback - Advanced)
    if not metrics_dict:
        print("      ⚠️ Attempting Fallback to Standard Advanced Stats...")
        try:
            adv_stats = leaguedashteamstats.LeagueDashTeamStats(
                season=season,
                measure_type_detailed_defense='Advanced',
                timeout=60
            ).get_data_frames()[0]

            if not adv_stats.empty:
                # Map Standard columns to the expected E_ columns
                adv_stats['E_OFF_RATING'] = adv_stats['OFF_RATING']
                adv_stats['E_DEF_RATING'] = adv_stats['DEF_RATING']
                adv_stats['E_NET_RATING'] = adv_stats['NET_RATING']
                adv_stats['E_PACE'] = adv_stats['PACE']
                adv_stats['E_EFG_PCT'] = adv_stats['EFG_PCT']
                adv_stats['E_OREB_PCT'] = adv_stats['OREB_PCT']

                cols = ['E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING', 'E_PACE', 'E_EFG_PCT', 'E_OREB_PCT', 'GP', 'W', 'L']
                metrics_dict = adv_stats.set_index('TEAM_ID')[cols].to_dict('index')
                print(f"      ✅ Successfully fetched Advanced Stats (Fallback).")
            else:
                print("      ❌ Error: Advanced Stats returned empty.")
        except Exception as e:
            print(f"      ❌ Failed to fetch fallback stats: {e}")

    return metrics_dict

# --- Fetch Team Defense Rankings (DUAL PARAM FETCH) ---
def get_defense_ranks(season):
    print("   🛡️ Fetching Opponent Defense Rankings (Calculating Manually)...")
    stats = pd.DataFrame()
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(season=season, per_mode_detailed='PerGame', measure_type_detailed_defense='Opponent', timeout=60).get_data_frames()[0]
    except Exception: pass
    if stats.empty:
        try: stats = leaguedashteamstats.LeagueDashTeamStats(season=season, per_mode_detailed='PerGame', measure_type_nullable='Opponent', timeout=60).get_data_frames()[0]
        except Exception: pass

    if stats.empty: return {}, {}

    try:
        col_pts = 'OPP_PTS' if 'OPP_PTS' in stats.columns else 'PTS'
        col_reb = 'OPP_REB' if 'OPP_REB' in stats.columns else 'REB'
        col_ast = 'OPP_AST' if 'OPP_AST' in stats.columns else 'AST'
        col_3pm = 'OPP_FG3M' if 'OPP_FG3M' in stats.columns else 'FG3M'

        if col_pts in stats.columns: stats['RANK_PTS'] = stats[col_pts].rank(ascending=True)
        else: stats['RANK_PTS'] = 15
        if col_reb in stats.columns: stats['RANK_REB'] = stats[col_reb].rank(ascending=True)
        else: stats['RANK_REB'] = 15
        if col_ast in stats.columns: stats['RANK_AST'] = stats[col_ast].rank(ascending=True)
        else: stats['RANK_AST'] = 15
        if col_3pm in stats.columns: stats['RANK_FG3M'] = stats[col_3pm].rank(ascending=True)
        else: stats['RANK_FG3M'] = 15

        ranks = {}
        raw_stats = {}
        for _, row in stats.iterrows():
            tid = str(row['TEAM_ID'])
            ranks[tid] = {'PTS': int(row['RANK_PTS']), 'REB': int(row['RANK_REB']), 'AST': int(row['RANK_AST']), 'FG3M': int(row['RANK_FG3M'])}
            raw_stats[tid] = {'PTS': float(row.get(col_pts, 115.0)), 'REB': float(row.get(col_reb, 44.0)), 'AST': float(row.get(col_ast, 25.0)), 'FG3M': float(row.get(col_3pm, 12.0))}
        return ranks, raw_stats
    except Exception as e: return {}, {}

# --- Feng (Massey) Ratings using Linear Algebra ---
def get_nba_feng_ratings(season):
    print("   🧮 Calculating Feng/Massey Ratings (Strength of Schedule)...")
    try:
        games = leaguegamefinder.LeagueGameFinder(season_nullable=season, league_id_nullable='00', season_type_nullable='Regular Season', timeout=60).get_data_frames()[0]
        if games.empty: return {}
        games = games[games['WL'].isin(['W', 'L'])]
        games['GAME_DATE'] = pd.to_datetime(games['GAME_DATE'])
        team_ids = sorted(games['TEAM_ID'].unique())
        n_teams = len(team_ids)
        team_to_idx = {t: i for i, t in enumerate(team_ids)}
        M = np.zeros((n_teams, n_teams))
        p = np.zeros(n_teams)
        grouped = games.groupby('GAME_ID')
        for _, g_df in grouped:
            if len(g_df) != 2: continue
            row1 = g_df.iloc[0]; row2 = g_df.iloc[1]
            t1, t2 = row1['TEAM_ID'], row2['TEAM_ID']
            s1, s2 = row1['PTS'], row2['PTS']
            margin = s1 - s2
            i, j = team_to_idx[t1], team_to_idx[t2]
            p[i] += margin; p[j] -= margin
            M[i, i] += 1; M[j, j] += 1; M[i, j] -= 1; M[j, i] -= 1
        M[-1, :] = 1; p[-1] = 0
        ratings = np.linalg.solve(M, p)
        return {str(team_ids[i]): ratings[i] for i in range(n_teams)}
    except Exception as e:
        print(f"   ⚠️ Feng Ratings calculation failed: {e}")
        return {}

# --- Calculate Streak vs Line ---
def calculate_prop_streak(player_id, prop_map, line):
    try:
        gamelog_file = os.path.join(GAMELOG_LAKE_DIR, f"{player_id}.csv")
        if not os.path.exists(gamelog_file): return 0, "-"
        df = pd.read_csv(gamelog_file)
        if df.empty: return 0, "-"
        df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
        df = df.sort_values(by='GAME_DATE', ascending=False).reset_index(drop=True)
        val_series = None
        if prop_map == 'PTS': val_series = df['PTS']
        elif prop_map == 'REB': val_series = df['REB']
        elif prop_map == 'AST': val_series = df['AST']
        elif prop_map == 'FG3M': val_series = df['FG3M']
        elif prop_map == 'PRA': val_series = df['PTS'] + df['REB'] + df['AST']
        elif prop_map == 'P+R': val_series = df['PTS'] + df['REB']
        elif prop_map == 'P+A': val_series = df['PTS'] + df['AST']
        elif prop_map == 'R+A': val_series = df['REB'] + df['AST']
        elif prop_map == 'DD': val_series = ((df['PTS']>=10).astype(int) + (df['REB']>=10).astype(int) + (df['AST']>=10).astype(int)) >= 2

        if val_series is None: return 0, "-"

        streak = 0
        for val in val_series:
            if prop_map == 'DD':
                if val: streak += 1
                else: break
            else:
                if val > line: streak += 1
                else: break

        l10_slice = val_series.head(10)
        hits_in_l10 = 0
        if prop_map == 'DD': hits_in_l10 = l10_slice.sum()
        else: hits_in_l10 = (l10_slice > line).sum()

        return streak, f"{int(hits_in_l10)}/10"
    except Exception: return 0, "-"

# --- Robust H2H ---
def calculate_head_to_head(player_id, prop_map, line, opponent_abbr, season_year):
    try:
        gamelog_file = os.path.join(GAMELOG_LAKE_DIR, f"{player_id}.csv")
        if not os.path.exists(gamelog_file): return "-"
        df = pd.read_csv(gamelog_file)
        if df.empty: return "-"
        df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
        season_start = pd.Timestamp(f"{season_year}-09-01")
        df = df[df['GAME_DATE'] >= season_start]
        if df.empty: return "0/0"
        target_opp = str(opponent_abbr).upper()
        abbr_map = {'PHX': 'PHO', 'PHO': 'PHX', 'GSW': 'GS', 'GS': 'GSW', 'NOP': 'NO', 'NO': 'NOP', 'SAS': 'SA', 'SA': 'SAS', 'NYK': 'NY', 'NY': 'NYK', 'UTA': 'UT', 'UT': 'UTA', 'WAS': 'WSH', 'WSH': 'WAS'}
        alt_opp = abbr_map.get(target_opp, target_opp)
        def is_matchup(m_str):
            if pd.isna(m_str): return False
            try:
                m_str = m_str.replace('.', ''); tokens = m_str.split()
                actual_opp = tokens[-1].upper()
                return actual_opp == target_opp or actual_opp == alt_opp
            except: return False
        df_vs = df[df['MATCHUP'].apply(is_matchup)]
        if df_vs.empty: return "0/0"
        val_series = None
        if prop_map == 'PTS': val_series = df_vs['PTS']
        elif prop_map == 'REB': val_series = df_vs['REB']
        elif prop_map == 'AST': val_series = df_vs['AST']
        elif prop_map == 'FG3M': val_series = df_vs['FG3M']
        elif prop_map == 'PRA': val_series = df_vs['PTS'] + df_vs['REB'] + df_vs['AST']
        elif prop_map == 'P+R': val_series = df_vs['PTS'] + df_vs['REB']
        elif prop_map == 'P+A': val_series = df_vs['PTS'] + df_vs['AST']
        elif prop_map == 'R+A': val_series = df_vs['REB'] + df_vs['AST']
        elif prop_map == 'DD': val_series = ((df_vs['PTS']>=10).astype(int) + (df_vs['REB']>=10).astype(int) + (df_vs['AST']>=10).astype(int)) >= 2
        if val_series is None: return "-"
        hits = 0
        if prop_map == 'DD': hits = val_series.sum()
        else: hits = (val_series > line).sum()
        return f"{int(hits)}/{len(df_vs)}"
    except Exception: return "-"

# --- RESTORED HELPER FUNCTIONS ---
def get_live_schedule_from_nba_data(iso_date, season_year):
    schedule = {}; print(f"--- Fetching Schedule for {iso_date} via API ---")
    try:
        board = scoreboardv2.ScoreboardV2(game_date=iso_date, timeout=60); games = board.game_header.get_data_frame()
        if games.empty: print(f"Warning: NBA API returned no games for {iso_date}"); return {}
        valid_count = 0
        for _, game in games.iterrows():
            h_id = game['HOME_TEAM_ID']; v_id = game['VISITOR_TEAM_ID']
            if pd.isna(h_id) or pd.isna(v_id): continue
            try: h_str = str(int(h_id)); v_str = str(int(v_id))
            except: continue
            if not h_str.startswith('1610612') or not v_str.startswith('1610612'): continue
            schedule[h_str] = v_str; schedule[v_str] = h_str; valid_count += 1
        print(f"✅ Found {valid_count} valid games on the schedule."); return schedule
    except Exception as e: print(f"❌ Critical Error fetching schedule: {e}"); return {}

def get_draft_class_ids(season_year):
    try: df_draft = drafthistory.DraftHistory(season_year_nullable=season_year, timeout=60).get_data_frames()[0]; return set(df_draft['PERSON_ID'].unique())
    except Exception: return set()
def identify_team_changes(stats_last_season, stats_current_season):
    if stats_last_season.empty or stats_current_season.empty: return set()
    merged = pd.merge(stats_last_season[['PLAYER_ID', 'TEAM_ID']], stats_current_season[['PLAYER_ID', 'TEAM_ID']], on='PLAYER_ID', suffixes=('_last', '_current'))
    return set(merged[merged['TEAM_ID_last'] != merged['TEAM_ID_current']]['PLAYER_ID'].unique())
def get_player_experience(player_id, rookie_ids, traded_player_ids, stats_last_season, stats_current_season):
    gp_last = stats_last_season.loc[stats_last_season['PLAYER_ID'] == player_id, 'GP'].iloc[0] if not stats_last_season.loc[stats_last_season['PLAYER_ID'] == player_id].empty else 99
    gp_current = stats_current_season.loc[stats_current_season['PLAYER_ID'] == player_id, 'GP'].iloc[0] if not stats_current_season.loc[stats_current_season['PLAYER_ID'] == player_id].empty else 0
    is_rookie, is_traded = player_id in rookie_ids, player_id in traded_player_ids; is_acclimated = gp_current >= CONFIG["GAMES_PLAYED_THRESHOLD_ACCLIMATED"]
    if is_rookie: return "Traded Rookie" if is_traded and not is_acclimated else "Rookie"
    else: return "Traded Veteran" if is_traded and not is_acclimated else "Low Volume" if gp_last < CONFIG["GAMES_PLAYED_THRESHOLD_LOW_VOLUME"] else "Veteran"
def get_and_update_gamelogs(player_id, current_season, prev_season):
    gamelog_file = os.path.join(GAMELOG_LAKE_DIR, f"{player_id}.csv")
    if os.path.exists(gamelog_file):
        local_logs = pd.read_csv(gamelog_file); local_logs['GAME_DATE'] = pd.to_datetime(local_logs['GAME_DATE']); last_date = local_logs['GAME_DATE'].max()
        try:
            file_mod_time = datetime.fromtimestamp(os.path.getmtime(gamelog_file)).date()
            if file_mod_time == datetime.now().date(): return pd.read_csv(gamelog_file)
            time.sleep(0.4)
            new_logs = playergamelog.PlayerGameLog(player_id=player_id, date_from_nullable=last_date.strftime('%m/%d/%Y'), timeout=30).get_data_frames()[0]
            if not new_logs.empty:
                new_logs['GAME_DATE'] = pd.to_datetime(new_logs['GAME_DATE']); new_logs = new_logs[new_logs['GAME_DATE'] > last_date]
                if not new_logs.empty: updated_logs = pd.concat([new_logs, local_logs], ignore_index=True).sort_values(by="GAME_DATE", ascending=False).drop_duplicates(subset=['Game_ID']); updated_logs.to_csv(gamelog_file, index=False); return updated_logs
            return local_logs
        except Exception: return local_logs
    else:
        try:
            time.sleep(0.4)
            cur = playergamelog.PlayerGameLog(player_id=player_id, season=current_season, timeout=30).get_data_frames()[0]
            prev = playergamelog.PlayerGameLog(player_id=player_id, season=prev_season, timeout=30).get_data_frames()[0]
            full_logs = pd.concat([cur, prev], ignore_index=True).sort_values(by="GAME_DATE", ascending=False).drop_duplicates(subset=['Game_ID']); full_logs.to_csv(gamelog_file, index=False); return full_logs
        except Exception: return pd.DataFrame()

def calculate_live_features(logs):
    if logs.empty or len(logs) < 5: return {}
    logs_for_calc = logs.copy(); features = {}
    logs_for_calc['MIN'] = logs_for_calc['MIN'].apply(convert_min_to_decimal)
    base_stats = ['PTS', 'REB', 'AST', 'FGA', 'FG3A', 'FG3M', 'FTA', 'TOV']
    for col in base_stats:
        if col in logs_for_calc.columns: logs_for_calc[col] = pd.to_numeric(logs_for_calc[col], errors='coerce')
        else: logs_for_calc[col] = np.nan
    logs_for_calc.dropna(subset=['MIN'], inplace=True);
    if logs_for_calc.empty: return {}
    avg_minutes = logs_for_calc['MIN'].mean()
    weights = np.ones(len(logs_for_calc)); weights[logs_for_calc['MIN'] < 0.7 * avg_minutes] = 0.5; weights[logs_for_calc['MIN'] > 1.15 * avg_minutes] = 1.2
    logs_for_calc['WEIGHT'] = weights
    logs_safe = logs_for_calc[logs_for_calc['MIN'] > 0].copy()
    stats_for_ratios = ['PTS', 'REB', 'AST', 'FGA', 'FG3A', 'FTA', 'TOV', 'FG3M']
    for stat in stats_for_ratios:
        if stat in logs_safe.columns: logs_safe[f'{stat}_PM'] = logs_safe[stat] / logs_safe['MIN']
        else: logs_safe[f'{stat}_PM'] = 0
    stats_to_roll = ['PTS', 'REB', 'AST', 'MIN', 'FG3M', 'FG3A'] + [f'{s}_PM' for s in stats_for_ratios]
    for stat in stats_to_roll:
        if stat not in logs_safe.columns: continue
        for w in ROLLING_WINDOWS:
            recent_logs = logs_safe.head(w)
            features[f'W_ROLL_AVG_{stat}_{w}'] = np.average(recent_logs[stat].fillna(0), weights=recent_logs['WEIGHT']) if not recent_logs.empty else np.nan
        features[f'TREND_{stat}'] = features.get(f'W_ROLL_AVG_{stat}_{ROLLING_WINDOWS[0]}') - features.get(f'W_ROLL_AVG_{stat}_{ROLLING_WINDOWS[-1]}')
    for stat in ['PTS', 'REB', 'AST', 'FG3M']:
        if stat in logs_safe.columns:
            last_20 = logs_safe.head(20)
            if not last_20.empty:
                std_dev = last_20[stat].std(); mean_val = last_20[stat].mean()
                cv = (std_dev / mean_val) if mean_val > 0.5 else 9.9
                features[f'CV_{stat}'] = cv
            else: features[f'CV_{stat}'] = 9.9
    features['DAYS_OF_REST'] = (datetime.utcnow().replace(tzinfo=pytz.utc) - pd.to_datetime(logs_for_calc['GAME_DATE'].iloc[0]).tz_localize('UTC')).days
    for stat in stats_for_ratios: features[f'{stat}_PM'] = logs_safe.head(20)[f'{stat}_PM'].mean() if f'{stat}_PM' in logs_safe.columns else 0
    return features

def calculate_ou_probability(projection, line):
    if pd.isna(projection) or pd.isna(line) or projection <= 0: return 0.5, 0.5
    prob_over = 1 - poisson.cdf(k=line, mu=projection); prob_under = poisson.cdf(k=line - 1, mu=projection)
    return prob_over, prob_under

def get_previous_run_data(current_date_str, output_dir):
    today_file = os.path.join(output_dir, f"{current_date_str} NBA Player Props.xlsx")
    if os.path.exists(today_file):
        print(f"✅ Found earlier run from TODAY ({current_date_str}). Calculating Pro vs Info Window movement...")
        try: return pd.read_excel(today_file, sheet_name='DraftKings_EV_Bets')
        except: return pd.DataFrame()
    else:
        print(f"ℹ️ No earlier run found for TODAY ({current_date_str}). This is likely the 'Pro Window' (First Run).")
        return pd.DataFrame()

def apply_movement_and_ranking(curr_df, prev_df, sharp_props_map):
    if curr_df.empty: return curr_df
    if not prev_df.empty:
        curr_df['Player'] = curr_df['Player'].astype(str); prev_df['Player'] = prev_df['Player'].astype(str)
        if 'Bookmaker' in prev_df.columns:
            curr_df['Bookmaker'] = curr_df['Bookmaker'].astype(str); prev_df['Bookmaker'] = prev_df['Bookmaker'].astype(str)
            prev_clean = prev_df[['Player', 'Prop', 'Bookmaker', 'Edge (%)']].rename(columns={'Edge (%)': 'Prev_Edge'})
            merged = pd.merge(curr_df, prev_clean, on=['Player', 'Prop', 'Bookmaker'], how='left', suffixes=('', '_prev'))
        else:
            prev_clean = prev_df[['Player', 'Prop', 'Edge (%)']].rename(columns={'Edge (%)': 'Prev_Edge'})
            merged = pd.merge(curr_df, prev_clean, on=['Player', 'Prop'], how='left', suffixes=('', '_prev'))
    else: merged = curr_df.copy(); merged['Prev_Edge'] = np.nan

    def get_sharp_data(row):
        key = (row['match_key'], row['Prop'])
        sharp_lines_list = sharp_props_map.get(key, [])
        if sharp_lines_list:
            avg_sharp_line = sum(sharp_lines_list) / len(sharp_lines_list)
            diff = row['Projection'] - avg_sharp_line
            return round(avg_sharp_line, 1), diff
        return np.nan, np.nan
    merged[['Sharp Line', 'Proj vs Sharp']] = merged.apply(lambda r: pd.Series(get_sharp_data(r)), axis=1)

    def get_signal(row):
        curr_edge, prev_edge = row.get('Edge (%)', 0), row.get('Prev_Edge')
        if pd.isna(prev_edge): return "-"
        delta = curr_edge - prev_edge
        if delta > 1.5: return "PRICE IMPROVED"
        if delta < -1.5 and curr_edge > 5: return "SHARPS AGREE"
        return "-"
    merged['Sharp Signal'] = merged.apply(get_signal, axis=1)

    def get_consistency_badge(row):
        cv_pts = row.get('CV_PTS', 9.9); cv_reb = row.get('CV_REB', 9.9); cv_ast = row.get('CV_AST', 9.9)
        prop = row['Prop']
        cv = 9.9
        if prop == 'PTS': cv = cv_pts
        elif prop == 'REB': cv = cv_reb
        elif prop == 'AST': cv = cv_ast
        elif prop == 'FG3M': cv = row.get('CV_FG3M', 9.9)
        elif prop == 'PRA': cv = (cv_pts + cv_reb + cv_ast) / 3
        elif prop == 'P+R': cv = (cv_pts + cv_reb) / 2
        elif prop == 'P+A': cv = (cv_pts + cv_ast) / 2
        elif prop == 'R+A': cv = (cv_reb + cv_ast) / 2
        else: return "-"
        if cv > 5.0: return "-"
        if cv < 0.25: return "🔒 High"
        elif cv < 0.45: return "✅ Med"
        else: return "⚡ Volatile"
    merged['Consistency'] = merged.apply(get_consistency_badge, axis=1)

    def get_ranking(row):
        edge, signal = row.get('Edge (%)', 0), row['Sharp Signal']
        consistency = row.get('Consistency', '-')
        bet_type = row.get('Bet')
        has_sharp_line = not pd.isna(row.get('Sharp Line'))
        is_breakout_candidate = (edge > 25.0) and (row.get('Pred_MIN', 0) > 28) and (row.get('Line', 0) < 15) and (bet_type == 'Over')
        if is_breakout_candidate: return "🚀 BREAKOUT OPP"
        prob = row.get('Prob_Over', 0) if bet_type == 'Over' else row.get('Prob_Under', 0)
        if edge > 35.0: return "Avoid / Lottery"
        if has_sharp_line:
            sharp_line = row['Sharp Line']
            line = row['Line']
            if (bet_type == 'Over' and sharp_line > line) or (bet_type == 'Under' and sharp_line < line):
                if edge > 10.0: return "SHARP MISMATCH"
        if signal == 'PRICE IMPROVED' and edge > 15.0: return "💎 PRIME VALUE"
        if signal == 'PRICE IMPROVED': return "Most Profitable"
        if signal == 'SHARPS AGREE': return "Most Validated"
        if prob > 0.60 and consistency != "⚡ Volatile": return "Safest Bet"
        if edge > 10.0: return "Strong Value"
        if edge > 5.0: return "Solid Play"
        return "-"
    merged['Ranking'] = merged.apply(get_ranking, axis=1)
    return merged

# --- Part 5: Main Execution Workflow ---
def run_daily_projections():
    proj_df = pd.DataFrame()

    try:
        if not os.path.exists('/content/drive'): drive.mount('/content/drive', force_remount=True)
    except Exception as e: print(f"⚠️ Drive mount check skipped: {e}")

    eastern = pytz.timezone('US/Eastern')
    SIMULATION_DATE = None

    if 'SIMULATION_DATE' in globals() and globals()['SIMULATION_DATE'] is not None:
         try: datetime.strptime(globals()['SIMULATION_DATE'], "%Y-%m-%d"); SIMULATION_DATE = globals()['SIMULATION_DATE']
         except: SIMULATION_DATE = None

    if SIMULATION_DATE:
        live_now_et = datetime.strptime(f"{SIMULATION_DATE} 12:00:00", "%Y-%m-%d %H:%M:%S").astimezone(eastern)
        print(f"\n⚠️ RUNNING IN SIMULATION MODE for date: {SIMULATION_DATE}")
        iso_today_str = SIMULATION_DATE
        file_today_str = live_now_et.strftime('%m-%d-%Y')
    else:
        live_now_et = datetime.now(eastern)
        print(f"\n📅 Running Live Mode. Current ET Time: {live_now_et.strftime('%Y-%m-%d %H:%M:%S')}")
        iso_today_str = live_now_et.strftime('%Y-%m-%d')
        file_today_str = live_now_et.strftime('%m-%d-%Y')

    if live_now_et.month >= 9: start_year = live_now_et.year; end_year = live_now_et.year + 1
    else: start_year = live_now_et.year - 1; end_year = live_now_et.year
    season = f"{start_year}-{str(end_year)[-2:]}"; prev_season = f"{start_year-1}-{str(start_year)[-2:]}"; season_year_for_url = str(start_year)

    print(f"Running projections for {iso_today_str} (Season: {season})")

    try:
        models = {'minutes': joblib.load(os.path.join(MODEL_DIR, "minutes_lgbm_model.pkl")), 'pts': joblib.load(os.path.join(MODEL_DIR, "pts_lgbm_model.pkl")), 'reb': joblib.load(os.path.join(MODEL_DIR, "reb_lgbm_model.pkl")), 'ast': joblib.load(os.path.join(MODEL_DIR, "ast_lgbm_model.pkl")), 'fg3m': joblib.load(os.path.join(MODEL_DIR, "fg3m_lgbm_model.pkl")), 'dd': joblib.load(os.path.join(MODEL_DIR, "dd_lgbm_model.pkl")), 'td': joblib.load(os.path.join(MODEL_DIR, "td_lgbm_model.pkl"))}
        imputers = {name: joblib.load(os.path.join(MODEL_DIR, f"{name}_imputer.pkl")) for name in ['minutes', 'stats']}
        min_feature_names = joblib.load(os.path.join(MODEL_DIR, "minutes_feature_names.pkl")); stat_feature_names = joblib.load(os.path.join(MODEL_DIR, "stats_feature_names.pkl")); mae_values = json.load(open(os.path.join(MODEL_DIR, "mae_values.json")))
        print("✅ All models, imputers, and MAE values loaded successfully.")
    except FileNotFoundError as e: print(f"CRITICAL ERROR: A required model file is missing. Please run Script 2. Details: {e}"); return

    injury_df = fetch_espn_nba_injuries()

    # *** LOAD PLAYER POSITIONS FROM EXCEL ***
    position_map = {}
    POSITIONS_FILE = os.path.join(GDRIVE_BASE_PATH, "Player Positions/2025-26 NBA Player Positions.xlsx")
    if os.path.exists(POSITIONS_FILE):
        try:
            print(f"Loading Player Positions from: {POSITIONS_FILE}")
            pos_df = pd.read_excel(POSITIONS_FILE)
            if 'Player' in pos_df.columns and 'Pos' in pos_df.columns:
                pos_df['match_key'] = pos_df['Player'].apply(normalize_player_name)
                position_map = pos_df.set_index('match_key')['Pos'].to_dict()
                print(f"✅ Successfully loaded {len(position_map)} player positions.")
            else: print("⚠️ Warning: Position file missing 'Player' or 'Pos' columns.")
        except Exception as e: print(f"⚠️ Error loading Position file: {e}")
    else: print(f"⚠️ Warning: Position file not found at {POSITIONS_FILE}. Using proxy logic.")

    scratched_players = []
    if not injury_df.empty:
        scratched_players = injury_df[injury_df['Status'].isin(['Out', 'Injured Reserve', 'Suspended'])]['match_key'].tolist()

    prev_file = os.path.join(OUTPUT_DIR, f"{file_today_str} NBA Player Props.xlsx")
    if os.path.exists(prev_file):
        try:
            old_df = pd.read_excel(prev_file, sheet_name='DraftKings_EV_Bets')
            if 'Status' in old_df.columns and 'Player' in old_df.columns:
                old_active = old_df[old_df['Status'] == 'Active']['Player'].apply(normalize_player_name).tolist()
                newly_scratched = [p for p in old_active if p in scratched_players]
                if newly_scratched:
                    print(f"   🚨 AUTOMATED ALERT: Detected new scratches: {newly_scratched}")
        except: pass

    if scratched_players: print(f"   ℹ️ Detected {len(scratched_players)} ruled-out players. Adjusting team usage automatically...")

    print("\n--- Preparing Context & Stats ---")
    stats_current_season, stats_last_season = pd.DataFrame(), pd.DataFrame()
    for attempt in range(3):
        try:
            stats_current_season = leaguedashplayerstats.LeagueDashPlayerStats(season=season, timeout=120).get_data_frames()[0]
            stats_last_season = leaguedashplayerstats.LeagueDashPlayerStats(season=prev_season, timeout=120).get_data_frames()[0]
            if not stats_current_season.empty and not stats_last_season.empty: print("✅ Successfully fetched season stats."); break
        except Exception: time.sleep(5)

    if stats_current_season.empty: print("⚠️ Warning: Could not fetch current season stats.")

    # --- GET DEFENSE RANKS & RAW STATS ---
    def_ranks, def_stats_raw = get_defense_ranks(season)
    if not def_ranks: print("   ⚠️ Warning: Defense Ranks are empty. API might be down or Season mismatch.")

    # --- GET FENG RATINGS ---
    feng_ratings = get_nba_feng_ratings(season)

    team_metrics = get_live_team_metrics(season)
    schedule = get_live_schedule_from_nba_data(iso_today_str, season_year_for_url) # USE ISO DATE HERE
    if not schedule: print("⚠️ Warning: NBA API returned 0 games. Will attempt to rebuild schedule from Betting Odds.")

    schedule = {str(k): str(v) for k, v in schedule.items() if k is not None and v is not None}
    if not stats_current_season.empty: stats_current_season['TEAM_ID'] = stats_current_season['TEAM_ID'].astype(str).str.split('.').str[0]

    rookie_ids = get_draft_class_ids(season_year_for_url); traded_player_ids = identify_team_changes(stats_last_season, stats_current_season)
    props_df, live_game_lines = pd.DataFrame(), {}
    key, game_events = None, []
    team_abbr_map = get_team_abbr_map()

    print("\n--- Running Initial Projections & Monte Carlo Sims ---")
    all_rosters = stats_current_season
    projection_rows, model_game_rows = [], []

    LEAGUE_AVG_PACE = 99.0
    LEAGUE_AVG_OFF = 115.0

    if team_metrics:
        paces = [m['E_PACE'] for m in team_metrics.values() if m['E_PACE'] > 0]
        offs = [m['E_OFF_RATING'] for m in team_metrics.values() if m['E_OFF_RATING'] > 0]
        if paces: LEAGUE_AVG_PACE = sum(paces) / len(paces)
        if offs: LEAGUE_AVG_OFF = sum(offs) / len(offs)

    if schedule:
        for team_id, opponent_id in tqdm(schedule.items(), desc="Generating Projections"):
            # --- 1. PLAYER PROJECTIONS ---
            roster = all_rosters[all_rosters['TEAM_ID'] == str(team_id)]
            team_scratches = [p for p in scratched_players if p in roster['PLAYER_NAME'].apply(normalize_player_name).tolist()]
            usage_boost_factor = 1.0 + (len(team_scratches) * 0.05)

            for _, row in roster.iterrows():
                if 'GP' in row and row['GP'] > 0 and (row['MIN'] / row['GP']) < 6.0: continue
                player_id, player_name, team_abbr = row['PLAYER_ID'], row['PLAYER_NAME'], row['TEAM_ABBREVIATION']
                match_key = normalize_player_name(player_name)
                logs = get_and_update_gamelogs(player_id, season, prev_season); features = calculate_live_features(logs)
                if not features: continue
                try: opp_id_int = int(opponent_id); opp_metrics = team_metrics.get(opp_id_int, {})
                except: opp_metrics = {}

                features['OPP_ALLOWED_PTS_AVG_30'] = opp_metrics.get('E_DEF_RATING', 115)
                features['OPP_ALLOWED_REB_AVG_30'] = 44.0; features['OPP_ALLOWED_AST_AVG_30'] = 25.0; features['OPP_ALLOWED_FG3M_AVG_30'] = 12.0

                if str(opponent_id) in def_stats_raw:
                    features['OPP_ALLOWED_PTS_AVG_30'] = def_stats_raw[str(opponent_id)]['PTS']
                    features['OPP_ALLOWED_REB_AVG_30'] = def_stats_raw[str(opponent_id)]['REB']
                    features['OPP_ALLOWED_AST_AVG_30'] = def_stats_raw[str(opponent_id)]['AST']
                    features['OPP_ALLOWED_FG3M_AVG_30'] = def_stats_raw[str(opponent_id)]['FG3M']

                features['VEGAS_SPREAD'] = 0; features['VEGAS_TOTAL'] = 225

                try:
                    min_input = pd.DataFrame([features])[min_feature_names]; pred_min = models['minutes'].predict(imputers['minutes'].transform(min_input))[0]; features['Proj_MIN'] = pred_min
                    if usage_boost_factor > 1.0:
                        features['FGA_PM'] = features.get('FGA_PM', 0) * usage_boost_factor
                        pred_min = pred_min * 1.02

                    stat_input = imputers['stats'].transform(pd.DataFrame([features])[stat_feature_names])
                    p_pts = models['pts'].predict(stat_input)[0]; p_reb = models['reb'].predict(stat_input)[0]
                    p_ast = models['ast'].predict(stat_input)[0]; p_3pm = models['fg3m'].predict(stat_input)[0]

                    temp_proj = {'Pred_PTS': p_pts, 'Pred_REB': p_reb, 'Pred_AST': p_ast}
                    if match_key in position_map: player_pos = position_map[match_key]
                    else: player_pos = derive_position(temp_proj)

                    proj = {
                        'PLAYER_ID': player_id, 'match_key': match_key, 'Player': player_name,
                        'Pos': player_pos, 'Experience': get_player_experience(player_id, rookie_ids, traded_player_ids, stats_last_season, stats_current_season),
                        'Team': team_abbr, 'Opponent': teams.find_team_name_by_id(int(opponent_id))['abbreviation'] if opponent_id else "UNK",
                        'OpponentID': str(opponent_id), 'Pred_MIN': pred_min, 'Pred_FGA_PM': features.get('FGA_PM', 0),
                        'Pred_PTS': p_pts, 'Pred_REB': p_reb, 'Pred_AST': p_ast, 'Pred_FG3M': p_3pm
                    }
                    proj['CV_PTS'] = features.get('CV_PTS', 9.9); proj['CV_REB'] = features.get('CV_REB', 9.9)
                    proj['CV_AST'] = features.get('CV_AST', 9.9); proj['CV_FG3M'] = features.get('CV_FG3M', 9.9)
                    clf_features = np.array([proj['Pred_PTS'], proj['Pred_REB'], proj['Pred_AST']]).reshape(1, -1)
                    proj['Pred_DD'] = models['dd'].predict_proba(clf_features)[0][1] * 100; proj['Pred_TD'] = models['td'].predict_proba(clf_features)[0][1] * 100
                    proj['Pred_PRA'] = proj['Pred_PTS'] + proj['Pred_REB'] + proj['Pred_AST']; proj['Pred_P+R'] = proj['Pred_PTS'] + proj['Pred_REB']; proj['Pred_P+A'] = proj['Pred_PTS'] + proj['Pred_AST']; proj['Pred_R+A'] = proj['Pred_REB'] + proj['Pred_AST']
                    projection_rows.append(proj)
                except Exception: continue

            # --- 2. GAME ENGINE (MONTE CARLO UPGRADE) ---
            home_stats, away_stats = team_metrics.get(int(team_id), {}), team_metrics.get(int(opponent_id), {})
            if home_stats and away_stats:
                h_off, a_off = home_stats.get('E_OFF_RATING', 110), away_stats.get('E_OFF_RATING', 110)
                h_def, a_def = home_stats.get('E_DEF_RATING', 110), away_stats.get('E_DEF_RATING', 110)
                h_pace, a_pace = home_stats.get('E_PACE', 99), away_stats.get('E_PACE', 99)
                poss = (h_pace + a_pace) / 2; HCA = 2.5
                h_exp = (h_off/100) * (a_def/100) * (100 / (LEAGUE_AVG_OFF/100)) * poss + (HCA/2)
                a_exp = (a_off/100) * (h_def/100) * (100 / (LEAGUE_AVG_OFF/100)) * poss - (HCA/2)
                SIMS = 5000
                scores = np.random.multivariate_normal([h_exp, a_exp], [[169, 50], [50, 169]], size=SIMS)
                h_sc, a_sc = scores[:,0], scores[:,1]
                model_spread = -(np.mean(h_sc) - np.mean(a_sc))
                model_total = np.mean(h_sc) + np.mean(a_sc)
                model_win_prob = np.mean(h_sc > a_sc)
                pyth_h = h_off**13.91 / (h_off**13.91 + h_def**13.91)
                pyth_a = a_off**13.91 / (a_off**13.91 + a_def**13.91)
                log5_prob = (pyth_h - pyth_h * pyth_a) / (pyth_h + pyth_a - 2 * pyth_h * pyth_a)
                actual_win_pct = home_stats.get('W', 0) / home_stats.get('GP', 1)
                luck_metric = actual_win_pct - pyth_h
                r_home, r_away = feng_ratings.get(str(team_id), 0), feng_ratings.get(str(opponent_id), 0)
                feng_spread = -(r_home - r_away + HCA) if r_home and r_away else None

                model_game_rows.append({
                    'TeamID': int(team_id),
                    'Model Spread': round(model_spread, 1),
                    'Model Total': round(model_total, 1),
                    'Model Win %': model_win_prob,
                    'Log5 %': log5_prob,
                    'Feng Spread': round(feng_spread, 1) if feng_spread else "-",
                    'Luck': luck_metric,
                    'NetRtg': home_stats.get('E_NET_RATING'),
                    'eFG%': home_stats.get('E_EFG_PCT'),
                    'OREB%': home_stats.get('E_OREB_PCT')
                })

    proj_df = pd.DataFrame(projection_rows)
    if proj_df.empty:
        print(f"\n❌ NO PROJECTIONS GENERATED."); return
    else:
        proj_df['match_key'] = proj_df['Player'].apply(normalize_player_name)
        print(f"✅ Success: Generated {len(proj_df)} player projections. Proceeding to fetch odds...")

    # 5. FETCH ODDS
    game_lines_rows = []
    sharp_props_map = {}

    for k in ODDS_API_KEYS:
        try:
            r = requests.get(f"https://api.the-odds-api.com/v4/sports/basketball_nba/odds?apiKey={k}&regions=us&markets=h2h", timeout=10); r.raise_for_status()
            if int(r.headers.get('x-requests-remaining', 0)) > 20: key = k; print(f"Using API key ending in ...{k[-4:]}"); break
        except Exception: continue

    if key:
        print("--- Fetching Sharp & Retail Odds ---")
        try:
            game_markets_to_fetch = ",".join(ACTIVE_GAME_MARKETS)
            books_str = ",".join(ALL_BOOKS_FOR_API)
            url_games = f"https://api.the-odds-api.com/v4/sports/basketball_nba/odds/?apiKey={key}&regions=us,eu&markets={game_markets_to_fetch or 'h2h'}&bookmakers={books_str}"
            response_games = requests.get(url_games, timeout=15); response_games.raise_for_status(); games_data = response_games.json()
            game_events = [game for game in games_data if parser.isoparse(game["commence_time"]).astimezone(eastern).strftime('%Y-%m-%d') == iso_today_str] # USE ISO DATE HERE
            game_events.sort(key=lambda x: x["commence_time"])

            for game in game_events:
                home_team_list = teams.find_teams_by_full_name(game['home_team']); away_team_list = teams.find_teams_by_full_name(game['away_team'])
                if not home_team_list or not away_team_list: continue
                home_team, away_team = home_team_list[0], away_team_list[0]
                home_id, away_id = str(home_team['id']), str(away_team['id'])
                if home_id not in schedule: schedule[home_id] = away_id; schedule[away_id] = home_id

                home_abbr, away_abbr = home_team['abbreviation'], away_team['abbreviation']
                spreads, totals, mls = [], [], []; retail_spread, retail_total, retail_ml, retail_book = np.nan, np.nan, np.nan, "Consensus"
                home_spread_sharp_list, home_spread_retail_val = [], None; over_sharp_list, over_retail_val = [], None

                for book in game.get('bookmakers', []):
                    if book['key'] in SHARP_BOOKS:
                        for market in book.get('markets', []):
                            if market['key'] == 'spreads': spreads.append(market['outcomes'][0]['point'])
                            elif market['key'] == 'totals': totals.append(market['outcomes'][0]['point'])
                            elif market['key'] == 'h2h':
                                for o in market['outcomes']:
                                    if o['name'] == game['home_team']: mls.append(o['price'])
                    if book['key'] in ['draftkings', 'fanduel']:
                        if pd.isna(retail_spread): retail_book = book['title']
                        for market in book.get('markets', []):
                            if market['key'] == 'spreads' and pd.isna(retail_spread): retail_spread = market['outcomes'][0]['point']
                            if market['key'] == 'totals' and pd.isna(retail_total): retail_total = market['outcomes'][0]['point']
                            if market['key'] == 'h2h' and pd.isna(retail_ml):
                                for o in market['outcomes']:
                                    if o['name'] == game['home_team']: retail_ml = o['price']

                    for market in book.get('markets', []):
                        if market['key'] == 'spreads':
                            for o in market['outcomes']:
                                if o['name'] == game['home_team']:
                                    if book['key'] in SHARP_BOOKS: home_spread_sharp_list.append(o['point'])
                                    elif book['key'] in RETAIL_BOOKS and home_spread_retail_val is None: home_spread_retail_val = o['point']
                        if market['key'] == 'totals':
                            for o in market['outcomes']:
                                if o['name'] == 'Over':
                                    if book['key'] in SHARP_BOOKS: over_sharp_list.append(o['point'])
                                    elif book['key'] in RETAIL_BOOKS and over_retail_val is None: over_retail_val = o['point']

                sharp_spread = sum(spreads)/len(spreads) if spreads else None; sharp_total = sum(totals)/len(totals) if totals else None
                try: live_game_lines[int(home_id)] = {'spread': retail_spread, 'total': retail_total}; live_game_lines[int(away_id)] = {'spread': -retail_spread if not pd.isna(retail_spread) else np.nan, 'total': retail_total}
                except: pass

                model_data = next((item for item in model_game_rows if item["TeamID"] == int(home_id)), {})
                spread_diff = abs(retail_spread - sharp_spread) if (not pd.isna(retail_spread) and sharp_spread is not None) else 0
                total_diff = abs(retail_total - sharp_total) if (not pd.isna(retail_total) and sharp_total is not None) else 0
                signal = "SHARP MISMATCH" if (spread_diff >= 1.5 or total_diff >= 1.5) else ("Diff Found" if (spread_diff >= 0.5) else "-")

                pick_spread, pick_total, pick_ml = "-", "-", "-"
                if home_spread_retail_val is not None and home_spread_sharp_list:
                    avg_home_sharp = sum(home_spread_sharp_list) / len(home_spread_sharp_list)
                    if abs(home_spread_retail_val - avg_home_sharp) >= 1.0:
                        pick_spread = f"{home_abbr} {home_spread_retail_val:+.1f}" if avg_home_sharp < home_spread_retail_val else f"{away_abbr} {-1 * home_spread_retail_val:+.1f}"

                if over_retail_val is not None and over_sharp_list:
                    avg_over_sharp = sum(over_sharp_list) / len(over_sharp_list)
                    if abs(over_retail_val - avg_over_sharp) >= 1.0:
                        pick_total = f"Over {over_retail_val}" if avg_over_sharp > over_retail_val else f"Under {over_retail_val}"

                if not pd.isna(retail_ml) and 'Model Win %' in model_data:
                    if retail_ml > 1.0 and model_data['Model Win %'] > ((1.0 / retail_ml) + 0.05): pick_ml = f"{home_abbr} ML"

                game_lines_rows.append({
                    'Game': f"{away_abbr} @ {home_abbr}",
                    'Time': parser.isoparse(game["commence_time"]).astimezone(eastern).strftime('%-I:%M %p'),
                    'Retail Book': retail_book,
                    'Retail Spread': retail_spread, 'Sharp Spread': round(sharp_spread, 1) if sharp_spread else "-",
                    'Model Spread': model_data.get('Model Spread', '-'),
                    'Feng Spread': model_data.get('Feng Spread', '-'),
                    'Spread Diff': round(spread_diff, 1), 'Spread Pick': pick_spread,
                    'Retail Total': retail_total, 'Sharp Total': round(sharp_total, 1) if sharp_total else "-",
                    'Model Total': model_data.get('Model Total', '-'),
                    'Total Diff': round(total_diff, 1), 'Total Pick': pick_total,
                    'Retail ML': retail_ml,
                    'Model Win %': f"{model_data.get('Model Win %', 0)*100:.1f}%" if 'Model Win %' in model_data else "-",
                    'Log5 Win %': f"{model_data.get('Log5 %', 0)*100:.1f}%" if 'Log5 %' in model_data else "-",
                    'ML Pick': pick_ml,
                    'Home Luck': round(model_data.get('Luck', 0), 3) if 'Luck' in model_data else "-",
                    'Home Net Rtg': model_data.get('NetRtg', '-'),
                    'Home eFG%': model_data.get('eFG%', '-'),
                    'Signal': signal
                })
        except Exception as e: print(f"Failed to fetch game list/lines: {e}"); game_events = []

        if game_events:
            all_props_raw = []
            market_query = ','.join(ALL_PROP_MARKETS_LIST) if ALL_PROP_MARKETS_LIST else ""
            if market_query:
                for game in tqdm(game_events, desc="Fetching Player Props"):
                    event_id = game['id']
                    away_abbr = team_abbr_map.get(game['away_team'], '?'); home_abbr = team_abbr_map.get(game['home_team'], '?')
                    books_str = ",".join(ALL_BOOKS_FOR_API)
                    url_props = f"https://api.the-odds-api.com/v4/sports/basketball_nba/events/{event_id}/odds?apiKey={key}&regions=us,eu&markets={market_query}&bookmakers={books_str}"
                    try:
                        r = requests.get(url_props, timeout=20); r.raise_for_status(); payload = r.json()
                        for book in payload.get('bookmakers', []):
                            for market in book.get('markets', []):
                                for outcome in market.get('outcomes', []):
                                    player_name = outcome['description']
                                    norm_name = normalize_player_name(player_name)
                                    prop_name = MARKET_NAME_MAP.get(market['key'])
                                    line = outcome.get('point')

                                    if book['key'] in SHARP_BOOKS and line is not None:
                                        dict_key = (norm_name, prop_name)
                                        if dict_key not in sharp_props_map: sharp_props_map[dict_key] = []
                                        sharp_props_map[dict_key].append(line)
                                        continue

                                    if book['key'] in ['draftkings', 'fanduel']:
                                        all_props_raw.append({'Game': f"{away_abbr} @ {home_abbr}", 'Game_Time': parser.isoparse(game["commence_time"]).astimezone(eastern).strftime('%-I:%M %p'), 'Sort_Time': game["commence_time"], 'Player': player_name, 'Prop': prop_name, 'Line': line, 'Decimal_Odds': outcome.get('price'), 'OutcomeName': outcome['name'], 'Bookmaker': book['title']})
                    except Exception: continue
            if all_props_raw:
                props_df = pd.DataFrame(all_props_raw)
                props_df['match_key'] = props_df['Player'].apply(normalize_player_name)
                print(f"Debug Props Keys: {props_df['match_key'].head(3).tolist()}")

    if props_df.empty: print("\n--- Displaying Projections Only (No Betting Odds Found) ---"); display(proj_df[['Player', 'Experience', 'Team', 'Opponent', 'Pred_MIN', 'Pred_PTS', 'Pred_REB', 'Pred_AST', 'Pred_FG3M']].round(2).head(20)); return

    print("\n--- Running Value Analysis & Filtering Injuries ---")
    ou_props = props_df[props_df['OutcomeName'].isin(['Over', 'Under'])].copy(); yn_props = props_df[props_df['OutcomeName'].isin(['Yes', 'No'])].copy()
    merged_ou = pd.DataFrame()
    if not ou_props.empty:
        ou_pivot = ou_props.pivot_table(index=['match_key', 'Player', 'Prop', 'Line', 'Bookmaker', 'Game', 'Game_Time', 'Sort_Time'], columns='OutcomeName', values='Decimal_Odds').reset_index().rename(columns={'Over': 'Over_Odds', 'Under': 'Under_Odds'})
        merged_ou = pd.merge(ou_pivot, proj_df.drop(columns=['Player'], errors='ignore'), on='match_key', how='inner')
        if not merged_ou.empty:
            merged_ou['Projection'] = merged_ou.apply(lambda r: r.get(f"Pred_{r['Prop']}", np.nan), axis=1)
            merged_ou['Proj Range Low'] = (merged_ou['Projection'] - merged_ou['Prop'].map(mae_values)).clip(lower=0); merged_ou['Proj Range High'] = merged_ou['Projection'] + merged_ou['Prop'].map(mae_values)
            merged_ou['Proj Range'] = merged_ou.apply(lambda r: f"{r['Proj Range Low']:.1f}-{r['Proj Range High']:.1f}", axis=1)
            merged_ou[['Prob_Over', 'Prob_Under']] = merged_ou.apply(lambda r: calculate_ou_probability(r['Projection'], r['Line']), axis=1, result_type='expand')
            def determine_bet(row):
                if row['Line'] < row['Proj Range Low'] and row['Prob_Over'] > 0.53: return 'Over'
                elif row['Line'] > row['Proj Range High'] and row['Prob_Under'] > 0.53: return 'Under'
                return 'Pass'
            merged_ou['Bet'] = merged_ou.apply(determine_bet, axis=1)
            merged_ou['Edge (%)'] = np.where(merged_ou['Bet'] == 'Over', (merged_ou['Prob_Over'] * merged_ou['Over_Odds']) - 1, np.where(merged_ou['Bet'] == 'Under', (merged_ou['Prob_Under'] * merged_ou['Under_Odds']) - 1, 0)) * 100

    merged_yn = pd.DataFrame()
    if not yn_props.empty:
        yn_pivot = yn_props.pivot_table(index=['match_key', 'Player', 'Prop', 'Bookmaker', 'Game', 'Game_Time', 'Sort_Time'], columns='OutcomeName', values='Decimal_Odds').reset_index().rename(columns={'Yes': 'Yes_Odds', 'No': 'No_Odds'})
        merged_yn = pd.merge(yn_pivot, proj_df.drop(columns=['Player'], errors='ignore'), on='match_key', how='inner')
        if not merged_yn.empty:
            merged_yn['Projection'] = merged_yn.apply(lambda r: r.get(f"Pred_{r['Prop']}", np.nan), axis=1)
            merged_yn['Model Score (%)'] = merged_yn['Projection']; merged_yn['Implied Prob (%)'] = (1 / merged_yn['Yes_Odds']) * 100
            merged_yn['Edge (%)'] = merged_yn['Model Score (%)'] - merged_yn['Implied Prob (%)']; merged_yn['Bet'] = np.where(merged_yn['Edge (%)'] > 5, 'Yes', 'Pass')

    full_analysis_df = pd.concat([merged_ou, merged_yn])
    if not injury_df.empty and not full_analysis_df.empty: full_analysis_df = pd.merge(full_analysis_df, injury_df[['match_key', 'Status']], on='match_key', how='left'); full_analysis_df['Status'] = full_analysis_df['Status'].fillna('Active')
    elif not full_analysis_df.empty: full_analysis_df['Status'] = 'Active'

    if not full_analysis_df.empty and FILTER_OUT_PLAYERS: full_analysis_df = full_analysis_df[~full_analysis_df['Status'].isin(['Out', 'Injured Reserve', 'Suspended'])]

    prev_run_df = get_previous_run_data(file_today_str, OUTPUT_DIR) # USE FILE DATE HERE

    full_analysis_df = apply_movement_and_ranking(full_analysis_df, prev_run_df, sharp_props_map)

    # ==============================================================================
    # --- NEW FEATURE: STREAKS, DEFENSE RANKS & HEAD-TO-HEAD ---
    # ==============================================================================
    if not full_analysis_df.empty:
        print("\n--- Calculating Player Streaks, Opponent Ranks & H2H ---")

        streak_data = []
        rank_data = []
        h2h_data = [] # New list

        for idx, row in tqdm(full_analysis_df.iterrows(), total=len(full_analysis_df), desc="Analyzing Trends"):
            # 1. Defense Rank
            # *** FIX: Use ID instead of Abbreviation lookup which was failing ***
            opp_id = row.get('OpponentID') # Passed from Proj
            opp_abbr = row.get('Opponent') # For H2H lookup
            prop_cat = row.get('Prop')
            rank_val = 99
            stat_key_map = {'Points': 'PTS', 'Rebounds': 'REB', 'Assists': 'AST', 'Threes': 'FG3M'}

            if opp_id and opp_id in def_ranks and prop_cat in stat_key_map:
                rank_val = def_ranks[opp_id][stat_key_map[prop_cat]]
            rank_data.append(rank_val)

            # Setup Map for Logs
            pid = row.get('PLAYER_ID')
            line = row.get('Line')
            log_map = {'Points': 'PTS', 'Rebounds': 'REB', 'Assists': 'AST', 'Threes': 'FG3M', 'Combos': 'PRA'}
            mapped_prop = log_map.get(prop_cat, prop_cat)
            if prop_cat == 'DoubleDouble': mapped_prop = 'DD'
            if 'Pts+Reb+Ast' in str(row.get('OutcomeName', '')): mapped_prop = 'PRA'
            elif 'Pts+Reb' in str(row.get('OutcomeName', '')): mapped_prop = 'P+R'
            elif 'Pts+Ast' in str(row.get('OutcomeName', '')): mapped_prop = 'P+A'
            elif 'Reb+Ast' in str(row.get('OutcomeName', '')): mapped_prop = 'R+A'

            # 2. Streaks (10 Game Window)
            s_curr, s_l10 = calculate_prop_streak(pid, mapped_prop, line)
            # *** FIX: Reverted to W{n}. Shows W0 if 0. ***
            streak_str = f"W{s_curr} ({s_l10})"
            streak_data.append(streak_str)

            # 3. Head-to-Head (vs Opponent this season)
            # We use start_year from the configuration variables
            h2h_val = calculate_head_to_head(pid, mapped_prop, line, opp_abbr, start_year)
            h2h_data.append(h2h_val)

        full_analysis_df['Opp Rank'] = rank_data
        full_analysis_df['Streak'] = streak_data
        full_analysis_df['vs Opp'] = h2h_data # Add new column

        def fmt_rank(r):
            if r == 99: return "-"
            if r >= 25: return f"{r} (Easy)"
            if r <= 5: return f"{r} (Hard)"
            return str(r)

        full_analysis_df['Opp Rank'] = full_analysis_df['Opp Rank'].apply(fmt_rank)

    # ==============================================================================

    if not full_analysis_df.empty:
        full_analysis_df['Projection'] = full_analysis_df['Projection'].round(2)
        if 'Edge (%)' in full_analysis_df.columns: full_analysis_df['Edge (%)'] = full_analysis_df['Edge (%)'].round(1)
        if 'Proj vs Sharp' in full_analysis_df.columns: full_analysis_df['Proj vs Sharp'] = full_analysis_df['Proj vs Sharp'].astype(float).round(1)

        final_bets = full_analysis_df[full_analysis_df['Bet'] != 'Pass'].copy()
    else: final_bets = pd.DataFrame()

    out_file = os.path.join(OUTPUT_DIR, f"{file_today_str} NBA Player Props.xlsx") # USE FILE DATE HERE

    with pd.ExcelWriter(out_file, engine='xlsxwriter') as writer:
        workbook  = writer.book
        header_fmt = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'bg_color': '#D9D9D9', 'border': 1})
        center_fmt = workbook.add_format({'align': 'center', 'valign': 'vcenter', 'border': 1})
        green_fmt = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        red_fmt = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        orange_fmt = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C5700', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        blue_fmt = workbook.add_format({'bg_color': '#BDD7EE', 'font_color': '#1F4E78', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        purple_fmt = workbook.add_format({'bg_color': '#CCC0DA', 'font_color': '#3F3F76', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        perfect_fmt = workbook.add_format({'bg_color': '#00FF00', 'font_color': '#000000', 'align': 'center', 'valign': 'vcenter', 'border': 1, 'bold': True})
        dd_td_fmt = workbook.add_format({'bg_color': '#9FC5E8', 'font_color': '#000000', 'align': 'center', 'valign': 'vcenter', 'border': 1, 'bold': True})

        green_row_fmt = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        blue_row_fmt = workbook.add_format({'bg_color': '#BDD7EE', 'font_color': '#1F4E78', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        orange_row_fmt = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C5700', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        red_row_fmt = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        purple_row_fmt = workbook.add_format({'bg_color': '#CCC0DA', 'font_color': '#3F3F76', 'align': 'left', 'valign': 'vcenter', 'border': 1})

        ws_howto = workbook.add_worksheet("HOW TO USE")
        ws_howto.set_column(0, 0, 30); ws_howto.set_column(1, 1, 120); ws_howto.freeze_panes(1, 0)
        ws_howto.write(0, 0, "Item", header_fmt); ws_howto.write(0, 1, "Description", header_fmt)

        ws_howto.write(1, 0, "Ranking: SHARP MISMATCH", purple_row_fmt); ws_howto.write(1, 1, "Massive Value + Price Improved. The line moved in your favor AND the Edge is huge.", purple_row_fmt)
        ws_howto.write(2, 0, "Ranking: Most Profitable", green_row_fmt); ws_howto.write(2, 1, "Price Improved. Line moved in your favor since yesterday.", green_row_fmt)
        ws_howto.write(3, 0, "Ranking: Most Validated", orange_row_fmt); ws_howto.write(3, 1, "Sharps Agree. Line moved against you, but model still sees value.", orange_row_fmt)
        ws_howto.write(4, 0, "Ranking: Safest Bet", blue_row_fmt); ws_howto.write(4, 1, "High Probability (>60%) of winning regardless of odds.", blue_row_fmt)
        ws_howto.write(5, 0, "Ranking: Avoid / Lottery", red_row_fmt); ws_howto.write(5, 1, "Suspiciously high edge (>35%). Likely an injury risk.", red_row_fmt)
        ws_howto.write(6, 0, "Ranking: PRIME VALUE", blue_row_fmt); ws_howto.write(6, 1, "High Edge + Price Improved, but no Sharp Line confirmation available yet.", blue_row_fmt)
        ws_howto.write(7, 0, "Ranking: BREAKOUT OPP", red_row_fmt); ws_howto.write(7, 1, "Model projects Big Minutes for a low-line player. High Upside but volatile.", red_row_fmt)

        ws_howto.write(8, 0, "BETTING SCHEDULE", header_fmt)
        ws_howto.write(9, 0, "4 Hours Out (Pro Window)", blue_row_fmt); ws_howto.write(9, 1, "Sportsbooks raise limits. Professional money enters. Movement here signals Syndicate preference before public money floods in.", blue_row_fmt)
        ws_howto.write(10, 0, "90 Minutes Out (Info Window)", orange_row_fmt); ws_howto.write(10, 1, "Official injury reports released. Movement here is the market reaction to hard facts. Whale money enters.", orange_row_fmt)

        ws_howto.write(12, 0, "MODEL ACCURACY RANKING", header_fmt)
        ws_howto.write(13, 0, "1. Threes (FG3M)", green_row_fmt); ws_howto.write(13, 1, "Most Accurate (MAE ~0.13). Trust small edges here. Green Highlight at > 5%.", green_row_fmt)
        ws_howto.write(14, 0, "2. Assists (AST)", green_row_fmt); ws_howto.write(14, 1, "Very Accurate (MAE ~0.28). Great for Point Guard props. Green Highlight at > 8%.", green_row_fmt)
        ws_howto.write(15, 0, "3. Rebounds (REB)", blue_row_fmt); ws_howto.write(15, 1, "Strong (MAE ~0.50). Focus on Centers/consistent rebounders. Green Highlight at > 8%.", blue_row_fmt)
        ws_howto.write(16, 0, "4. Points (PTS)", orange_row_fmt); ws_howto.write(16, 1, "Highest Variance (MAE ~1.15). Only bet if Edge > 10% or it's a Breakout scenario. Green Highlight at > 12%.", orange_row_fmt)

        ws_howto.write(18, 0, "CONSISTENCY GUIDE", header_fmt)
        ws_howto.write(19, 0, "Consistency: 🔒 High", green_row_fmt); ws_howto.write(19, 1, "Reliable Player. Low Variance (CV < 0.25). Safe to trust projection.", green_row_fmt)
        ws_howto.write(20, 0, "Consistency: ⚡ Volatile", red_row_fmt); ws_howto.write(20, 1, "Boom or Bust. High Variance (CV > 0.45). Bet with caution even if edge is high.", red_row_fmt)

        sheet_config = ['DraftKings_EV_Bets', 'DraftKings_EV_SGP', 'FanDuel_EV_Bets', 'FanDuel_EV_SGP']
        for sheet_name in sheet_config:
            bookmaker = 'DraftKings' if 'DraftKings' in sheet_name else 'FanDuel'
            is_sgp = 'SGP' in sheet_name
            book_df = final_bets[final_bets['Bookmaker'] == bookmaker].copy() if not final_bets.empty else pd.DataFrame()
            if not book_df.empty:
                # *** CUSTOM COLUMN ORDER ***
                out_cols = [
                    'Game_Time', 'Game', 'Status', 'Experience', 'Team', 'Pos', 'Player', 'Prop',
                    'Line', 'Bet', 'Sharp Line', 'Proj vs Sharp',
                    'Projection', 'Proj Range', 'Edge (%)',
                    'Consistency', 'Opp Rank', 'Streak', 'vs Opp', 'Sharp Signal', 'Ranking'
                ]

                if 'Sort_Time' in book_df.columns:
                    book_df['sort_rank'] = book_df['Ranking'].map({
                        "SHARP MISMATCH": 0, "💎 PRIME VALUE": 1, "Most Profitable": 2, "Most Validated": 3, "Safest Bet": 4,
                        "🚀 BREAKOUT OPP": 5, "Strong Value": 6, "Solid Play": 7, "Avoid / Lottery": 99
                    }).fillna(99)

                    if is_sgp: final_df = book_df.sort_values(['Sort_Time', 'Game', 'sort_rank', 'Edge (%)'], ascending=[True, True, True, False]).reindex(columns=out_cols)
                    else: final_df = book_df.sort_values(['sort_rank', 'Edge (%)'], ascending=[True, False]).reindex(columns=out_cols)
                else: final_df = book_df.reindex(columns=out_cols)

                final_df.to_excel(writer, sheet_name=sheet_name, index=False, startrow=1, header=False)
                ws = writer.sheets[sheet_name]; ws.freeze_panes(1, 0)
                ws.autofilter(0, 0, len(final_df), len(out_cols) - 1)
                for col, val in enumerate(final_df.columns):
                    ws.write(0, col, val, header_fmt)
                    max_len = len(str(val))
                    for cell_val in final_df[val].astype(str): max_len = max(max_len, len(cell_val))
                    ws.set_column(col, col, max_len + 4, center_fmt)

                last_row = len(final_df) + 1
                def get_col(name): return xl_col_to_name(out_cols.index(name))

                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'SHARP', 'format': purple_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Profitable', 'format': green_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'PRIME', 'format': blue_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Validated', 'format': orange_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Safest', 'format': blue_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Avoid', 'format': red_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'BREAKOUT', 'format': red_fmt})

                ws.conditional_format(f'{get_col("Consistency")}2:{get_col("Consistency")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'High', 'format': green_fmt})
                ws.conditional_format(f'{get_col("Consistency")}2:{get_col("Consistency")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Volatile', 'format': red_fmt})
                ws.conditional_format(f'{get_col("Status")}2:{get_col("Status")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Questionable', 'format': orange_fmt})

                # Highlight Easy/Hard Matchups
                ws.conditional_format(f'{get_col("Opp Rank")}2:{get_col("Opp Rank")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Easy', 'format': green_fmt})
                ws.conditional_format(f'{get_col("Opp Rank")}2:{get_col("Opp Rank")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Hard', 'format': red_fmt})

                # Highlight Perfect H2H (1/1, 2/2, etc.)
                for h2h_val in ['1/1', '2/2', '3/3', '4/4']:
                    ws.conditional_format(f'{get_col("vs Opp")}2:{get_col("vs Opp")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': h2h_val, 'format': green_fmt})

                # Highlight Strong Recent Form (8/10, 9/10, 10/10)
                for trend_val in ['8/10', '9/10', '10/10']:
                    ws.conditional_format(f'{get_col("Streak")}2:{get_col("Streak")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': trend_val, 'format': green_fmt})

                f_threes = f'=AND(${get_col("Prop")}2="Threes", ${get_col("Edge (%)")}2>5)'
                ws.conditional_format(f'{get_col("Edge (%)")}2:{get_col("Edge (%)")}{last_row}', {'type': 'formula', 'criteria': f_threes, 'format': green_fmt})
                f_pts = f'=AND(${get_col("Prop")}2="Points", ${get_col("Edge (%)")}2>12)'
                ws.conditional_format(f'{get_col("Edge (%)")}2:{get_col("Edge (%)")}{last_row}', {'type': 'formula', 'criteria': f_pts, 'format': green_fmt})
                f_ar = f'=AND(OR(${get_col("Prop")}2="Assists", ${get_col("Prop")}2="Rebounds"), ${get_col("Edge (%)")}2>8)'
                ws.conditional_format(f'{get_col("Edge (%)")}2:{get_col("Edge (%)")}{last_row}', {'type': 'formula', 'criteria': f_ar, 'format': green_fmt})

                # *** PERFECT BET FORMULA UPDATE ***
                # Added: ISERROR(SEARCH("Hard", ${get_col("Opp Rank")}2))
                # This ensures a "Perfect Bet" (Green Row) is NOT a Hard Matchup.
                base_criteria = f'AND(OR(ISNUMBER(SEARCH("SHARP", ${get_col("Ranking")}2)), ISNUMBER(SEARCH("Validated", ${get_col("Ranking")}2))), ${get_col("Edge (%)")}2>=8, ${get_col("Edge (%)")}2<=30, OR(ISNUMBER(SEARCH("AGREE", ${get_col("Sharp Signal")}2)), ISNUMBER(SEARCH("IMPROVED", ${get_col("Sharp Signal")}2))), ISERROR(SEARCH("Hard", ${get_col("Opp Rank")}2)))'
                pos_criteria = f'IF(${get_col("Prop")}2="Assists", ISNUMBER(SEARCH("G", ${get_col("Pos")}2)), IF(${get_col("Prop")}2="Rebounds", OR(ISNUMBER(SEARCH("F", ${get_col("Pos")}2)), ISNUMBER(SEARCH("C", ${get_col("Pos")}2))), TRUE))'
                final_formula = f'=AND({base_criteria}, {pos_criteria})'
                ws.conditional_format(f'{get_col("Player")}2:{get_col("Player")}{last_row}', {'type': 'formula', 'criteria': final_formula, 'format': perfect_fmt})

            else: writer.book.add_worksheet(sheet_name).write(0, 0, "No bets found.")

        if game_lines_rows and ACTIVE_GAME_MARKETS:
            gl_df = pd.DataFrame(game_lines_rows)
            gl_cols = ['Time', 'Game', 'Retail Book', 'Retail Spread', 'Sharp Spread', 'Model Spread', 'Feng Spread', 'Spread Diff', 'Spread Pick', 'Retail Total', 'Sharp Total', 'Model Total', 'Total Diff', 'Total Pick', 'Retail ML', 'Model Win %', 'Log5 Win %', 'ML Pick', 'Home Luck', 'Home Net Rtg', 'Home eFG%', 'Signal']
            gl_df = gl_df.reindex(columns=gl_cols).sort_values(['Signal', 'Spread Diff'], ascending=[False, False])
            gl_df.to_excel(writer, sheet_name='Game_Lines', index=False); ws_gl = writer.sheets['Game_Lines']
            ws_gl.freeze_panes(1, 0)
            ws_gl.autofilter(0, 0, len(gl_df), len(gl_cols) - 1)
            for col, val in enumerate(gl_df.columns):
                ws_gl.write(0, col, val, header_fmt)
                ws_gl.set_column(col, col, 15 + 4, center_fmt)
            last_row = len(gl_df) + 1; sig_col = xl_col_to_name(gl_cols.index('Signal'))
            ws_gl.conditional_format(f'{sig_col}2:{sig_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'SHARP', 'format': purple_fmt})
            ws_gl.conditional_format(f'{sig_col}2:{sig_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'MODEL', 'format': blue_fmt})
            ws_gl.conditional_format(f'{sig_col}2:{sig_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Diff', 'format': orange_fmt})

    print(f"\n✅ Ultimate Analysis Exported: {out_file}")

    if not final_bets.empty:
        print("\n" + "="*60); print("🏆 BEST BET PER GAME"); print("="*60)
        best_card = final_bets.copy()
        best_card['Win_Prob'] = np.where(best_card['Bet'] == 'Over', best_card['Prob_Over'], best_card['Prob_Under'])
        best_card['Priority'] = best_card['Sharp Signal'].apply(lambda x: 1 if x in ['PRICE IMPROVED', 'SHARPS AGREE'] else 0)
        best_card = best_card.sort_values(['Game', 'Priority', 'Edge (%)'], ascending=[True, False, False])
        quick_card = best_card.groupby('Game').head(1).sort_values('Game_Time')
        quick_card['Win_Prob'] = (quick_card['Win_Prob'] * 100).round(1).astype(str) + "%"
        quick_card['Edge (%)'] = quick_card['Edge (%)'].round(1).astype(str) + "%"
        display(quick_card[['Game_Time', 'Game', 'Player', 'Status', 'Prop', 'Bet', 'Consistency', 'Line', 'Edge (%)', 'Win_Prob', 'Sharp Signal']].rename(columns={'Sharp Signal': 'Signal'}))

    if final_bets.empty: print("\nNOTE: No bets met the strict criteria.")
    print("\n✅ Script Completed. Exiting...")

if __name__ == "__main__":
    pd.set_option('display.max_rows', 100); pd.set_option('display.max_columns', None)
    run_daily_projections()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.0/319.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.1 MB/s eta 0:00:00

📅 Running Live Mode. Current ET Time: 2026-01-30 20:22:54
Running projections for 2026-01-30 (Season: 2025-26)
✅ All models, imputers, and MAE values loaded successfully.

Fetching live injury report from ESPN API (NBA)...


Fetching Team Rosters for Injuries: 100%|██████████| 30/30 [00:07<00:00,  4.19it/s]


   ✅ Successfully fetched 104 injury entries.
Loading Player Positions from: /content/drive/MyDrive/NBA/Player Positions/2025-26 NBA Player Positions.xlsx
✅ Successfully loaded 509 player positions.
   ℹ️ Detected 92 ruled-out players. Adjusting team usage automatically...

--- Preparing Context & Stats ---
✅ Successfully fetched season stats.
   🛡️ Fetching Opponent Defense Rankings (Calculating Manually)...
   🧮 Calculating Feng/Massey Ratings (Strength of Schedule)...
   📊 Fetching Team Metrics for 2025-26...
      ⚠️ Attempting Fallback to Standard Advanced Stats...
      ✅ Successfully fetched Advanced Stats (Fallback).
--- Fetching Schedule for 2026-01-30 via API ---
✅ Found 9 valid games on the schedule.

--- Running Initial Projections & Monte Carlo Sims ---


Generating Projections: 100%|██████████| 18/18 [06:54<00:00, 23.01s/it]


✅ Success: Generated 281 player projections. Proceeding to fetch odds...
Using API key ending in ...94b4
--- Fetching Sharp & Retail Odds ---


...

Debug Props Keys: ['luka doncic', 'luka doncic', 'alex sarr']

--- Running Value Analysis & Filtering Injuries ---
ℹ️ No earlier run found for TODAY (01-30-2026). This is likely the 'Pro Window' (First Run).

--- Calculating Player Streaks, Opponent Ranks & H2H ---


Analyzing Trends: 100%|██████████| 1123/1123 [00:21<00:00, 52.86it/s]



✅ Ultimate Analysis Exported: /content/drive/MyDrive/NBA/Predictions/01-30-2026 NBA Player Props.xlsx

🏆 BEST BET PER GAME


,Game_Time,Game,Player,Status,Prop,Bet,Consistency,Line,Edge (%),Win_Prob,Signal
29,10:10 PM,DET @ GSW,Al Horford,Active,REB,Over,✅ Med,5.5,99.9%,92.5%,-
75,7:10 PM,LAL @ WAS,Bilal Coulibaly,Active,REB,Over,⚡ Volatile,2.5,136.2%,93.0%,-
987,7:41 PM,MEM @ NOP,Yves Missi,Active,REB,Over,⚡ Volatile,4.5,122.9%,99.5%,-
789,7:41 PM,SAC @ BOS,Payton Pritchard,Active,PTS,Under,⚡ Volatile,26.5,92.4%,97.2%,-
815,7:42 PM,POR @ NYK,Robert Williams,Active,PTS,Over,⚡ Volatile,4.5,134.9%,100.0%,-
873,7:43 PM,TOR @ ORL,Scottie Barnes,Active,AST,Over,⚡ Volatile,4.5,109.2%,87.2%,-
217,9:00 PM,CLE @ PHX,Dean Wade,Active,P+R,Over,⚡ Volatile,11.5,93.8%,97.9%,-
436,9:30 PM,BKN @ UTA,Isaiah Collier,Active,P+A,Over,✅ Med,15.5,95.4%,98.7%,-



✅ Script Completed. Exiting...


In [ ]:
# ==============================================================================
# SCRIPT 3: NBA Professional Sharp & ML Engine (v96 - Advanced Game Engine)
# Features: Monte Carlo Sims, Feng Ratings, Log5, Luck Metric, + Prop Features
# ==============================================================================

# --- Part 0: Setup & Installs ---
!pip install -q xlsxwriter nba_api scikit-learn pandas joblib pytz unidecode requests lightgbm python-dateutil tqdm openpyxl scipy

import os
import re
import time
import joblib
import warnings
import requests
import json
import numpy as np
import pandas as pd
import pytz
from datetime import datetime, timedelta
from dateutil import parser
from unidecode import unidecode
from tqdm import tqdm
from decimal import Decimal
from scipy.stats import poisson, norm
from google.colab import drive
from IPython.display import display, clear_output
from xlsxwriter.utility import xl_col_to_name

from nba_api.stats.endpoints import playergamelog, leaguedashplayerstats, teamestimatedmetrics, drafthistory, scoreboardv2, leaguedashteamstats, leaguegamefinder
from nba_api.stats.static import teams, players

warnings.filterwarnings("ignore")

# --- Part 1: Configuration ---
GDRIVE_BASE_PATH = "/content/drive/MyDrive/NBA/"
MODEL_DIR = os.path.join(GDRIVE_BASE_PATH, "Training Model Advanced/")
OUTPUT_DIR = os.path.join(GDRIVE_BASE_PATH, "Predictions/")
GAMELOG_LAKE_DIR = os.path.join(GDRIVE_BASE_PATH, "gamelog_lake/")
os.makedirs(OUTPUT_DIR, exist_ok=True); os.makedirs(GAMELOG_LAKE_DIR, exist_ok=True)

# *** 1. MARKET TOGGLES ***
ENABLE_MARKETS = {
    "Moneyline": True, "Spreads": True, "Totals": True,
    "Points": True, "Rebounds": True, "Assists": True, "Threes": True,
    "Combos": True, "DoubleDouble": True, "TripleDouble": True
}

FILTER_OUT_PLAYERS = True

_GAME_MAP = {"Moneyline": "h2h", "Spreads": "spreads", "Totals": "totals"}
_PROP_MAP = {
    "Points": {'player_points': 'PTS'}, "Rebounds": {'player_rebounds': 'REB'}, "Assists": {'player_assists': 'AST'},
    "Threes": {'player_threes': 'FG3M'},
    "Combos": {'player_points_rebounds_assists': 'PRA', 'player_points_rebounds': 'P+R', 'player_points_assists': 'P+A', 'player_rebounds_assists': 'R+A'},
    "DoubleDouble": {'player_double_double': 'DD'}, "TripleDouble": {'player_triple_double': 'TD'}
}

ACTIVE_GAME_MARKETS = [v for k, v in _GAME_MAP.items() if ENABLE_MARKETS.get(k, False)]
MARKET_NAME_MAP = {}
for category, enabled in ENABLE_MARKETS.items():
    if enabled and category in _PROP_MAP: MARKET_NAME_MAP.update(_PROP_MAP[category])
ALL_PROP_MARKETS_LIST = list(MARKET_NAME_MAP.keys())

# *** EXACT KEYS ***
ODDS_API_KEYS = [
    "9b96859c5e0781bb4c5809215a5ed94b", "f4cfcea7cc6120a97559f69d6049596e", "7b9405e2febc561e554b7169a42cefd1", "146dff42ae19a83208b059d4855095c9",
    "f8474479351ccc685ccae9f4716c23c5", "39756c093848b6968c899a4e1ded85f9", "9705a1f6b609305047eb8ec9243cc54a", "4217574ccdd90732483d98f24772f66b",
    "66953d96fcb2ec72a47e7f39901258e7", "94304f58fb95fd1d45159b9c8d292724", "e384411e342cac5c9408804c622703f1", "dab5c3da80b0415f64d4d94890974cdf",
    "9705a1f6b609305047eb8ec9243cc54a", "3aae94b51cc3dcb8e0623bf2633a695a", "a4e4d3eef7d360055b96a1351560817f", "1bfcac154fb2b490bf55a80a18275db1",
    "7b64d450f91079f60211cb33033594b4", "ec9dc1e1aa0543c1b3fa3d2576fabffd", "6bbfe433274aef6fee5aa5543ad69055", "1f5e9cdc190854c605b533917b2c9b82",
    "858b7234b5f77661345f205b1c5ec121", "39756c093848b6968c899a4e1ded85f9", "21caebe8058c338a7ba76f6c9fb550cb", "94304f58fb95fd1d45159b9c8d292724",
    "129b9d482b270ff3809e11b0c445ef5c", "a11767c08916b29f824150bffbbf6d9c", "e7e814c95b32c4a9fb5b72acefb7389b", "3a3cd8a78555c46afa842aa8bdf1455e",
    "3b9adf0f0e27964f58de1f9b4af3adfc", "f098da95884f42852df5ff12f5704741", "768fb25c7f33328bc53b61b81e607bd9"
]

SHARP_BOOKS = ["pinnacle", "betfair_ex_eu", "betonlineag", "bovada", "betus", "betrivers", "betmgm", "espnbet"]
RETAIL_BOOKS = ["draftkings", "fanduel"]
ALL_BOOKS_FOR_API = list(set(SHARP_BOOKS + RETAIL_BOOKS))
ROLLING_WINDOWS = [5, 10, 20]; CONFIG = { "GAMES_PLAYED_THRESHOLD_ACCLIMATED": 15, "GAMES_PLAYED_THRESHOLD_LOW_VOLUME": 20 }

# --- Part 2: Helper Functions ---
def normalize_player_name(name):
    if name is None: return None
    try: name = str(name)
    except: return None
    name = unidecode(name.lower()); name = re.sub(r'\s+(jr|sr|ii|iii|iv)\.?$', '', name); name = re.sub(r'[^\w\s]', '', name); return re.sub(r'\s+', ' ', name).strip()
def get_team_abbr_map(): return {team['full_name']: team['abbreviation'] for team in teams.get_teams()}

def convert_min_to_decimal(min_str):
    if isinstance(min_str, str) and ':' in min_str:
        try: minutes, seconds = map(int, min_str.split(':')); return minutes + seconds / 60.0
        except: return np.nan
    return pd.to_numeric(min_str, errors='coerce')

# *** Smart Proxy Position Logic ***
def derive_position(row):
    pts = row.get('Pred_PTS', 0); reb = row.get('Pred_REB', 0); ast = row.get('Pred_AST', 0)
    if reb >= 8.0: return 'C'
    if reb >= 6.0 and ast < 2.5: return 'C/F'
    if ast >= 6.0: return 'PG'
    if ast >= 4.5 and reb < 4.0: return 'PG'
    if reb >= 5.0 and pts >= 12.0: return 'F'
    if pts >= 10.0 and reb < 4.5: return 'G'
    return 'G/F'

def fetch_espn_nba_injuries():
    print("\nFetching live injury report from ESPN API (NBA)...")
    try:
        teams_url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams"
        teams_data = requests.get(teams_url, timeout=10).json()
        team_list = teams_data['sports'][0]['leagues'][0]['teams']
        all_injuries = []
        for team in tqdm(team_list, desc="Fetching Team Rosters for Injuries"):
            team_id = team['team']['id']
            roster_url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams/{team_id}?enable=roster"
            try: roster_data = requests.get(roster_url, timeout=10).json()
            except Exception: continue
            for player in roster_data.get('team', {}).get('athletes', []):
                if 'injuries' in player and player['injuries']:
                    injury_obj = player['injuries'][0]
                    status = injury_obj.get('status', 'Unknown')
                    all_injuries.append({'Player_ESPN': player.get('displayName'), 'Status': status, 'Injury Details': injury_obj.get('date', '') + " " + injury_obj.get('type', {}).get('description', '')})

        if not all_injuries: print("   ⚠️ WARNING: No injury entries found."); return pd.DataFrame()
        injury_df = pd.DataFrame(all_injuries)
        injury_df['match_key'] = injury_df['Player_ESPN'].apply(normalize_player_name)
        print(f"   ✅ Successfully fetched {len(injury_df)} injury entries.")
        return injury_df
    except Exception as e: print(f"   ❌ FAILED to fetch ESPN injuries: {e}"); return pd.DataFrame()

# --- NEW HELPER: Robust Team Metrics (Fail-Safe) ---
def get_live_team_metrics(season):
    print(f"   📊 Fetching Team Metrics for {season}...")
    metrics_dict = {}

    # ATTEMPT 1: TeamEstimatedMetrics (Preferred)
    try:
        metrics_df = teamestimatedmetrics.TeamEstimatedMetrics(season=season, league_id='00', timeout=60).get_data_frames()[0]
        if not metrics_df.empty:
            cols = ['E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING', 'E_PACE', 'E_EFG_PCT', 'E_OREB_PCT', 'GP', 'W', 'L']
            metrics_dict = metrics_df.set_index('TEAM_ID')[cols].to_dict('index')
            print(f"      ✅ Successfully fetched Estimated Metrics.")
            return metrics_dict
    except Exception: pass

    # ATTEMPT 2: LeagueDashTeamStats (Fallback - Advanced)
    if not metrics_dict:
        print("      ⚠️ Attempting Fallback to Standard Advanced Stats...")
        try:
            adv_stats = leaguedashteamstats.LeagueDashTeamStats(
                season=season,
                measure_type_detailed_defense='Advanced',
                timeout=60
            ).get_data_frames()[0]

            if not adv_stats.empty:
                # Map Standard columns to the expected E_ columns
                adv_stats['E_OFF_RATING'] = adv_stats['OFF_RATING']
                adv_stats['E_DEF_RATING'] = adv_stats['DEF_RATING']
                adv_stats['E_NET_RATING'] = adv_stats['NET_RATING']
                adv_stats['E_PACE'] = adv_stats['PACE']
                adv_stats['E_EFG_PCT'] = adv_stats['EFG_PCT']
                adv_stats['E_OREB_PCT'] = adv_stats['OREB_PCT']

                cols = ['E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING', 'E_PACE', 'E_EFG_PCT', 'E_OREB_PCT', 'GP', 'W', 'L']
                metrics_dict = adv_stats.set_index('TEAM_ID')[cols].to_dict('index')
                print(f"      ✅ Successfully fetched Advanced Stats (Fallback).")
            else:
                print("      ❌ Error: Advanced Stats returned empty.")
        except Exception as e:
            print(f"      ❌ Failed to fetch fallback stats: {e}")

    return metrics_dict

# --- NEW HELPER: Fetch Team Defense Rankings (DUAL PARAM FETCH) ---
def get_defense_ranks(season):
    print("   🛡️ Fetching Opponent Defense Rankings (Calculating Manually)...")
    stats = pd.DataFrame()
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(season=season, per_mode_detailed='PerGame', measure_type_detailed_defense='Opponent', timeout=60).get_data_frames()[0]
    except Exception: pass
    if stats.empty:
        try: stats = leaguedashteamstats.LeagueDashTeamStats(season=season, per_mode_detailed='PerGame', measure_type_nullable='Opponent', timeout=60).get_data_frames()[0]
        except Exception: pass

    if stats.empty: return {}, {}

    try:
        col_pts = 'OPP_PTS' if 'OPP_PTS' in stats.columns else 'PTS'
        col_reb = 'OPP_REB' if 'OPP_REB' in stats.columns else 'REB'
        col_ast = 'OPP_AST' if 'OPP_AST' in stats.columns else 'AST'
        col_3pm = 'OPP_FG3M' if 'OPP_FG3M' in stats.columns else 'FG3M'

        if col_pts in stats.columns: stats['RANK_PTS'] = stats[col_pts].rank(ascending=True)
        else: stats['RANK_PTS'] = 15
        if col_reb in stats.columns: stats['RANK_REB'] = stats[col_reb].rank(ascending=True)
        else: stats['RANK_REB'] = 15
        if col_ast in stats.columns: stats['RANK_AST'] = stats[col_ast].rank(ascending=True)
        else: stats['RANK_AST'] = 15
        if col_3pm in stats.columns: stats['RANK_FG3M'] = stats[col_3pm].rank(ascending=True)
        else: stats['RANK_FG3M'] = 15

        ranks = {}
        raw_stats = {}
        for _, row in stats.iterrows():
            tid = str(row['TEAM_ID'])
            ranks[tid] = {'PTS': int(row['RANK_PTS']), 'REB': int(row['RANK_REB']), 'AST': int(row['RANK_AST']), 'FG3M': int(row['RANK_FG3M'])}
            raw_stats[tid] = {'PTS': float(row.get(col_pts, 115.0)), 'REB': float(row.get(col_reb, 44.0)), 'AST': float(row.get(col_ast, 25.0)), 'FG3M': float(row.get(col_3pm, 12.0))}
        return ranks, raw_stats
    except Exception as e: return {}, {}

# --- NEW HELPER: Feng (Massey) Ratings using Linear Algebra ---
def get_nba_feng_ratings(season):
    print("   🧮 Calculating Feng/Massey Ratings (Strength of Schedule)...")
    try:
        games = leaguegamefinder.LeagueGameFinder(season_nullable=season, league_id_nullable='00', season_type_nullable='Regular Season', timeout=60).get_data_frames()[0]
        if games.empty: return {}

        # Filter for completed games only
        games = games[games['WL'].isin(['W', 'L'])]
        games['GAME_DATE'] = pd.to_datetime(games['GAME_DATE'])
        # Drop duplicates (since API returns one row per team per game)
        unique_games = games.drop_duplicates(subset=['GAME_ID'])

        # Need full matchup to calc margin. Since we have one row, we need to find the opponent score.
        # Actually, simpler method: Process pairs.
        # Create a dictionary of results

        team_ids = sorted(games['TEAM_ID'].unique())
        n_teams = len(team_ids)
        team_to_idx = {t: i for i, t in enumerate(team_ids)}

        M = np.zeros((n_teams, n_teams))
        p = np.zeros(n_teams)

        # We need pairs. Group by GAME_ID
        grouped = games.groupby('GAME_ID')

        for _, g_df in grouped:
            if len(g_df) != 2: continue
            row1 = g_df.iloc[0]; row2 = g_df.iloc[1]
            t1, t2 = row1['TEAM_ID'], row2['TEAM_ID']
            s1, s2 = row1['PTS'], row2['PTS']

            margin = s1 - s2
            i, j = team_to_idx[t1], team_to_idx[t2]

            p[i] += margin
            p[j] -= margin

            M[i, i] += 1
            M[j, j] += 1
            M[i, j] -= 1
            M[j, i] -= 1

        # Constraint to solve singular matrix
        M[-1, :] = 1
        p[-1] = 0

        ratings = np.linalg.solve(M, p)
        return {str(team_ids[i]): ratings[i] for i in range(n_teams)}

    except Exception as e:
        print(f"   ⚠️ Feng Ratings calculation failed: {e}")
        return {}

# --- NEW HELPER: Calculate Streak vs Line ---
def calculate_prop_streak(player_id, prop_map, line):
    try:
        gamelog_file = os.path.join(GAMELOG_LAKE_DIR, f"{player_id}.csv")
        if not os.path.exists(gamelog_file): return 0, "-"
        df = pd.read_csv(gamelog_file)
        if df.empty: return 0, "-"
        df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
        df = df.sort_values(by='GAME_DATE', ascending=False).reset_index(drop=True)
        val_series = None
        if prop_map == 'PTS': val_series = df['PTS']
        elif prop_map == 'REB': val_series = df['REB']
        elif prop_map == 'AST': val_series = df['AST']
        elif prop_map == 'FG3M': val_series = df['FG3M']
        elif prop_map == 'PRA': val_series = df['PTS'] + df['REB'] + df['AST']
        elif prop_map == 'P+R': val_series = df['PTS'] + df['REB']
        elif prop_map == 'P+A': val_series = df['PTS'] + df['AST']
        elif prop_map == 'R+A': val_series = df['REB'] + df['AST']
        elif prop_map == 'DD': val_series = ((df['PTS']>=10).astype(int) + (df['REB']>=10).astype(int) + (df['AST']>=10).astype(int)) >= 2

        if val_series is None: return 0, "-"

        streak = 0
        for val in val_series:
            if prop_map == 'DD':
                if val: streak += 1
                else: break
            else:
                if val > line: streak += 1
                else: break

        l10_slice = val_series.head(10)
        hits_in_l10 = 0
        if prop_map == 'DD': hits_in_l10 = l10_slice.sum()
        else: hits_in_l10 = (l10_slice > line).sum()

        return streak, f"{int(hits_in_l10)}/10"
    except Exception: return 0, "-"

# --- NEW HELPER: Robust H2H ---
def calculate_head_to_head(player_id, prop_map, line, opponent_abbr, season_year):
    try:
        gamelog_file = os.path.join(GAMELOG_LAKE_DIR, f"{player_id}.csv")
        if not os.path.exists(gamelog_file): return "-"
        df = pd.read_csv(gamelog_file)
        if df.empty: return "-"
        df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
        season_start = pd.Timestamp(f"{season_year}-09-01")
        df = df[df['GAME_DATE'] >= season_start]
        if df.empty: return "0/0"
        target_opp = str(opponent_abbr).upper()
        abbr_map = {'PHX': 'PHO', 'PHO': 'PHX', 'GSW': 'GS', 'GS': 'GSW', 'NOP': 'NO', 'NO': 'NOP', 'SAS': 'SA', 'SA': 'SAS', 'NYK': 'NY', 'NY': 'NYK', 'UTA': 'UT', 'UT': 'UTA', 'WAS': 'WSH', 'WSH': 'WAS'}
        alt_opp = abbr_map.get(target_opp, target_opp)
        def is_matchup(m_str):
            if pd.isna(m_str): return False
            try:
                m_str = m_str.replace('.', ''); tokens = m_str.split()
                actual_opp = tokens[-1].upper()
                return actual_opp == target_opp or actual_opp == alt_opp
            except: return False
        df_vs = df[df['MATCHUP'].apply(is_matchup)]
        if df_vs.empty: return "0/0"
        val_series = None
        if prop_map == 'PTS': val_series = df_vs['PTS']
        elif prop_map == 'REB': val_series = df_vs['REB']
        elif prop_map == 'AST': val_series = df_vs['AST']
        elif prop_map == 'FG3M': val_series = df_vs['FG3M']
        elif prop_map == 'PRA': val_series = df_vs['PTS'] + df_vs['REB'] + df_vs['AST']
        elif prop_map == 'P+R': val_series = df_vs['PTS'] + df_vs['REB']
        elif prop_map == 'P+A': val_series = df_vs['PTS'] + df_vs['AST']
        elif prop_map == 'R+A': val_series = df_vs['REB'] + df_vs['AST']
        elif prop_map == 'DD': val_series = ((df_vs['PTS']>=10).astype(int) + (df_vs['REB']>=10).astype(int) + (df_vs['AST']>=10).astype(int)) >= 2
        if val_series is None: return "-"
        hits = 0
        if prop_map == 'DD': hits = val_series.sum()
        else: hits = (val_series > line).sum()
        return f"{int(hits)}/{len(df_vs)}"
    except Exception: return "-"

# --- Part 5: Main Execution Workflow (Updated for Monte Carlo) ---
def run_daily_projections():
    proj_df = pd.DataFrame()

    try:
        if not os.path.exists('/content/drive'): drive.mount('/content/drive', force_remount=True)
    except Exception as e: print(f"⚠️ Drive mount check skipped: {e}")

    eastern = pytz.timezone('US/Eastern')
    SIMULATION_DATE = None

    if 'SIMULATION_DATE' in globals() and globals()['SIMULATION_DATE'] is not None:
         try: datetime.strptime(globals()['SIMULATION_DATE'], "%Y-%m-%d"); SIMULATION_DATE = globals()['SIMULATION_DATE']
         except: SIMULATION_DATE = None

    if SIMULATION_DATE:
        live_now_et = datetime.strptime(f"{SIMULATION_DATE} 12:00:00", "%Y-%m-%d %H:%M:%S").astimezone(eastern)
        print(f"\n⚠️ RUNNING IN SIMULATION MODE for date: {SIMULATION_DATE}")
        iso_today_str = SIMULATION_DATE
        file_today_str = live_now_et.strftime('%m-%d-%Y')
    else:
        live_now_et = datetime.now(eastern)
        print(f"\n📅 Running Live Mode. Current ET Time: {live_now_et.strftime('%Y-%m-%d %H:%M:%S')}")
        iso_today_str = live_now_et.strftime('%Y-%m-%d')
        file_today_str = live_now_et.strftime('%m-%d-%Y')

    if live_now_et.month >= 9: start_year = live_now_et.year; end_year = live_now_et.year + 1
    else: start_year = live_now_et.year - 1; end_year = live_now_et.year
    season = f"{start_year}-{str(end_year)[-2:]}"; prev_season = f"{start_year-1}-{str(start_year)[-2:]}"; season_year_for_url = str(start_year)

    print(f"Running projections for {iso_today_str} (Season: {season})")

    try:
        models = {'minutes': joblib.load(os.path.join(MODEL_DIR, "minutes_lgbm_model.pkl")), 'pts': joblib.load(os.path.join(MODEL_DIR, "pts_lgbm_model.pkl")), 'reb': joblib.load(os.path.join(MODEL_DIR, "reb_lgbm_model.pkl")), 'ast': joblib.load(os.path.join(MODEL_DIR, "ast_lgbm_model.pkl")), 'fg3m': joblib.load(os.path.join(MODEL_DIR, "fg3m_lgbm_model.pkl")), 'dd': joblib.load(os.path.join(MODEL_DIR, "dd_lgbm_model.pkl")), 'td': joblib.load(os.path.join(MODEL_DIR, "td_lgbm_model.pkl"))}
        imputers = {name: joblib.load(os.path.join(MODEL_DIR, f"{name}_imputer.pkl")) for name in ['minutes', 'stats']}
        min_feature_names = joblib.load(os.path.join(MODEL_DIR, "minutes_feature_names.pkl")); stat_feature_names = joblib.load(os.path.join(MODEL_DIR, "stats_feature_names.pkl")); mae_values = json.load(open(os.path.join(MODEL_DIR, "mae_values.json")))
        print("✅ All models, imputers, and MAE values loaded successfully.")
    except FileNotFoundError as e: print(f"CRITICAL ERROR: A required model file is missing. Please run Script 2. Details: {e}"); return

    injury_df = fetch_espn_nba_injuries()

    # *** LOAD PLAYER POSITIONS FROM EXCEL ***
    position_map = {}
    POSITIONS_FILE = os.path.join(GDRIVE_BASE_PATH, "Player Positions/2025-26 NBA Player Positions.xlsx")
    if os.path.exists(POSITIONS_FILE):
        try:
            print(f"Loading Player Positions from: {POSITIONS_FILE}")
            pos_df = pd.read_excel(POSITIONS_FILE)
            if 'Player' in pos_df.columns and 'Pos' in pos_df.columns:
                pos_df['match_key'] = pos_df['Player'].apply(normalize_player_name)
                position_map = pos_df.set_index('match_key')['Pos'].to_dict()
                print(f"✅ Successfully loaded {len(position_map)} player positions.")
            else: print("⚠️ Warning: Position file missing 'Player' or 'Pos' columns.")
        except Exception as e: print(f"⚠️ Error loading Position file: {e}")
    else: print(f"⚠️ Warning: Position file not found at {POSITIONS_FILE}. Using proxy logic.")

    scratched_players = []
    if not injury_df.empty:
        scratched_players = injury_df[injury_df['Status'].isin(['Out', 'Injured Reserve', 'Suspended'])]['match_key'].tolist()

    prev_file = os.path.join(OUTPUT_DIR, f"{file_today_str} NBA Player Props.xlsx")
    if os.path.exists(prev_file):
        try:
            old_df = pd.read_excel(prev_file, sheet_name='DraftKings_EV_Bets')
            if 'Status' in old_df.columns and 'Player' in old_df.columns:
                old_active = old_df[old_df['Status'] == 'Active']['Player'].apply(normalize_player_name).tolist()
                newly_scratched = [p for p in old_active if p in scratched_players]
                if newly_scratched:
                    print(f"   🚨 AUTOMATED ALERT: Detected new scratches: {newly_scratched}")
        except: pass

    if scratched_players: print(f"   ℹ️ Detected {len(scratched_players)} ruled-out players. Adjusting team usage automatically...")

    print("\n--- Preparing Context & Stats ---")
    stats_current_season, stats_last_season = pd.DataFrame(), pd.DataFrame()
    for attempt in range(3):
        try:
            stats_current_season = leaguedashplayerstats.LeagueDashPlayerStats(season=season, timeout=120).get_data_frames()[0]
            stats_last_season = leaguedashplayerstats.LeagueDashPlayerStats(season=prev_season, timeout=120).get_data_frames()[0]
            if not stats_current_season.empty and not stats_last_season.empty: print("✅ Successfully fetched season stats."); break
        except Exception: time.sleep(5)

    if stats_current_season.empty: print("⚠️ Warning: Could not fetch current season stats.")

    # --- GET DEFENSE RANKS & RAW STATS ---
    def_ranks, def_stats_raw = get_defense_ranks(season)
    if not def_ranks: print("   ⚠️ Warning: Defense Ranks are empty. API might be down or Season mismatch.")

    # --- GET FENG RATINGS ---
    feng_ratings = get_nba_feng_ratings(season)

    team_metrics = get_live_team_metrics(season)
    schedule = get_live_schedule_from_nba_data(iso_today_str, season_year_for_url) # USE ISO DATE HERE
    if not schedule: print("⚠️ Warning: NBA API returned 0 games. Will attempt to rebuild schedule from Betting Odds.")

    schedule = {str(k): str(v) for k, v in schedule.items() if k is not None and v is not None}
    if not stats_current_season.empty: stats_current_season['TEAM_ID'] = stats_current_season['TEAM_ID'].astype(str).str.split('.').str[0]

    rookie_ids = get_draft_class_ids(season_year_for_url); traded_player_ids = identify_team_changes(stats_last_season, stats_current_season)
    props_df, live_game_lines = pd.DataFrame(), {}
    key, game_events = None, []
    team_abbr_map = get_team_abbr_map()

    print("\n--- Running Initial Projections & Monte Carlo Sims ---")
    all_rosters = stats_current_season
    projection_rows, model_game_rows = [], []

    LEAGUE_AVG_PACE = 99.0 # Approximate default
    LEAGUE_AVG_OFF = 115.0 # Approximate default

    # Calculate League Averages if metrics available
    if team_metrics:
        paces = [m['E_PACE'] for m in team_metrics.values() if m['E_PACE'] > 0]
        offs = [m['E_OFF_RATING'] for m in team_metrics.values() if m['E_OFF_RATING'] > 0]
        if paces: LEAGUE_AVG_PACE = sum(paces) / len(paces)
        if offs: LEAGUE_AVG_OFF = sum(offs) / len(offs)

    if schedule:
        for team_id, opponent_id in tqdm(schedule.items(), desc="Generating Projections"):
            # --- 1. PLAYER PROJECTIONS (EXISTING LOGIC) ---
            roster = all_rosters[all_rosters['TEAM_ID'] == str(team_id)]
            team_scratches = [p for p in scratched_players if p in roster['PLAYER_NAME'].apply(normalize_player_name).tolist()]
            usage_boost_factor = 1.0 + (len(team_scratches) * 0.05)

            for _, row in roster.iterrows():
                if 'GP' in row and row['GP'] > 0 and (row['MIN'] / row['GP']) < 6.0: continue
                player_id, player_name, team_abbr = row['PLAYER_ID'], row['PLAYER_NAME'], row['TEAM_ABBREVIATION']
                match_key = normalize_player_name(player_name)
                logs = get_and_update_gamelogs(player_id, season, prev_season); features = calculate_live_features(logs)
                if not features: continue
                try: opp_id_int = int(opponent_id); opp_metrics = team_metrics.get(opp_id_int, {})
                except: opp_metrics = {}

                features['OPP_ALLOWED_PTS_AVG_30'] = opp_metrics.get('E_DEF_RATING', 115)
                features['OPP_ALLOWED_REB_AVG_30'] = 44.0; features['OPP_ALLOWED_AST_AVG_30'] = 25.0; features['OPP_ALLOWED_FG3M_AVG_30'] = 12.0

                if str(opponent_id) in def_stats_raw:
                    features['OPP_ALLOWED_PTS_AVG_30'] = def_stats_raw[str(opponent_id)]['PTS']
                    features['OPP_ALLOWED_REB_AVG_30'] = def_stats_raw[str(opponent_id)]['REB']
                    features['OPP_ALLOWED_AST_AVG_30'] = def_stats_raw[str(opponent_id)]['AST']
                    features['OPP_ALLOWED_FG3M_AVG_30'] = def_stats_raw[str(opponent_id)]['FG3M']

                features['VEGAS_SPREAD'] = 0; features['VEGAS_TOTAL'] = 225

                try:
                    min_input = pd.DataFrame([features])[min_feature_names]; pred_min = models['minutes'].predict(imputers['minutes'].transform(min_input))[0]; features['Proj_MIN'] = pred_min
                    if usage_boost_factor > 1.0:
                        features['FGA_PM'] = features.get('FGA_PM', 0) * usage_boost_factor
                        pred_min = pred_min * 1.02

                    stat_input = imputers['stats'].transform(pd.DataFrame([features])[stat_feature_names])
                    p_pts = models['pts'].predict(stat_input)[0]; p_reb = models['reb'].predict(stat_input)[0]
                    p_ast = models['ast'].predict(stat_input)[0]; p_3pm = models['fg3m'].predict(stat_input)[0]

                    temp_proj = {'Pred_PTS': p_pts, 'Pred_REB': p_reb, 'Pred_AST': p_ast}
                    if match_key in position_map: player_pos = position_map[match_key]
                    else: player_pos = derive_position(temp_proj)

                    proj = {
                        'PLAYER_ID': player_id, 'match_key': match_key, 'Player': player_name,
                        'Pos': player_pos, 'Experience': get_player_experience(player_id, rookie_ids, traded_player_ids, stats_last_season, stats_current_season),
                        'Team': team_abbr, 'Opponent': teams.find_team_name_by_id(int(opponent_id))['abbreviation'] if opponent_id else "UNK",
                        'OpponentID': str(opponent_id), 'Pred_MIN': pred_min, 'Pred_FGA_PM': features.get('FGA_PM', 0),
                        'Pred_PTS': p_pts, 'Pred_REB': p_reb, 'Pred_AST': p_ast, 'Pred_FG3M': p_3pm
                    }
                    proj['CV_PTS'] = features.get('CV_PTS', 9.9); proj['CV_REB'] = features.get('CV_REB', 9.9)
                    proj['CV_AST'] = features.get('CV_AST', 9.9); proj['CV_FG3M'] = features.get('CV_FG3M', 9.9)
                    clf_features = np.array([proj['Pred_PTS'], proj['Pred_REB'], proj['Pred_AST']]).reshape(1, -1)
                    proj['Pred_DD'] = models['dd'].predict_proba(clf_features)[0][1] * 100; proj['Pred_TD'] = models['td'].predict_proba(clf_features)[0][1] * 100
                    proj['Pred_PRA'] = proj['Pred_PTS'] + proj['Pred_REB'] + proj['Pred_AST']; proj['Pred_P+R'] = proj['Pred_PTS'] + proj['Pred_REB']; proj['Pred_P+A'] = proj['Pred_PTS'] + proj['Pred_AST']; proj['Pred_R+A'] = proj['Pred_REB'] + proj['Pred_AST']
                    projection_rows.append(proj)
                except Exception: continue

            # --- 2. GAME ENGINE (MONTE CARLO UPGRADE) ---
            home_stats, away_stats = team_metrics.get(int(team_id), {}), team_metrics.get(int(opponent_id), {})
            if home_stats and away_stats:
                # Basic Vars
                h_off, a_off = home_stats.get('E_OFF_RATING', 110), away_stats.get('E_OFF_RATING', 110)
                h_def, a_def = home_stats.get('E_DEF_RATING', 110), away_stats.get('E_DEF_RATING', 110)
                h_pace, a_pace = home_stats.get('E_PACE', 99), away_stats.get('E_PACE', 99)

                # Monte Carlo Vars
                poss = (h_pace + a_pace) / 2
                HCA = 2.5 # Home Court Advantage Points

                # Expected Scores (KenPom Style)
                h_exp = (h_off/100) * (a_def/100) * (100 / (LEAGUE_AVG_OFF/100)) * poss + (HCA/2)
                a_exp = (a_off/100) * (h_def/100) * (100 / (LEAGUE_AVG_OFF/100)) * poss - (HCA/2)

                # SIMULATION
                SIMS = 5000
                scores = np.random.multivariate_normal([h_exp, a_exp], [[169, 50], [50, 169]], size=SIMS) # 13 std dev approx
                h_sc, a_sc = scores[:,0], scores[:,1]

                # Derived Metrics
                model_spread = -(np.mean(h_sc) - np.mean(a_sc))
                model_total = np.mean(h_sc) + np.mean(a_sc)
                model_win_prob = np.mean(h_sc > a_sc)

                # Log5 (Theoretical Win %)
                pyth_h = h_off**13.91 / (h_off**13.91 + h_def**13.91)
                pyth_a = a_off**13.91 / (a_off**13.91 + a_def**13.91)
                log5_prob = (pyth_h - pyth_h * pyth_a) / (pyth_h + pyth_a - 2 * pyth_h * pyth_a)

                # Luck Calculation
                actual_win_pct = home_stats.get('W', 0) / home_stats.get('GP', 1)
                luck_metric = actual_win_pct - pyth_h # Pos = Lucky, Neg = Unlucky

                # Feng Rating Spread
                r_home, r_away = feng_ratings.get(str(team_id), 0), feng_ratings.get(str(opponent_id), 0)
                feng_spread = -(r_home - r_away + HCA) if r_home and r_away else None

                model_game_rows.append({
                    'TeamID': int(team_id),
                    'Model Spread': round(model_spread, 1),
                    'Model Total': round(model_total, 1),
                    'Model Win %': model_win_prob,
                    'Log5 %': log5_prob,
                    'Feng Spread': round(feng_spread, 1) if feng_spread else "-",
                    'Luck': luck_metric,
                    'NetRtg': home_stats.get('E_NET_RATING'),
                    'eFG%': home_stats.get('E_EFG_PCT'),
                    'OREB%': home_stats.get('E_OREB_PCT')
                })

    proj_df = pd.DataFrame(projection_rows)
    if proj_df.empty:
        print(f"\n❌ NO PROJECTIONS GENERATED."); return
    else:
        proj_df['match_key'] = proj_df['Player'].apply(normalize_player_name)
        print(f"✅ Success: Generated {len(proj_df)} player projections. Proceeding to fetch odds...")

    # 5. FETCH ODDS
    game_lines_rows = []
    sharp_props_map = {}

    for k in ODDS_API_KEYS:
        try:
            r = requests.get(f"https://api.the-odds-api.com/v4/sports/basketball_nba/odds?apiKey={k}&regions=us&markets=h2h", timeout=10); r.raise_for_status()
            if int(r.headers.get('x-requests-remaining', 0)) > 20: key = k; print(f"Using API key ending in ...{k[-4:]}"); break
        except Exception: continue

    if key:
        print("--- Fetching Sharp & Retail Odds ---")
        try:
            game_markets_to_fetch = ",".join(ACTIVE_GAME_MARKETS)
            books_str = ",".join(ALL_BOOKS_FOR_API)
            url_games = f"https://api.the-odds-api.com/v4/sports/basketball_nba/odds/?apiKey={key}&regions=us,eu&markets={game_markets_to_fetch or 'h2h'}&bookmakers={books_str}"
            response_games = requests.get(url_games, timeout=15); response_games.raise_for_status(); games_data = response_games.json()
            game_events = [game for game in games_data if parser.isoparse(game["commence_time"]).astimezone(eastern).strftime('%Y-%m-%d') == iso_today_str] # USE ISO DATE HERE
            game_events.sort(key=lambda x: x["commence_time"])

            for game in game_events:
                home_team_list = teams.find_teams_by_full_name(game['home_team']); away_team_list = teams.find_teams_by_full_name(game['away_team'])
                if not home_team_list or not away_team_list: continue
                home_team, away_team = home_team_list[0], away_team_list[0]
                home_id, away_id = str(home_team['id']), str(away_team['id'])
                if home_id not in schedule: schedule[home_id] = away_id; schedule[away_id] = home_id

                home_abbr, away_abbr = home_team['abbreviation'], away_team['abbreviation']
                spreads, totals, mls = [], [], []; retail_spread, retail_total, retail_ml, retail_book = np.nan, np.nan, np.nan, "Consensus"
                home_spread_sharp_list, home_spread_retail_val = [], None; over_sharp_list, over_retail_val = [], None

                for book in game.get('bookmakers', []):
                    if book['key'] in SHARP_BOOKS:
                        for market in book.get('markets', []):
                            if market['key'] == 'spreads': spreads.append(market['outcomes'][0]['point'])
                            elif market['key'] == 'totals': totals.append(market['outcomes'][0]['point'])
                            elif market['key'] == 'h2h':
                                for o in market['outcomes']:
                                    if o['name'] == game['home_team']: mls.append(o['price'])
                    if book['key'] in ['draftkings', 'fanduel']:
                        if pd.isna(retail_spread): retail_book = book['title']
                        for market in book.get('markets', []):
                            if market['key'] == 'spreads' and pd.isna(retail_spread): retail_spread = market['outcomes'][0]['point']
                            if market['key'] == 'totals' and pd.isna(retail_total): retail_total = market['outcomes'][0]['point']
                            if market['key'] == 'h2h' and pd.isna(retail_ml):
                                for o in market['outcomes']:
                                    if o['name'] == game['home_team']: retail_ml = o['price']

                    for market in book.get('markets', []):
                        if market['key'] == 'spreads':
                            for o in market['outcomes']:
                                if o['name'] == game['home_team']:
                                    if book['key'] in SHARP_BOOKS: home_spread_sharp_list.append(o['point'])
                                    elif book['key'] in RETAIL_BOOKS and home_spread_retail_val is None: home_spread_retail_val = o['point']
                        if market['key'] == 'totals':
                            for o in market['outcomes']:
                                if o['name'] == 'Over':
                                    if book['key'] in SHARP_BOOKS: over_sharp_list.append(o['point'])
                                    elif book['key'] in RETAIL_BOOKS and over_retail_val is None: over_retail_val = o['point']

                sharp_spread = sum(spreads)/len(spreads) if spreads else None; sharp_total = sum(totals)/len(totals) if totals else None
                try: live_game_lines[int(home_id)] = {'spread': retail_spread, 'total': retail_total}; live_game_lines[int(away_id)] = {'spread': -retail_spread if not pd.isna(retail_spread) else np.nan, 'total': retail_total}
                except: pass

                model_data = next((item for item in model_game_rows if item["TeamID"] == int(home_id)), {})
                spread_diff = abs(retail_spread - sharp_spread) if (not pd.isna(retail_spread) and sharp_spread is not None) else 0
                total_diff = abs(retail_total - sharp_total) if (not pd.isna(retail_total) and sharp_total is not None) else 0
                signal = "SHARP MISMATCH" if (spread_diff >= 1.5 or total_diff >= 1.5) else ("Diff Found" if (spread_diff >= 0.5) else "-")

                pick_spread, pick_total, pick_ml = "-", "-", "-"
                if home_spread_retail_val is not None and home_spread_sharp_list:
                    avg_home_sharp = sum(home_spread_sharp_list) / len(home_spread_sharp_list)
                    if abs(home_spread_retail_val - avg_home_sharp) >= 1.0:
                        pick_spread = f"{home_abbr} {home_spread_retail_val:+.1f}" if avg_home_sharp < home_spread_retail_val else f"{away_abbr} {-1 * home_spread_retail_val:+.1f}"

                if over_retail_val is not None and over_sharp_list:
                    avg_over_sharp = sum(over_sharp_list) / len(over_sharp_list)
                    if abs(over_retail_val - avg_over_sharp) >= 1.0:
                        pick_total = f"Over {over_retail_val}" if avg_over_sharp > over_retail_val else f"Under {over_retail_val}"

                if not pd.isna(retail_ml) and 'Model Win %' in model_data:
                    if retail_ml > 1.0 and model_data['Model Win %'] > ((1.0 / retail_ml) + 0.05): pick_ml = f"{home_abbr} ML"

                game_lines_rows.append({
                    'Game': f"{away_abbr} @ {home_abbr}",
                    'Time': parser.isoparse(game["commence_time"]).astimezone(eastern).strftime('%-I:%M %p'),
                    'Retail Book': retail_book,
                    'Retail Spread': retail_spread, 'Sharp Spread': round(sharp_spread, 1) if sharp_spread else "-",
                    'Model Spread': model_data.get('Model Spread', '-'),
                    'Feng Spread': model_data.get('Feng Spread', '-'), # ADDED
                    'Spread Diff': round(spread_diff, 1), 'Spread Pick': pick_spread,
                    'Retail Total': retail_total, 'Sharp Total': round(sharp_total, 1) if sharp_total else "-",
                    'Model Total': model_data.get('Model Total', '-'), # ADDED
                    'Total Diff': round(total_diff, 1), 'Total Pick': pick_total,
                    'Retail ML': retail_ml,
                    'Model Win %': f"{model_data.get('Model Win %', 0)*100:.1f}%" if 'Model Win %' in model_data else "-",
                    'Log5 Win %': f"{model_data.get('Log5 %', 0)*100:.1f}%" if 'Log5 %' in model_data else "-", # ADDED
                    'ML Pick': pick_ml,
                    'Home Luck': round(model_data.get('Luck', 0), 3) if 'Luck' in model_data else "-", # ADDED
                    'Home Net Rtg': model_data.get('NetRtg', '-'),
                    'Home eFG%': model_data.get('eFG%', '-'),
                    'Signal': signal
                })
        except Exception as e: print(f"Failed to fetch game list/lines: {e}"); game_events = []

        if game_events:
            all_props_raw = []
            market_query = ','.join(ALL_PROP_MARKETS_LIST) if ALL_PROP_MARKETS_LIST else ""
            if market_query:
                for game in tqdm(game_events, desc="Fetching Player Props"):
                    event_id = game['id']
                    away_abbr = team_abbr_map.get(game['away_team'], '?'); home_abbr = team_abbr_map.get(game['home_team'], '?')
                    books_str = ",".join(ALL_BOOKS_FOR_API)
                    url_props = f"https://api.the-odds-api.com/v4/sports/basketball_nba/events/{event_id}/odds?apiKey={key}&regions=us,eu&markets={market_query}&bookmakers={books_str}"
                    try:
                        r = requests.get(url_props, timeout=20); r.raise_for_status(); payload = r.json()
                        for book in payload.get('bookmakers', []):
                            for market in book.get('markets', []):
                                for outcome in market.get('outcomes', []):
                                    player_name = outcome['description']
                                    norm_name = normalize_player_name(player_name)
                                    prop_name = MARKET_NAME_MAP.get(market['key'])
                                    line = outcome.get('point')

                                    if book['key'] in SHARP_BOOKS and line is not None:
                                        dict_key = (norm_name, prop_name)
                                        if dict_key not in sharp_props_map: sharp_props_map[dict_key] = []
                                        sharp_props_map[dict_key].append(line)
                                        continue

                                    if book['key'] in ['draftkings', 'fanduel']:
                                        all_props_raw.append({'Game': f"{away_abbr} @ {home_abbr}", 'Game_Time': parser.isoparse(game["commence_time"]).astimezone(eastern).strftime('%-I:%M %p'), 'Sort_Time': game["commence_time"], 'Player': player_name, 'Prop': prop_name, 'Line': line, 'Decimal_Odds': outcome.get('price'), 'OutcomeName': outcome['name'], 'Bookmaker': book['title']})
                    except Exception: continue
            if all_props_raw:
                props_df = pd.DataFrame(all_props_raw)
                props_df['match_key'] = props_df['Player'].apply(normalize_player_name)
                print(f"Debug Props Keys: {props_df['match_key'].head(3).tolist()}")

    if props_df.empty: print("\n--- Displaying Projections Only (No Betting Odds Found) ---"); display(proj_df[['Player', 'Experience', 'Team', 'Opponent', 'Pred_MIN', 'Pred_PTS', 'Pred_REB', 'Pred_AST', 'Pred_FG3M']].round(2).head(20)); return

    print("\n--- Running Value Analysis & Filtering Injuries ---")
    ou_props = props_df[props_df['OutcomeName'].isin(['Over', 'Under'])].copy(); yn_props = props_df[props_df['OutcomeName'].isin(['Yes', 'No'])].copy()
    merged_ou = pd.DataFrame()
    if not ou_props.empty:
        ou_pivot = ou_props.pivot_table(index=['match_key', 'Player', 'Prop', 'Line', 'Bookmaker', 'Game', 'Game_Time', 'Sort_Time'], columns='OutcomeName', values='Decimal_Odds').reset_index().rename(columns={'Over': 'Over_Odds', 'Under': 'Under_Odds'})
        merged_ou = pd.merge(ou_pivot, proj_df.drop(columns=['Player'], errors='ignore'), on='match_key', how='inner')
        if not merged_ou.empty:
            merged_ou['Projection'] = merged_ou.apply(lambda r: r.get(f"Pred_{r['Prop']}", np.nan), axis=1)
            merged_ou['Proj Range Low'] = (merged_ou['Projection'] - merged_ou['Prop'].map(mae_values)).clip(lower=0); merged_ou['Proj Range High'] = merged_ou['Projection'] + merged_ou['Prop'].map(mae_values)
            merged_ou['Proj Range'] = merged_ou.apply(lambda r: f"{r['Proj Range Low']:.1f}-{r['Proj Range High']:.1f}", axis=1)
            merged_ou[['Prob_Over', 'Prob_Under']] = merged_ou.apply(lambda r: calculate_ou_probability(r['Projection'], r['Line']), axis=1, result_type='expand')
            def determine_bet(row):
                if row['Line'] < row['Proj Range Low'] and row['Prob_Over'] > 0.53: return 'Over'
                elif row['Line'] > row['Proj Range High'] and row['Prob_Under'] > 0.53: return 'Under'
                return 'Pass'
            merged_ou['Bet'] = merged_ou.apply(determine_bet, axis=1)
            merged_ou['Edge (%)'] = np.where(merged_ou['Bet'] == 'Over', (merged_ou['Prob_Over'] * merged_ou['Over_Odds']) - 1, np.where(merged_ou['Bet'] == 'Under', (merged_ou['Prob_Under'] * merged_ou['Under_Odds']) - 1, 0)) * 100

    merged_yn = pd.DataFrame()
    if not yn_props.empty:
        yn_pivot = yn_props.pivot_table(index=['match_key', 'Player', 'Prop', 'Bookmaker', 'Game', 'Game_Time', 'Sort_Time'], columns='OutcomeName', values='Decimal_Odds').reset_index().rename(columns={'Yes': 'Yes_Odds', 'No': 'No_Odds'})
        merged_yn = pd.merge(yn_pivot, proj_df.drop(columns=['Player'], errors='ignore'), on='match_key', how='inner')
        if not merged_yn.empty:
            merged_yn['Projection'] = merged_yn.apply(lambda r: r.get(f"Pred_{r['Prop']}", np.nan), axis=1)
            merged_yn['Model Score (%)'] = merged_yn['Projection']; merged_yn['Implied Prob (%)'] = (1 / merged_yn['Yes_Odds']) * 100
            merged_yn['Edge (%)'] = merged_yn['Model Score (%)'] - merged_yn['Implied Prob (%)']; merged_yn['Bet'] = np.where(merged_yn['Edge (%)'] > 5, 'Yes', 'Pass')

    full_analysis_df = pd.concat([merged_ou, merged_yn])
    if not injury_df.empty and not full_analysis_df.empty: full_analysis_df = pd.merge(full_analysis_df, injury_df[['match_key', 'Status']], on='match_key', how='left'); full_analysis_df['Status'] = full_analysis_df['Status'].fillna('Active')
    elif not full_analysis_df.empty: full_analysis_df['Status'] = 'Active'

    if not full_analysis_df.empty and FILTER_OUT_PLAYERS: full_analysis_df = full_analysis_df[~full_analysis_df['Status'].isin(['Out', 'Injured Reserve', 'Suspended'])]

    prev_run_df = get_previous_run_data(file_today_str, OUTPUT_DIR) # USE FILE DATE HERE

    full_analysis_df = apply_movement_and_ranking(full_analysis_df, prev_run_df, sharp_props_map)

    # ==============================================================================
    # --- NEW FEATURE: STREAKS, DEFENSE RANKS & HEAD-TO-HEAD ---
    # ==============================================================================
    if not full_analysis_df.empty:
        print("\n--- Calculating Player Streaks, Opponent Ranks & H2H ---")

        streak_data = []
        rank_data = []
        h2h_data = [] # New list

        for idx, row in tqdm(full_analysis_df.iterrows(), total=len(full_analysis_df), desc="Analyzing Trends"):
            # 1. Defense Rank
            # *** FIX: Use ID instead of Abbreviation lookup which was failing ***
            opp_id = row.get('OpponentID') # Passed from Proj
            opp_abbr = row.get('Opponent') # For H2H lookup
            prop_cat = row.get('Prop')
            rank_val = 99
            stat_key_map = {'Points': 'PTS', 'Rebounds': 'REB', 'Assists': 'AST', 'Threes': 'FG3M'}

            if opp_id and opp_id in def_ranks and prop_cat in stat_key_map:
                rank_val = def_ranks[opp_id][stat_key_map[prop_cat]]
            rank_data.append(rank_val)

            # Setup Map for Logs
            pid = row.get('PLAYER_ID')
            line = row.get('Line')
            log_map = {'Points': 'PTS', 'Rebounds': 'REB', 'Assists': 'AST', 'Threes': 'FG3M', 'Combos': 'PRA'}
            mapped_prop = log_map.get(prop_cat, prop_cat)
            if prop_cat == 'DoubleDouble': mapped_prop = 'DD'
            if 'Pts+Reb+Ast' in str(row.get('OutcomeName', '')): mapped_prop = 'PRA'
            elif 'Pts+Reb' in str(row.get('OutcomeName', '')): mapped_prop = 'P+R'
            elif 'Pts+Ast' in str(row.get('OutcomeName', '')): mapped_prop = 'P+A'
            elif 'Reb+Ast' in str(row.get('OutcomeName', '')): mapped_prop = 'R+A'

            # 2. Streaks (10 Game Window)
            s_curr, s_l10 = calculate_prop_streak(pid, mapped_prop, line)
            # *** FIX: Reverted to W{n}. Shows W0 if 0. ***
            streak_str = f"W{s_curr} ({s_l10})"
            streak_data.append(streak_str)

            # 3. Head-to-Head (vs Opponent this season)
            # We use start_year from the configuration variables
            h2h_val = calculate_head_to_head(pid, mapped_prop, line, opp_abbr, start_year)
            h2h_data.append(h2h_val)

        full_analysis_df['Opp Rank'] = rank_data
        full_analysis_df['Streak'] = streak_data
        full_analysis_df['vs Opp'] = h2h_data # Add new column

        def fmt_rank(r):
            if r == 99: return "-"
            if r >= 25: return f"{r} (Easy)"
            if r <= 5: return f"{r} (Hard)"
            return str(r)

        full_analysis_df['Opp Rank'] = full_analysis_df['Opp Rank'].apply(fmt_rank)

    # ==============================================================================

    if not full_analysis_df.empty:
        full_analysis_df['Projection'] = full_analysis_df['Projection'].round(2)
        if 'Edge (%)' in full_analysis_df.columns: full_analysis_df['Edge (%)'] = full_analysis_df['Edge (%)'].round(1)
        if 'Proj vs Sharp' in full_analysis_df.columns: full_analysis_df['Proj vs Sharp'] = full_analysis_df['Proj vs Sharp'].astype(float).round(1)

        final_bets = full_analysis_df[full_analysis_df['Bet'] != 'Pass'].copy()
    else: final_bets = pd.DataFrame()

    out_file = os.path.join(OUTPUT_DIR, f"{file_today_str} NBA Player Props.xlsx") # USE FILE DATE HERE

    with pd.ExcelWriter(out_file, engine='xlsxwriter') as writer:
        workbook  = writer.book
        header_fmt = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'bg_color': '#D9D9D9', 'border': 1})
        center_fmt = workbook.add_format({'align': 'center', 'valign': 'vcenter', 'border': 1})
        green_fmt = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        red_fmt = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        orange_fmt = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C5700', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        blue_fmt = workbook.add_format({'bg_color': '#BDD7EE', 'font_color': '#1F4E78', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        purple_fmt = workbook.add_format({'bg_color': '#CCC0DA', 'font_color': '#3F3F76', 'align': 'center', 'valign': 'vcenter', 'border': 1})
        perfect_fmt = workbook.add_format({'bg_color': '#00FF00', 'font_color': '#000000', 'align': 'center', 'valign': 'vcenter', 'border': 1, 'bold': True})
        dd_td_fmt = workbook.add_format({'bg_color': '#9FC5E8', 'font_color': '#000000', 'align': 'center', 'valign': 'vcenter', 'border': 1, 'bold': True})

        green_row_fmt = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        blue_row_fmt = workbook.add_format({'bg_color': '#BDD7EE', 'font_color': '#1F4E78', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        orange_row_fmt = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C5700', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        red_row_fmt = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'align': 'left', 'valign': 'vcenter', 'border': 1})
        purple_row_fmt = workbook.add_format({'bg_color': '#CCC0DA', 'font_color': '#3F3F76', 'align': 'left', 'valign': 'vcenter', 'border': 1})

        ws_howto = workbook.add_worksheet("HOW TO USE")
        ws_howto.set_column(0, 0, 30); ws_howto.set_column(1, 1, 120); ws_howto.freeze_panes(1, 0)
        ws_howto.write(0, 0, "Item", header_fmt); ws_howto.write(0, 1, "Description", header_fmt)

        ws_howto.write(1, 0, "Ranking: SHARP MISMATCH", purple_row_fmt); ws_howto.write(1, 1, "Massive Value + Price Improved. The line moved in your favor AND the Edge is huge.", purple_row_fmt)
        ws_howto.write(2, 0, "Ranking: Most Profitable", green_row_fmt); ws_howto.write(2, 1, "Price Improved. Line moved in your favor since yesterday.", green_row_fmt)
        ws_howto.write(3, 0, "Ranking: Most Validated", orange_row_fmt); ws_howto.write(3, 1, "Sharps Agree. Line moved against you, but model still sees value.", orange_row_fmt)
        ws_howto.write(4, 0, "Ranking: Safest Bet", blue_row_fmt); ws_howto.write(4, 1, "High Probability (>60%) of winning regardless of odds.", blue_row_fmt)
        ws_howto.write(5, 0, "Ranking: Avoid / Lottery", red_row_fmt); ws_howto.write(5, 1, "Suspiciously high edge (>35%). Likely an injury risk.", red_row_fmt)
        ws_howto.write(6, 0, "Ranking: PRIME VALUE", blue_row_fmt); ws_howto.write(6, 1, "High Edge + Price Improved, but no Sharp Line confirmation available yet.", blue_row_fmt)
        ws_howto.write(7, 0, "Ranking: BREAKOUT OPP", red_row_fmt); ws_howto.write(7, 1, "Model projects Big Minutes for a low-line player. High Upside but volatile.", red_row_fmt)

        ws_howto.write(8, 0, "BETTING SCHEDULE", header_fmt)
        ws_howto.write(9, 0, "4 Hours Out (Pro Window)", blue_row_fmt); ws_howto.write(9, 1, "Sportsbooks raise limits. Professional money enters. Movement here signals Syndicate preference before public money floods in.", blue_row_fmt)
        ws_howto.write(10, 0, "90 Minutes Out (Info Window)", orange_row_fmt); ws_howto.write(10, 1, "Official injury reports released. Movement here is the market reaction to hard facts. Whale money enters.", orange_row_fmt)

        ws_howto.write(12, 0, "MODEL ACCURACY RANKING", header_fmt)
        ws_howto.write(13, 0, "1. Threes (FG3M)", green_row_fmt); ws_howto.write(13, 1, "Most Accurate (MAE ~0.13). Trust small edges here. Green Highlight at > 5%.", green_row_fmt)
        ws_howto.write(14, 0, "2. Assists (AST)", green_row_fmt); ws_howto.write(14, 1, "Very Accurate (MAE ~0.28). Great for Point Guard props. Green Highlight at > 8%.", green_row_fmt)
        ws_howto.write(15, 0, "3. Rebounds (REB)", blue_row_fmt); ws_howto.write(15, 1, "Strong (MAE ~0.50). Focus on Centers/consistent rebounders. Green Highlight at > 8%.", blue_row_fmt)
        ws_howto.write(16, 0, "4. Points (PTS)", orange_row_fmt); ws_howto.write(16, 1, "Highest Variance (MAE ~1.15). Only bet if Edge > 10% or it's a Breakout scenario. Green Highlight at > 12%.", orange_row_fmt)

        ws_howto.write(18, 0, "CONSISTENCY GUIDE", header_fmt)
        ws_howto.write(19, 0, "Consistency: 🔒 High", green_row_fmt); ws_howto.write(19, 1, "Reliable Player. Low Variance (CV < 0.25). Safe to trust projection.", green_row_fmt)
        ws_howto.write(20, 0, "Consistency: ⚡ Volatile", red_row_fmt); ws_howto.write(20, 1, "Boom or Bust. High Variance (CV > 0.45). Bet with caution even if edge is high.", red_row_fmt)

        sheet_config = ['DraftKings_EV_Bets', 'DraftKings_EV_SGP', 'FanDuel_EV_Bets', 'FanDuel_EV_SGP']
        for sheet_name in sheet_config:
            bookmaker = 'DraftKings' if 'DraftKings' in sheet_name else 'FanDuel'
            is_sgp = 'SGP' in sheet_name
            book_df = final_bets[final_bets['Bookmaker'] == bookmaker].copy() if not final_bets.empty else pd.DataFrame()
            if not book_df.empty:
                # *** CUSTOM COLUMN ORDER ***
                out_cols = [
                    'Game_Time', 'Game', 'Status', 'Experience', 'Team', 'Pos', 'Player', 'Prop',
                    'Line', 'Bet', 'Sharp Line', 'Proj vs Sharp',
                    'Projection', 'Proj Range', 'Edge (%)',
                    'Consistency', 'Opp Rank', 'Streak', 'vs Opp', 'Sharp Signal', 'Ranking'
                ]

                if 'Sort_Time' in book_df.columns:
                    book_df['sort_rank'] = book_df['Ranking'].map({
                        "SHARP MISMATCH": 0, "💎 PRIME VALUE": 1, "Most Profitable": 2, "Most Validated": 3, "Safest Bet": 4,
                        "🚀 BREAKOUT OPP": 5, "Strong Value": 6, "Solid Play": 7, "Avoid / Lottery": 99
                    }).fillna(99)

                    if is_sgp: final_df = book_df.sort_values(['Sort_Time', 'Game', 'sort_rank', 'Edge (%)'], ascending=[True, True, True, False]).reindex(columns=out_cols)
                    else: final_df = book_df.sort_values(['sort_rank', 'Edge (%)'], ascending=[True, False]).reindex(columns=out_cols)
                else: final_df = book_df.reindex(columns=out_cols)

                final_df.to_excel(writer, sheet_name=sheet_name, index=False, startrow=1, header=False)
                ws = writer.sheets[sheet_name]; ws.freeze_panes(1, 0)
                ws.autofilter(0, 0, len(final_df), len(out_cols) - 1)
                for col, val in enumerate(final_df.columns):
                    ws.write(0, col, val, header_fmt)
                    max_len = len(str(val))
                    for cell_val in final_df[val].astype(str): max_len = max(max_len, len(cell_val))
                    ws.set_column(col, col, max_len + 4, center_fmt)

                last_row = len(final_df) + 1
                def get_col(name): return xl_col_to_name(out_cols.index(name))

                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'SHARP', 'format': purple_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Profitable', 'format': green_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'PRIME', 'format': blue_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Validated', 'format': orange_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Safest', 'format': blue_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Avoid', 'format': red_fmt})
                ws.conditional_format(f'{get_col("Ranking")}2:{get_col("Ranking")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'BREAKOUT', 'format': red_fmt})

                ws.conditional_format(f'{get_col("Consistency")}2:{get_col("Consistency")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'High', 'format': green_fmt})
                ws.conditional_format(f'{get_col("Consistency")}2:{get_col("Consistency")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Volatile', 'format': red_fmt})
                ws.conditional_format(f'{get_col("Status")}2:{get_col("Status")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Questionable', 'format': orange_fmt})

                # Highlight Easy/Hard Matchups
                ws.conditional_format(f'{get_col("Opp Rank")}2:{get_col("Opp Rank")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Easy', 'format': green_fmt})
                ws.conditional_format(f'{get_col("Opp Rank")}2:{get_col("Opp Rank")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Hard', 'format': red_fmt})

                # Highlight Perfect H2H (1/1, 2/2, etc.)
                for h2h_val in ['1/1', '2/2', '3/3', '4/4']:
                    ws.conditional_format(f'{get_col("vs Opp")}2:{get_col("vs Opp")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': h2h_val, 'format': green_fmt})

                # Highlight Strong Recent Form (8/10, 9/10, 10/10)
                for trend_val in ['8/10', '9/10', '10/10']:
                    ws.conditional_format(f'{get_col("Streak")}2:{get_col("Streak")}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': trend_val, 'format': green_fmt})

                f_threes = f'=AND(${get_col("Prop")}2="Threes", ${get_col("Edge (%)")}2>5)'
                ws.conditional_format(f'{get_col("Edge (%)")}2:{get_col("Edge (%)")}{last_row}', {'type': 'formula', 'criteria': f_threes, 'format': green_fmt})
                f_pts = f'=AND(${get_col("Prop")}2="Points", ${get_col("Edge (%)")}2>12)'
                ws.conditional_format(f'{get_col("Edge (%)")}2:{get_col("Edge (%)")}{last_row}', {'type': 'formula', 'criteria': f_pts, 'format': green_fmt})
                f_ar = f'=AND(OR(${get_col("Prop")}2="Assists", ${get_col("Prop")}2="Rebounds"), ${get_col("Edge (%)")}2>8)'
                ws.conditional_format(f'{get_col("Edge (%)")}2:{get_col("Edge (%)")}{last_row}', {'type': 'formula', 'criteria': f_ar, 'format': green_fmt})

                # *** PERFECT BET FORMULA UPDATE ***
                # Added: ISERROR(SEARCH("Hard", ${get_col("Opp Rank")}2))
                # This ensures a "Perfect Bet" (Green Row) is NOT a Hard Matchup.
                base_criteria = f'AND(OR(ISNUMBER(SEARCH("SHARP", ${get_col("Ranking")}2)), ISNUMBER(SEARCH("Validated", ${get_col("Ranking")}2))), ${get_col("Edge (%)")}2>=8, ${get_col("Edge (%)")}2<=30, OR(ISNUMBER(SEARCH("AGREE", ${get_col("Sharp Signal")}2)), ISNUMBER(SEARCH("IMPROVED", ${get_col("Sharp Signal")}2))), ISERROR(SEARCH("Hard", ${get_col("Opp Rank")}2)))'
                pos_criteria = f'IF(${get_col("Prop")}2="Assists", ISNUMBER(SEARCH("G", ${get_col("Pos")}2)), IF(${get_col("Prop")}2="Rebounds", OR(ISNUMBER(SEARCH("F", ${get_col("Pos")}2)), ISNUMBER(SEARCH("C", ${get_col("Pos")}2))), TRUE))'
                final_formula = f'=AND({base_criteria}, {pos_criteria})'
                ws.conditional_format(f'{get_col("Player")}2:{get_col("Player")}{last_row}', {'type': 'formula', 'criteria': final_formula, 'format': perfect_fmt})

            else: writer.book.add_worksheet(sheet_name).write(0, 0, "No bets found.")

        if game_lines_rows and ACTIVE_GAME_MARKETS:
            gl_df = pd.DataFrame(game_lines_rows)
            gl_cols = ['Time', 'Game', 'Retail Book', 'Retail Spread', 'Sharp Spread', 'Model Spread', 'Feng Spread', 'Spread Diff', 'Spread Pick', 'Retail Total', 'Sharp Total', 'Model Total', 'Total Diff', 'Total Pick', 'Retail ML', 'Model Win %', 'Log5 Win %', 'ML Pick', 'Home Luck', 'Home Net Rtg', 'Home eFG%', 'Signal']
            gl_df = gl_df.reindex(columns=gl_cols).sort_values(['Signal', 'Spread Diff'], ascending=[False, False])
            gl_df.to_excel(writer, sheet_name='Game_Lines', index=False); ws_gl = writer.sheets['Game_Lines']
            ws_gl.freeze_panes(1, 0)
            ws_gl.autofilter(0, 0, len(gl_df), len(gl_cols) - 1)
            for col, val in enumerate(gl_df.columns):
                ws_gl.write(0, col, val, header_fmt)
                ws_gl.set_column(col, col, 15 + 4, center_fmt)
            last_row = len(gl_df) + 1; sig_col = xl_col_to_name(gl_cols.index('Signal'))
            ws_gl.conditional_format(f'{sig_col}2:{sig_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'SHARP', 'format': purple_fmt})
            ws_gl.conditional_format(f'{sig_col}2:{sig_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'MODEL', 'format': blue_fmt})
            ws_gl.conditional_format(f'{sig_col}2:{sig_col}{last_row}', {'type': 'text', 'criteria': 'containing', 'value': 'Diff', 'format': orange_fmt})

    print(f"\n✅ Ultimate Analysis Exported: {out_file}")

    if not final_bets.empty:
        print("\n" + "="*60); print("🏆 BEST BET PER GAME"); print("="*60)
        best_card = final_bets.copy()
        best_card['Win_Prob'] = np.where(best_card['Bet'] == 'Over', best_card['Prob_Over'], best_card['Prob_Under'])
        best_card['Priority'] = best_card['Sharp Signal'].apply(lambda x: 1 if x in ['PRICE IMPROVED', 'SHARPS AGREE'] else 0)
        best_card = best_card.sort_values(['Game', 'Priority', 'Edge (%)'], ascending=[True, False, False])
        quick_card = best_card.groupby('Game').head(1).sort_values('Game_Time')
        quick_card['Win_Prob'] = (quick_card['Win_Prob'] * 100).round(1).astype(str) + "%"
        quick_card['Edge (%)'] = quick_card['Edge (%)'].round(1).astype(str) + "%"
        display(quick_card[['Game_Time', 'Game', 'Player', 'Status', 'Prop', 'Bet', 'Consistency', 'Line', 'Edge (%)', 'Win_Prob', 'Sharp Signal']].rename(columns={'Sharp Signal': 'Signal'}))

    if final_bets.empty: print("\nNOTE: No bets met the strict criteria.")
    print("\n✅ Script Completed. Exiting...")

if __name__ == "__main__":
    pd.set_option('display.max_rows', 100); pd.set_option('display.max_columns', None)
    run_daily_projections()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.1 MB/s eta 0:00:00

📅 Running Live Mode. Current ET Time: 2026-01-27 15:19:26
Running projections for 2026-01-27 (Season: 2025-26)
✅ All models, imputers, and MAE values loaded successfully.

Fetching live injury report from ESPN API (NBA)...


Fetching Team Rosters for Injuries: 100%|██████████| 30/30 [00:03<00:00,  8.28it/s]


   ✅ Successfully fetched 116 injury entries.
Loading Player Positions from: /content/drive/MyDrive/NBA/Player Positions/2025-26 NBA Player Positions.xlsx
✅ Successfully loaded 509 player positions.
   ℹ️ Detected 85 ruled-out players. Adjusting team usage automatically...

--- Preparing Context & Stats ---
✅ Successfully fetched season stats.
   🛡️ Fetching Opponent Defense Rankings (Calculating Manually)...
   🧮 Calculating Feng/Massey Ratings (Strength of Schedule)...
   📊 Fetching Team Metrics for 2025-26...
      ⚠️ Attempting Fallback to Standard Advanced Stats...
      ✅ Successfully fetched Advanced Stats (Fallback).


NameError: name 'get_live_schedule_from_nba_data' is not defined

# **NBA Schedule & Trap Timer** with multiple dates and .ics output


Checking at these two specific times allows you to see the transition from **Model-based betting** to I**nformation-based betting**:

**4 Hours Out (The Pro Window):** Sportsbooks raise their betting limits, allowing professionals to finally place their "max" bets. Movement here tells you which side the **professional models and syndicates** prefer before the general public floods the market.

**90 Minutes Out (The Information Window):** Official injury reports and "inactive" lists are released. Movement here represents the **market's reaction to hard facts**. This is the "heavy" money from whales who wait for 100% certainty before risking large sums.

**The Bottom Line:** Comparing these two windows reveals if the "big money" is doubling down on their early position or if late-breaking news has completely flipped the script. **Always follow the 90-minute move**, as it represents the most informed and highest-limit money in the market.

In [ ]:
# ==============================================================================
# SCRIPT: NBA Optimal Run-Time & ICS Generator (Interactive)
# Features: Date Ranges + ICS Download + Inactives Strategy + Calendar Icons
# ==============================================================================

# 1. INSTALL LIBRARIES
!pip install -q xlsxwriter pandas numpy requests pytz

import requests
import pandas as pd
from datetime import datetime, timedelta
import pytz
import uuid
import time
import os
from google.colab import drive, files

# --- Configuration ---
try:
    drive.mount('/content/drive', force_remount=True)
    OUTPUT_DIR = "/content/drive/MyDrive/NBA/Schedules/" # Adjusted for NBA folder
    os.makedirs(OUTPUT_DIR, exist_ok=True)
except Exception as e:
    print(f"⚠️ Google Drive not available. Files will be downloaded locally. Error: {e}")
    OUTPUT_DIR = "./" # Fallback to local directory

def format_time_no_zero(dt_val):
    """Formats time as 5:00 PM (removes leading zero)."""
    if pd.isna(dt_val): return "N/A"
    return dt_val.strftime('%-I:%M %p')

def get_yes_no(prompt):
    """Gets a simple yes/no response from the user."""
    while True:
        choice = input(f"{prompt} (y/n): ").strip().lower()
        if choice in ['y', 'yes']: return True
        if choice in ['n', 'no']: return False
        print("Please enter 'y' or 'n'.")

def get_date_input(prompt, default_today=False):
    """Gets a date from the user with validation."""
    while True:
        user_input = input(f"{prompt} (MM-DD-YY) [Enter for Today]: ").strip()
        if not user_input and default_today:
            return datetime.now()
        try:
            return datetime.strptime(user_input, "%m-%d-%y")
        except:
            print("❌ Invalid format. Use MM-DD-YY.")

def create_ics_event(start_time, summary, description):
    """Helper to format a single event for an ICS calendar file."""
    dt_utc = start_time.astimezone(pytz.utc)
    dt_end_utc = dt_utc + timedelta(minutes=15)
    fmt = "%Y%m%dT%H%M%SZ"

    return f"""BEGIN:VEVENT
UID:{uuid.uuid4()}
DTSTAMP:{datetime.now(pytz.utc).strftime(fmt)}
DTSTART:{dt_utc.strftime(fmt)}
DTEND:{dt_end_utc.strftime(fmt)}
SUMMARY:{summary}
DESCRIPTION:{description}
BEGIN:VALARM
TRIGGER:-PT5M
ACTION:DISPLAY
DESCRIPTION:Reminder
END:VALARM
END:VEVENT"""

def fetch_nba_day_schedule(target_dt):
    """Fetches First and Last Tip-off for a specific date from ESPN NBA."""
    espn_date_str = target_dt.strftime("%Y%m%d")
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard"
    params = {"limit": "100", "dates": espn_date_str}

    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
        events = data.get('events', [])
    except requests.exceptions.RequestException as e:
        print(f"   (API request failed: {e})")
        return None

    if not events: return None

    game_times = []
    utc = pytz.utc
    est = pytz.timezone('US/Eastern')

    for event in events:
        try:
            date_str = event['date']
            dt_utc = datetime.strptime(date_str, "%Y-%m-%dT%H:%MZ").replace(tzinfo=utc)
            dt_est = dt_utc.astimezone(est)

            # Get Matchup Name
            competitions = event['competitions'][0]
            team1_data = next(c for c in competitions['competitors'] if c['homeAway'] == 'home')
            team2_data = next(c for c in competitions['competitors'] if c['homeAway'] == 'away')
            team1 = team1_data['team']['abbreviation']
            team2 = team2_data['team']['abbreviation']
            matchup = f"{team2} @ {team1}"

            status = event['status']['type']['state']
            if status not in ['post', 'canceled']:
                game_times.append({'time': dt_est, 'matchup': matchup})
        except (KeyError, ValueError, StopIteration):
            continue

    if not game_times: return None

    # Sort by time to find the first game
    df = pd.DataFrame(game_times).sort_values('time')

    # *** BATCHING LOGIC: Group games into distinct time windows ***
    batches = []
    if not df.empty:
        current_batch_start = df.iloc[0]['time']
        current_batch_games = []

        for _, row in df.iterrows():
            # If the gap between this game and the batch start is > 2.5 hours, start a new batch
            if (row['time'] - current_batch_start).total_seconds() > 9000:
                batches.append({'start': current_batch_start, 'games': current_batch_games})
                current_batch_start = row['time']
                current_batch_games = [row['matchup']]
            else:
                current_batch_games.append(row['matchup'])

        # Append the last batch
        batches.append({'start': current_batch_start, 'games': current_batch_games})

    return batches

def generate_ics_content(all_events):
    """Generates the text content for the .ics file."""
    lines = [
        "BEGIN:VCALENDAR",
        "VERSION:2.0",
        "PRODID:-//NBA Trap Strategy Script//EN",
        "CALSCALE:GREGORIAN",
        "METHOD:PUBLISH"
    ]
    lines.extend(all_events)
    lines.append("END:VCALENDAR")
    return "\n".join(lines)

def run_scheduler():
    print("\n🏀 NBA TRAP STRATEGY SCHEDULER 🏀\n")

    # 1. User Inputs
    start_date = get_date_input("Enter Start Date", default_today=True)

    use_range = get_yes_no("Do you want to generate a schedule for a date range?")
    if use_range:
        end_date = get_date_input("Enter End Date", default_today=False)
    else:
        end_date = start_date

    generate_ics = get_yes_no("Generate Calendar (.ics) file?")

    # 2. Iterate Dates
    current_date = start_date
    calendar_events = []

    print("\n" + "="*60)
    print(f" 📅  GENERATING SCHEDULE ({start_date.strftime('%m/%d')} - {end_date.strftime('%m/%d')})")
    print("="*60)

    delta = end_date - start_date
    for i in range(delta.days + 1):
        current_dt = start_date + timedelta(days=i)
        pretty_date = current_dt.strftime("%Y-%m-%d")

        print(f"\nChecking {pretty_date}...", end=" ")

        batches = fetch_nba_day_schedule(current_dt)

        if not batches:
            print("❌ No active games found.")
            continue

        print(f"✅ Found {len(batches)} distinct game windows.")

        for idx, batch in enumerate(batches):
            batch_start = batch['start']
            first_game = batch['games'][0]
            game_count = len(batch['games'])

            # Calculate Times
            run1_time = batch_start - timedelta(hours=4)
            run2_time = batch_start - timedelta(minutes=90)

            batch_label = f"Batch {idx+1}"

            print(f"   🔹 {batch_label} ({first_game} + {game_count-1} others starting at {format_time_no_zero(batch_start)})")
            print(f"      Run 1 (Pro Window):  {format_time_no_zero(run1_time)}")
            print(f"      Run 2 (Info Window): {format_time_no_zero(run2_time)}")

            # Prepare ICS Data
            desc_run1 = f"NBA Trap Strategy RUN 1 (Baseline).\nBatch {idx+1} First game: {first_game}.\nAction: Run Script 3 and save Excel."
            desc_run2 = f"NBA Trap Strategy RUN 2 (Execution).\nBatch {idx+1} First game: {first_game}.\nAction: OFFICIAL INACTIVES RELEASED. Run Script 3. BET NOW."

            calendar_events.append(create_ics_event(run1_time, f"🏀 NBA Run 1 ({pretty_date} B{idx+1})", desc_run1))
            calendar_events.append(create_ics_event(run2_time, f"🏀 NBA Run 2 ({pretty_date} B{idx+1})", desc_run2))

        time.sleep(0.5) # Be polite to the API

    # 3. Generate & Download ICS if requested
    if generate_ics and calendar_events:
        content = generate_ics_content(calendar_events)
        filename = f"NBA_Schedule_{start_date.strftime('%m%d')}_{end_date.strftime('%m%d')}.ics"
        full_path = os.path.join(OUTPUT_DIR, filename)

        # Save to Drive/Local
        with open(full_path, "w", encoding='utf-8') as f:
            f.write(content)

        print("\n" + "-"*60)
        print(f"✅ Calendar File Saved to: {full_path}")
        print("📥 Downloading file to local machine...")

        # Trigger Download
        try:
            files.download(full_path)
        except Exception as e:
             print(f"   (Auto-download failed, check your output folder: {e})")

if __name__ == "__main__":
    run_scheduler()

Mounted at /content/drive

🏈 NFL TRAP STRATEGY SCHEDULER 🏈

Enter Start Date (MM-DD-YY) [Enter for Today]: 1-12-26
Do you want to generate a schedule for a date range? (y/n): y
Enter End Date (MM-DD-YY) [Enter for Today]: 1-31-26
Generate Calendar (.ics) file? (y/n): y

 📅  GENERATING SCHEDULE (01/12 - 01/31)

Checking 2026-01-12... ✅ Found 1 distinct game windows.
   🔹 Batch 1 (HOU @ PIT + 0 others)
      Run 1 (Pro Window):  4:15 PM
      Run 2 (Info Window): 6:45 PM

Checking 2026-01-13... ❌ No active games found.

Checking 2026-01-14... ❌ No active games found.

Checking 2026-01-15... ❌ No active games found.

Checking 2026-01-16... ❌ No active games found.

Checking 2026-01-17... ✅ Found 1 distinct game windows.
   🔹 Batch 1 (BUF @ DEN + 1 others)
      Run 1 (Pro Window):  8:00 PM
      Run 2 (Info Window): 10:30 PM

Checking 2026-01-18... ✅ Found 1 distinct game windows.
   🔹 Batch 1 (LAR @ CHI + 1 others)
      Run 1 (Pro Window):  8:00 PM
      Run 2 (Info Window): 10:30 PM

C

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **SCRIPT 2: Model Training Engine (v7 - FINAL w/ 3-Pointers Model + Optuna)**

In [ ]:
# ==============================================================================
# SCRIPT 2: Model Training Engine (v7 - FINAL w/ 3-Pointers Model + Optuna)
# ==============================================================================
!pip install -q lightgbm scikit-learn pandas joblib optuna

import pandas as pd
import numpy as np
import joblib
import os
import json
import lightgbm as lgb
import optuna
from google.colab import drive
from sklearn.metrics import mean_squared_error, mean_absolute_error, roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold

# --- Optuna Objective Function ---
def objective_lgbm(trial, X, y):
    params = {
        'objective': 'rmse',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0)
    }

    cv = KFold(n_splits=3, shuffle=True, random_state=42)
    rmse_scores = []

    # Fast Cross-Validation for Tuning
    for train_idx, val_idx in cv.split(X, y):
        X_train_fold, X_val_fold = X[train_idx], X[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        model = lgb.LGBMRegressor(**params)
        model.fit(X_train_fold, y_train_fold)
        preds = model.predict(X_val_fold)
        rmse_scores.append(np.sqrt(mean_squared_error(y_val_fold, preds)))

    return np.mean(rmse_scores)

def run_optuna_tuning(X, y, target_name, n_trials=20):
    print(f"   >>> Tuning Hyperparameters for {target_name} ({n_trials} trials)...")
    study = optuna.create_study(direction='minimize')
    study.optimize(lambda trial: objective_lgbm(trial, X, y), n_trials=n_trials)
    print(f"   ✅ Best params for {target_name}: {study.best_params}")
    return study.best_params

def train_and_validate_models_v7():
    drive.mount('/content/drive', force_remount=True); BASE_DIR = "/content/drive/MyDrive/NBA/"; DATA_DIR = os.path.join(BASE_DIR, "Historical Data/"); MODEL_DIR = os.path.join(BASE_DIR, "Training Model Advanced"); os.makedirs(MODEL_DIR, exist_ok=True)
    df = pd.read_csv(os.path.join(DATA_DIR, "nba_advanced_training_data_v6.csv")); df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE']); print(f"Loaded {len(df)} rows from v6 file.")

    print("\n--- Performing Chronological Train/Test Split ---")
    df = df.sort_values('GAME_DATE'); split_date = df['GAME_DATE'].max() - pd.DateOffset(months=2)
    train_df = df[df['GAME_DATE'] < split_date]; test_df = df[df['GAME_DATE'] >= split_date]
    print(f"Training data: {len(train_df)} rows | Test data: {len(test_df)} rows")
    mae_values = {}

    print("\n--- STAGE 1: Training & Validating Minutes Model (w/ Optuna) ---")
    min_features = [c for c in df.columns if 'W_ROLL_AVG_MIN' in c or 'TREND_MIN' in c] + ['DAYS_OF_REST']
    X_train_min, y_train_min = train_df[min_features], train_df['TARGET_MIN']; X_test_min, y_test_min = test_df[min_features], test_df['TARGET_MIN']

    imputer_min = SimpleImputer(strategy='mean'); X_train_min_imputed = imputer_min.fit_transform(X_train_min)

    # *** OPTUNA TUNING FOR MINUTES ***
    best_params_min = run_optuna_tuning(X_train_min_imputed, y_train_min, "Minutes")
    minutes_model = lgb.LGBMRegressor(**best_params_min, random_state=42)
    minutes_model.fit(X_train_min_imputed, y_train_min)

    X_test_min_imputed = imputer_min.transform(X_test_min); preds_min = minutes_model.predict(X_test_min_imputed)
    mae_values['MIN'] = mean_absolute_error(y_test_min, preds_min); print(f"✅ Minutes Model Validation RMSE: {np.sqrt(mean_squared_error(y_test_min, preds_min)):.3f} | MAE: {mae_values['MIN']:.3f}")

    print("\n--- STAGE 2: Training & Validating Core Stats Models (w/ Optuna) ---")
    train_df['Proj_MIN'] = minutes_model.predict(X_train_min_imputed); test_df['Proj_MIN'] = minutes_model.predict(X_test_min_imputed)
    stat_features = [c for c in df.columns if ('W_ROLL_AVG' in c or 'TREND' in c or 'OPP_ALLOWED' in c or '_PM' in c or 'VEGAS' in c) and 'MIN' not in c] + ['DAYS_OF_REST', 'Proj_MIN']
    X_train_stats = train_df[stat_features]; X_test_stats = test_df[stat_features]
    imputer_stats = SimpleImputer(strategy='mean'); X_train_stats_imputed = imputer_stats.fit_transform(X_train_stats); X_test_stats_imputed = imputer_stats.transform(X_test_stats)

    test_preds = {}
    best_params_stats = {}

    # *** OPTUNA TUNING FOR STATS ***
    for prop in ['PTS', 'REB', 'AST', 'FG3M']:
        best_params_stats[prop] = run_optuna_tuning(X_train_stats_imputed, train_df[f'TARGET_{prop}'], prop)
        model = lgb.LGBMRegressor(**best_params_stats[prop], random_state=42)
        model.fit(X_train_stats_imputed, train_df[f'TARGET_{prop}'])

        preds_stat = model.predict(X_test_stats_imputed); test_preds[prop] = preds_stat
        mae_values[prop] = mean_absolute_error(test_df[f'TARGET_{prop}'], preds_stat)
        print(f"  - {prop} Model Validation RMSE: {np.sqrt(mean_squared_error(test_df[f'TARGET_{prop}'], preds_stat)):.3f} | MAE: {mae_values[prop]:.3f}")

    mae_values['PRA'] = mean_absolute_error(test_df['TARGET_PTS'] + test_df['TARGET_REB'] + test_df['TARGET_AST'], test_preds['PTS'] + test_preds['REB'] + test_preds['AST'])
    mae_values['P+R'] = mean_absolute_error(test_df['TARGET_PTS'] + test_df['TARGET_REB'], test_preds['PTS'] + test_preds['REB'])
    mae_values['P+A'] = mean_absolute_error(test_df['TARGET_PTS'] + test_df['TARGET_AST'], test_preds['PTS'] + test_preds['AST'])
    mae_values['R+A'] = mean_absolute_error(test_df['TARGET_REB'] + test_df['TARGET_AST'], test_preds['REB'] + test_preds['AST'])
    with open(os.path.join(MODEL_DIR, "mae_values.json"), 'w') as f: json.dump(mae_values, f, indent=4);
    print(f"✅ MAE values saved to mae_values.json")

    print("\n--- Finalizing Production Models (Re-training on all data) ---")
    imputer_min_final = SimpleImputer(strategy='mean'); X_min_full_imputed = imputer_min_final.fit_transform(df[min_features])

    # Use Tuned Params for Final Models
    minutes_model_final = lgb.LGBMRegressor(**best_params_min, random_state=42)
    minutes_model_final.fit(X_min_full_imputed, df['TARGET_MIN'])

    joblib.dump(minutes_model_final, os.path.join(MODEL_DIR, "minutes_lgbm_model.pkl")); joblib.dump(imputer_min_final, os.path.join(MODEL_DIR, "minutes_imputer.pkl")); joblib.dump(min_features, os.path.join(MODEL_DIR, "minutes_feature_names.pkl"))

    df['Proj_MIN'] = minutes_model_final.predict(X_min_full_imputed)
    imputer_stats_final = SimpleImputer(strategy='mean'); X_stats_full_imputed = imputer_stats_final.fit_transform(df[stat_features])
    joblib.dump(imputer_stats_final, os.path.join(MODEL_DIR, "stats_imputer.pkl")); joblib.dump(stat_features, os.path.join(MODEL_DIR, "stats_feature_names.pkl"))

    for prop in ['PTS', 'REB', 'AST', 'FG3M']:
        model_final = lgb.LGBMRegressor(**best_params_stats[prop], random_state=42)
        model_final.fit(X_stats_full_imputed, df[f'TARGET_{prop}'])
        joblib.dump(model_final, os.path.join(MODEL_DIR, f"{prop.lower()}_lgbm_model.pkl"))
    print("✅ Final Stats models saved.")

    df['Proj_PTS'] = joblib.load(os.path.join(MODEL_DIR, "pts_lgbm_model.pkl")).predict(X_stats_full_imputed)
    df['Proj_REB'] = joblib.load(os.path.join(MODEL_DIR, "reb_lgbm_model.pkl")).predict(X_stats_full_imputed)
    df['Proj_AST'] = joblib.load(os.path.join(MODEL_DIR, "ast_lgbm_model.pkl")).predict(X_stats_full_imputed)
    classification_features = ['Proj_PTS', 'Proj_REB', 'Proj_AST']

    # Classification models kept default for speed/stability as they are secondary
    for prop in ['DD', 'TD']:
        y_class_full = df[f'TARGET_{prop}']
        if y_class_full.sum() > 50:
            classifier_final = lgb.LGBMClassifier(objective='binary', class_weight='balanced', random_state=42); classifier_final.fit(df[classification_features], y_class_full)
            joblib.dump(classifier_final, os.path.join(MODEL_DIR, f"{prop.lower()}_lgbm_model.pkl"))
    print("✅ Final Classification models saved.")

if __name__ == "__main__":
    train_and_validate_models_v7()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 5.2 MB/s eta 0:00:00
Mounted at /content/drive
Loaded 226813 rows from v6 file.

--- Performing Chronological Train/Test Split ---


[I 2026-01-06 16:07:47,460] A new study created in memory with name: no-name-9fed7128-d19a-4d32-84ea-e5c907336a09


Training data: 217715 rows | Test data: 9098 rows

--- STAGE 1: Training & Validating Minutes Model (w/ Optuna) ---
   >>> Tuning Hyperparameters for Minutes (20 trials)...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-01-06 16:07:52,480] Trial 0 finished with value: 6.548351202656505 and parameters: {'n_estimators': 170, 'learning_rate': 0.26885685415000393, 'num_leaves': 45, 'max_depth': 10, 'min_child_samples': 9, 'subsample': 0.5445113533405561, 'colsample_bytree': 0.862158035669512, 'reg_alpha': 0.9277710287228064, 'reg_lambda': 0.605739529986156}. Best is trial 0 with value: 6.548351202656505.
/usr/local/lib/python3.12/dist-packag

   ✅ Best params for Minutes: {'n_estimators': 380, 'learning_rate': 0.03789529148832791, 'num_leaves': 51, 'max_depth': 4, 'min_child_samples': 10, 'subsample': 0.5689970783703133, 'colsample_bytree': 0.826276084353408, 'reg_alpha': 0.28680096809218614, 'reg_lambda': 0.379924647270642}


...

  - FG3M Model Validation RMSE: 0.323 | MAE: 0.129
✅ MAE values saved to mae_values.json

--- Finalizing Production Models (Re-training on all data) ---


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ Final Stats models saved.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ Final Classification models saved.


NBA.com (nba_api) to ESPN Player ID Mapper

In [ ]:
# Install libraries
!pip install requests nba-api pandas fuzzywuzzy python-levenshtein

import requests
import pandas as pd
from nba_api.stats.static import players
from fuzzywuzzy import process
import time
import re
import unicodedata

# --- Configuration ---
ODDS_API_KEY = 'f8474479351ccc685ccae9f4716c23c5'
NBA_SPORT_KEY = 'basketball_nba'

def normalize_name(name):
    if not isinstance(name, str): return ""
    # Normalize accents (e.g. Šarić -> Saric)
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('utf-8')
    name = name.lower().strip()
    # Remove suffixes and punctuation
    name = re.sub(r'\s(jr|sr|i|ii|iii|iv|v)\.?$', '', name)
    name = re.sub(r'[^\w\s]', '', name)
    return name

# 1. Get Official NBA IDs
def get_nba_api_players():
    print("Fetching NBA-API official records...")
    nba_players = players.get_players()
    return {normalize_name(p['full_name']): {'NBA_Official_Name': p['full_name'], 'NBA_ID': p['id']} for p in nba_players}

# 2. Get ALL ESPN Players (The "Secret" Endpoint)
def get_all_espn_athletes():
    espn_players = {}
    print("Fetching COMPLETE ESPN Athlete Database (Active & Inactive)...")

    # This URL pulls ALL players, not just active rosters. Limit set high to catch everyone.
    url = "https://sports.core.api.espn.com/v3/sports/basketball/nba/athletes?limit=18000"

    try:
        res = requests.get(url, timeout=20)
        res.raise_for_status()
        data = res.json()
        items = data.get('items', [])

        print(f"  > Downloaded {len(items)} athletes from ESPN.")

        for p in items:
            # We need first and last name to build full name
            fname = p.get('firstName')
            lname = p.get('lastName')
            pid = p.get('id')

            if fname and lname and pid:
                full_name = f"{fname} {lname}"
                norm = normalize_name(full_name)
                # Save to map
                espn_players[norm] = {'ESPN_Name': full_name, 'ESPN_ID': pid}

    except Exception as e:
        print(f"Error fetching ESPN Database: {e}")

    return espn_players

# 3. Get Odds API Whitelists
def get_odds_api_players():
    odds_players = {}
    base_url = f"https://api.the-odds-api.com/v4/sports/{NBA_SPORT_KEY}/participants"
    teams_url = f"{base_url}?apiKey={ODDS_API_KEY}"

    try:
        res = requests.get(teams_url, timeout=15)
        if res.status_code != 200: return {}
        team_list = res.json()
        print(f"Found {len(team_list)} Odds API teams. Fetching whitelists...")

        for team in team_list:
            tid = team.get('id')
            tname = team.get('name') or team.get('title') or team.get('description') or "Unknown"
            if not tid: continue

            # Undocumented players endpoint
            p_url = f"{base_url}/{tid}/players?apiKey={ODDS_API_KEY}"
            try:
                p_res = requests.get(p_url, timeout=10)
                if p_res.status_code == 200:
                    for p in p_res.json():
                        pname = p.get('full_name') or p.get('name')
                        pid = p.get('id')
                        if pname and pid:
                            norm = normalize_name(pname)
                            odds_players[norm] = {
                                'Odds_API_Name': pname,
                                'Odds_API_ID': pid,
                                'Odds_API_Team': tname
                            }
                time.sleep(0.05) # Faster sleep since we aren't doing deep searches
            except Exception: continue
    except Exception as e: print(f"Odds API Error: {e}")
    return odds_players

# 4. Master Mapping Logic
def generate_master_map():
    nba_map = get_nba_api_players()
    espn_map = get_all_espn_athletes() # This now has thousands of players
    odds_map = get_odds_api_players()

    results = []

    # We focus on mapping every player found in The Odds API
    # because that is likely your betting target list.
    target_list = list(odds_map.keys())

    # Also include anyone active on ESPN just in case
    # (Optional: Comment this line out if you ONLY want players in Odds API)
    target_list = list(set(target_list + list(espn_map.keys())))

    # Filter to ensure we only process reasonable data (remove empty strings)
    target_list = [t for t in target_list if t]

    print(f"\nCreating Master Map for {len(target_list)} unique players...")

    # Pre-calculate keys for faster matching
    espn_keys = list(espn_map.keys())
    nba_keys = list(nba_map.keys())

    for i, norm_name in enumerate(target_list):
        row = {'Normalized_Name': norm_name}

        # 1. Fill Odds API Data
        if norm_name in odds_map:
            row.update(odds_map[norm_name])

        # 2. Fill ESPN Data (Direct Match First)
        if norm_name in espn_map:
            row.update(espn_map[norm_name])
        else:
            # Fuzzy match only if direct match fails (for spelling differences)
            best_espn, score = process.extractOne(norm_name, espn_keys)
            if score >= 90:
                row.update(espn_map[best_espn])

        # 3. Fill NBA Official Data
        if norm_name in nba_map:
            row.update(nba_map[norm_name])
        else:
            best_nba, score = process.extractOne(norm_name, nba_keys)
            if score >= 90:
                row.update(nba_map[best_nba])

        # Only save rows that have at least one ID from our sources
        if 'Odds_API_ID' in row or 'ESPN_ID' in row:
            results.append(row)

        if i % 500 == 0 and i > 0:
            print(f"  Mapped {i} players...")

    return pd.DataFrame(results)

if __name__ == "__main__":
    start = time.time()
    df = generate_master_map()

    if not df.empty:
        # Organize columns
        desired_cols = [
            'NBA_Official_Name', 'NBA_ID',
            'ESPN_Name', 'ESPN_ID',
            'Odds_API_Name', 'Odds_API_ID', 'Odds_API_Team'
        ]
        final_cols = [c for c in desired_cols if c in df.columns]
        df = df[final_cols]

        # Save
        df.to_csv('nba_master_id_map.csv', index=False)
        print(f"\nSUCCESS: Mapped {len(df)} players in {round(time.time()-start, 2)}s")
        print(df.head(10))
    else:
        print("Failed to map players.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 2.8 MB/s eta 0:00:00
Fetching NBA-API official records...
Fetching COMPLETE ESPN Athlete Database (Active & Inactive)...
  > Downloaded 830 athletes from ESPN.
Found 30 Odds API teams. Fetching whitelists...

Creating Master Map for 923 unique players...
  Mapped 500 players...

SUCCESS: Mapped 923 players in 29.26s
    NBA_Official_Name     NBA_ID           ESPN_Name  ESPN_ID  \
0        Curtis Jones  1642938.0        Curtis Jones  4898369   
1      Isaiah Stewart  1630191.0      Isaiah Stewart  4432810   
2      Gary Payton II  1627780.0      Gary Payton II  3134903   
3     Bismack Biyombo   202687.0     Bismack Biyombo     6427   
4        Steven Adams   203500.0        Steven Adams  2991235   
5                 NaN        NaN                 NaN      NaN   
6  Michael Porter Jr.  1629008.0  Michael Porter Jr.  4278104   
7       Pascal Siakam  1627783.0       Pascal Siakam  3149673   
8                 NaN        NaN    Ca

Shot quality

In [ ]:
!pip install nba_api pandas

import pandas as pd
from nba_api.stats.static import players, teams
from nba_api.stats.endpoints import playerdashptshots
import time

# 1. SETUP: Find IDs for a Test Case (Steph Curry)
nba_players = players.get_players()
curry = [p for p in nba_players if p['full_name'] == 'Stephen Curry'][0]
curry_id = curry['id']

nba_teams = teams.get_teams()
warriors = [t for t in nba_teams if t['full_name'] == 'Golden State Warriors'][0]
warriors_id = warriors['id']

print(f"--- Fetching Shot Quality Data for {curry['full_name']} ---")

try:
    # 2. CALL THE CLASS: Using the endpoint parameters
    # This fetches data for the current season (2024-25)
    shot_analysis = playerdashptshots.PlayerDashPtShots(
        team_id=warriors_id,
        player_id=curry_id,
        season='2024-25'
    )

    # 3. EXTRACT DATASETS: Seeing the "Advanced Frontier" in action

    # Dataset 1: Closest Defender (Shot Quality/qSQ)
    df_defender = shot_analysis.closest_defender_shooting.get_data_frame()

    # Dataset 2: Shot Clock (Stress Factor)
    df_clock = shot_analysis.shot_clock_shooting.get_data_frame()

    # 4. DISPLAY RESULTS
    print("\n[TABLE 1] Shot Quality vs. Defender Distance")
    # We filter to relevant columns for easy reading
    cols_to_show = ['CLOSE_DEF_DIST_RANGE', 'FGA_FREQUENCY', 'FG_PCT', 'EFG_PCT']
    display(df_defender[cols_to_show])

    print("\n[TABLE 2] Shooting Performance vs. Shot Clock")
    cols_clock = ['SHOT_CLOCK_RANGE', 'FGA_FREQUENCY', 'FG_PCT']
    display(df_clock[cols_clock])

except Exception as e:
    print(f"Error: {e}")
    print("If you got a 403 or Timeout, the NBA servers are rate-limiting. Wait 60 seconds and try again.")

--- Fetching Shot Quality Data for Stephen Curry ---

[TABLE 1] Shot Quality vs. Defender Distance


,CLOSE_DEF_DIST_RANGE,FGA_FREQUENCY,FG_PCT,EFG_PCT
0,0-2 Feet - Very Tight,0.045,0.509,0.535
1,2-4 Feet - Tight,0.338,0.433,0.496
2,4-6 Feet - Open,0.434,0.452,0.601
3,6+ Feet - Wide Open,0.182,0.454,0.655



[TABLE 2] Shooting Performance vs. Shot Clock


,SHOT_CLOCK_RANGE,FGA_FREQUENCY,FG_PCT
0,24-22,0.008,0.400
1,22-18 Very Early,0.149,0.460
2,18-15 Early,0.222,0.416
3,15-7 Average,0.500,0.467
4,7-4 Late,0.064,0.494
5,4-0 Very Late,0.056,0.338
